# Assignment: Capacitated Facility Location Problem (CFLP)

## Comparison of Exact vs Tabu Search and Genetic Algorithm

**Course:** CS616 - Optimization Algorithms  
**Instructors:** Dr. Mahdi Khemakhem & Dr. Essra Aldessouki

---

## 1. Problem Description:

A company must decide which facilities to open to serve a set of clients at minimum total cost. Each facility has a **fixed opening cost** and a **limited capacity**. Each client has a **demand** that must be satisfied. Each client must be **assigned to exactly one open facility**, and the **total demand assigned to any facility cannot exceed its capacity**.

The total cost includes both **facility opening costs** and **client-to-facility assignment costs**.

The **Capacitated Facility Location Problem (CFLP)** is more challenging than the uncapacitated version because it adds capacity constraints, making it significantly harder for exact solvers while creating more realistic scenarios.

The CFLP appears in many real-world applications, such as:
- Warehouse location with storage limits
- Data center placement with processing capacity
- Service center allocation with staff limits
- Distribution hub planning with throughput constraints
- Healthcare facility location with patient capacity
- Retail store placement with inventory limits
- Supply chain network design with capacity restrictions

The global decision of a CFLP is to determine which facilities to open and how to assign clients to these facilities to minimize the total cost while respecting capacity constraints.

The CFLP is **strongly NP-hard**, making it computationally more challenging than the uncapacitated version. Exact methods like Mixed-Integer Programming (MIP) struggle with even moderate-sized instances, while metaheuristics such as **Tabu Search (TS)** and **Genetic Algorithms (GA)** can provide good solutions in reasonable time.

This assignment compares three approaches:
1. **Gurobi (Exact MIP)**: Guaranteed optimal solutions, but computationally expensive for large instances.
2. **Tabu Search (TS)**: A trajectory-based metaheuristic that uses a tabu list to avoid revisiting recent solutions and escape local optima.
3. **Genetic Algorithm (GA)**: A population-based metaheuristic inspired by natural evolution that explores the solution space through selection, crossover, and mutation.

The CFLP can be formulated as follows:

### Mathematical Formulation

To model the Capacitated Facility Location Problem, we define the following components:

#### Sets:
- $𝐼 = \{1, 2, \ldots, n\}$: Set of $n$ clients.
- $𝐽 = \{1, 2, \ldots, m\}$: Set of potential $m$ facilities.

#### Parameters:
- $f_j$: Fixed cost of opening facility $j \in J$.
- $c_{ij}$: Cost of assigning client $i \in I$ to facility $j \in J$.
- $q_j$: **Capacity** of facility $j \in J$ (maximum demand it can serve).
- $d_i$: **Demand** of client $i \in I$ (must be satisfied).

#### Decision Variables:
$$\forall i \in I, \forall j \in J, x_{𝑖𝑗} =
\begin{cases}
1 & \text{ if client } 𝑖 \text{ is assigned to facility } 𝑗 \\
0 & \text{ otherwise}
\end{cases}
$$
$$\forall j \in J, 𝑦_{𝑗} = \begin{cases}
1 & \text{ if facility } 𝑗 \text{ is opened} \\
0 & \text{ otherwise}
\end{cases}$$

#### Objective Function:

Minimize the total cost:

$$\text{Minimize } Z = \sum_{j \in J} f_j y_j + \sum_{i \in I} \sum_{j \in J} c_{ij} x_{ij}$$

#### Constraints:
1. **Demand satisfaction**: Each client is assigned to exactly one facility:
    $$\sum_{j \in J} x_{ij} = 1, \quad \forall i \in I$$

2. **Linking constraint**: Clients can only be assigned to open facilities:
   $$x_{ij} \leq y_{j}, \quad \forall i \in I, \forall j \in J$$

3. **Capacity constraint**: Total demand assigned to a facility cannot exceed its capacity:
   $$\sum_{i \in I} d_i \cdot x_{ij} \leq q_j \cdot y_j, \quad \forall j \in J$$

4. **Binary constraints** on decision variables:
   $$x_{ij} \in \{0, 1\}, \quad \forall i \in I, \forall j \in J$$
   $$y_{j} \in \{0, 1\}, \quad \forall j \in J$$

#### Problem Complexity:
The capacity constraints make CFLP significantly harder than uncapacitated FLP because:
- **Tighter LP relaxation**: Capacity constraints create stronger linear programming bounds
- **Combinatorial explosion**: Limited capacity forces more facilities to open
- **Packing problem**: Assigning clients becomes a bin-packing subproblem
- **Infeasibility risk**: Tight capacities can lead to infeasible partial solutions

This makes CFLP an excellent benchmark for comparing exact and metaheuristic methods!


---

## 2. Import Libraries

In this section we import the necessary libraries for data handling, optimization, and visualization.

If a library is not installed, the script will attempt to install it using `pip install`.

In [1]:
# Try to import necessary libraries, install if not available
try:
    import numpy as np
except ImportError:
    import sys
    !{sys.executable} -m pip install numpy
    import numpy as np
NUMPY_AVAILABLE = True

try:
    import pandas as pd
except ImportError:
    import sys
    !{sys.executable} -m pip install pandas
    import pandas as pd
PANDAS_AVAILABLE = True

try:
    import matplotlib.pyplot as plt
except ImportError:
    import sys
    !{sys.executable} -m pip install matplotlib
    import matplotlib.pyplot as plt
MATPLOTLIB_AVAILABLE = True

try:
    import seaborn as sns
except ImportError:
    import sys
    !{sys.executable} -m pip install seaborn
    import seaborn as sns
SEABORN_AVAILABLE = True

try:
    from scipy.interpolate import make_interp_spline
except ImportError:
    import sys
    !{sys.executable} -m pip install scipy
    from scipy.interpolate import make_interp_spline
SCIPY_AVAILABLE = True

try:
    import gurobipy as gp
    from gurobipy import GRB
except ImportError:
    import sys
    !{sys.executable} -m pip install gurobipy
    import gurobipy as gp
    from gurobipy import GRB
GUROBI_AVAILABLE = True

import time 
import os
import warnings
import shutil 
import json
from typing import Tuple, List

warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

if NUMPY_AVAILABLE and PANDAS_AVAILABLE and MATPLOTLIB_AVAILABLE and SEABORN_AVAILABLE and SCIPY_AVAILABLE and GUROBI_AVAILABLE:
    print("✅ All libraries imported successfully.")
else:
    print("❌ Some libraries failed to import.")


✅ All libraries imported successfully.


---

## 3. Instance Generator and Loader

In this section, we generate challenging **Capacitated Facility Location Problem (CFLP)** instances that create meaningful computational challenges for both exact and metaheuristic algorithms. Our instance generator creates **144 test cases** across 3 size categories and 3 difficulty levels, following a validated economic model with realistic capacity constraints.

### 3.1 Instance Generation Methodology

The generator implements a **research-based ratio model** for CFLP following operations research best practices:

$$\sum_{j} f_j = \alpha \times \sum_{i} \min_{j} c_{ij}$$

where:
- $\sum_{j} f_j$ = total fixed costs across all facilities
- $\alpha$ = fixed-to-**optimal**-service cost **ratio** (NOT multiplier!)
- $\sum_{i} \min_{j} c_{ij}$ = **optimal service cost** (minimum cost per client, NOT sum of all pairs!)
- Individual facility costs: $f_j = \frac{\alpha \cdot \text{optimal\_service}}{n} \cdot \frac{\bar{c}_j}{\bar{c}_{global}} \cdot \eta_j$

**CRITICAL**: α is computed against the **optimal service cost** (what you'd pay if each client uses their nearest facility), NOT the sum of all service cost pairs. This ensures proper difficulty calibration!

**Key Insight**: Computational hardness comes from **balanced trade-offs**, not high absolute costs!
- When $\alpha \approx 0.5$: Fixed costs = 50% of optimal service → Open many facilities → **EASY**
- When $\alpha \approx 1.0$: Fixed costs = 100% of optimal service → True trade-offs → **HARDEST!**
- When $\alpha \approx 3.0$: Fixed costs = 300% of optimal service → Open few facilities → **MEDIUM**

This creates a **U-shaped difficulty curve** where moderate instances (α ≈ 1.0) are the most challenging.

### 3.2 Size Categories (Optimized for Free Gurobi + CFLP Hardness)

Instances are carefully sized to **stay within free Gurobi limits (2000 variables, 2000 constraints)** while maximizing computational difficulty through **capacity constraints**:

**Variable Count**: $n \times m + m$ (assignment variables + facility variables)
**Constraint Count**: $n + n \times m + m$ (demand + linking + capacity constraints)

| Category | Facilities | Clients | Variables | Constraints | Characteristics |
|----------|------------|---------|-----------|-------------|-----------------|
| **Small** | 15-20 | 50-80 | 750-1,620 | 815-1,700 | Solvable but challenging with capacity |
| **Medium** | 18-24 | 60-75 | 1,080-1,824 | 1,158-1,923 | Very challenging with tight capacities |
| **Large** | 20-28 | 65-70 | 1,320-1,988 | 1,405-2,086 | **Near limit**, capacity creates extreme difficulty |

### 3.3 Difficulty Levels (Capacity Tightness + Cost Ratio Model)

| Level | α Range | β (Capacity/Demand) | Algorithmic Challenge |
|-------|---------|---------------------|----------------------|
| **Easy** | [0.3, 0.6] | [1.4, 1.7] | Loose capacity + cheap fixed → Simple |
| **Moderate** | [0.8, 1.5] | [1.1, 1.3] | **HARDEST! Tight capacity + balanced costs** |
| **Expensive** | [2.0, 4.0] | [1.2, 1.5] | Expensive facilities + moderate capacity |

### 3.4 Expected Computational Behavior

**Gurobi (Exact Method)**:
- Small easy: 2-10 seconds
- Small moderate: 10-60 seconds (tight capacity creates extreme difficulty!)
- Medium moderate: 40-180 seconds
- Large moderate: 120-300+ seconds

**Tabu Search (TS)**:
- Small: 0.5-3 seconds
- Medium: 1-5 seconds
- Large: 2-8 seconds
- Gap pattern: Easy 2-6%, Moderate 8-18%, Expensive 4-10%

**Genetic Algorithm (GA)**:
- Small: 1-5 seconds
- Medium: 2-8 seconds
- Large: 3-12 seconds
- Gap pattern: Easy 3-8%, Moderate 10-20%, Expensive 5-12%

**Key Insight**: **Capacity tightness** (β ≈ 1.2) combined with **cost balance** (α ≈ 1.0) creates the HARDEST instances. Both TS and GA trade solution quality for speed compared to Gurobi.

### 3.5 Instance Distribution

The generator creates **144 total instances**:
- 48 instances per size category (small, medium, large)
- Within each size: 16 instances per difficulty level
- Ensures balanced evaluation across all 9 combinations (3×3)

### 3.6 Output Format

Each instance is saved as a **JSON file** with keys: `num_facilities`, `num_clients`, `fixed_costs`, `service_costs`, `facility_capacities`, `client_demands`, `client_coordinates`, `facility_coordinates`.


In [2]:
def generate_fixed_costs_from_cost_matrix(service_costs: np.ndarray, alpha: float, 
                                         noise_range: tuple = (0.8, 1.2)) -> np.ndarray:
    """
    Generate fixed costs using α as RATIO of OPTIMAL service cost to fixed cost.
    
    CRITICAL: α is computed against OPTIMAL service cost (minimum per client),
    NOT against the sum of all service cost pairs!
    
    Research-based model: Total fixed costs should be α × optimal service costs
    This creates meaningful trade-offs:
        - α < 0.5: Fixed costs cheap → open many facilities → EASY
        - α ≈ 1.0: Fixed costs ≈ optimal service costs → hard trade-offs → HARDEST!
        - α > 2.0: Fixed costs expensive → open few facilities → MEDIUM
    
    Formula: 
        optimal_service = Σ(min_j c_ij)  ← Cost if each client uses nearest facility
        target_fixed = α × optimal_service
        f_j = (target_fixed / n) × (c̄_j / c̄_global) × η_j
    
    where:
        - α is the fixed-to-optimal-service cost ratio
        - optimal_service is the minimum achievable service cost
        - n is number of facilities (distributes total budget)
        - (c̄_j / c̄_global) scales by relative facility importance
        - η_j adds heterogeneity noise
    
    Parameters:
    -----------
    service_costs : np.ndarray
        Service cost matrix of shape (num_clients, num_facilities)
    alpha : float
        Fixed-to-optimal-service cost RATIO (0.3-0.6: easy, 0.8-1.5: hard, 2.0-4.0: medium)
    noise_range : tuple
        Range for multiplicative noise (default: ±20%)
    
    Returns:
    --------
    fixed_costs : np.ndarray
        Fixed costs for each facility
    """
    num_clients, num_facilities = service_costs.shape
    
    # CRITICAL FIX: Use OPTIMAL service cost (minimum per client) as baseline!
    # Not the sum of all pairs which is irrelevant to the actual problem
    optimal_service_cost = service_costs.min(axis=1).sum()
    
    # Target: Total fixed costs = α × OPTIMAL service cost
    # This is the CORRECT way to calibrate difficulty!
    target_total_fixed_cost = alpha * optimal_service_cost
    
    # Compute average cost per facility (for proportional distribution)
    avg_cost_per_facility = service_costs.mean(axis=0)
    global_avg_cost = avg_cost_per_facility.mean()
    
    # Distribute total budget proportionally to facility importance
    # Facilities serving expensive regions get higher fixed costs
    importance_weights = avg_cost_per_facility / global_avg_cost
    base_fixed_cost = target_total_fixed_cost / num_facilities
    
    # Scale by importance and add heterogeneity
    noise_factors = np.random.uniform(noise_range[0], noise_range[1], size=num_facilities)
    fixed_costs = base_fixed_cost * importance_weights * noise_factors
    
    # Normalize to maintain target total (compensate for noise)
    fixed_costs = fixed_costs * (target_total_fixed_cost / fixed_costs.sum())
    
    # Ensure positive and reasonable values
    fixed_costs = np.maximum(fixed_costs, 1.0)
    
    return fixed_costs


def generate_instances(folder: str = "cflp_instances", num_total_instances: int = 144, seed: int = 42):
    """
    Generate Capacitated Facility Location Problem (CFLP) instances with scientific rigor.
    
    Creates instances across 3 size categories (small, medium, large) and 
    3 difficulty levels (easy, moderate, expensive), following the methodology:
    
    Size Categories:
    - Small: 2-5 facilities, 30-60 clients
    - Medium: 6-10 facilities, 80-150 clients  
    - Large: 12-20 facilities, 200-400 clients
    
    Difficulty Levels (via α parameter):
    - Easy: α ∈ [5, 10] - Low fixed costs, most facilities profitable
    - Moderate: α ∈ [10, 20] - Balanced trade-offs
    - Expensive: α ∈ [20, 40] - High fixed costs, selective openings
    
    Parameters:
    -----------
    folder : str
        Directory to save generated instances
    num_total_instances : int
        Total number of instances to generate (distributed evenly)
    seed : int
        Random seed for reproducibility
    
    Returns:
    --------
    None (instances saved to disk with summary Excel file)
    """
    np.random.seed(seed)
    
    # ==================================================================================
    # CLEANUP: Remove all old instances before generating new ones
    # ==================================================================================
    print(f"\n{'='*80}")
    print(f"CLEANING UP OLD INSTANCES")
    print(f"{'='*80}")
    
    if os.path.exists(folder):
        file_count = 0
        dir_count = 0
        
        # Count and delete all contents
        for filename in os.listdir(folder):
            file_path = os.path.join(folder, filename)
            try:
                if os.path.isfile(file_path):
                    os.unlink(file_path)
                    file_count += 1
                elif os.path.isdir(file_path):
                    shutil.rmtree(file_path)
                    dir_count += 1
            except Exception as e:
                print(f"⚠️  Warning: Could not delete {file_path}: {e}")
        
        # Remove and recreate folder to ensure complete cleanup
        try:
            shutil.rmtree(folder)
            print(f"✓ Removed existing folder '{folder}'")
            print(f"  - Deleted {file_count} file(s)")
            print(f"  - Deleted {dir_count} subdirectory(ies)")
        except Exception as e:
            print(f"⚠️  Warning: Could not remove folder {folder}: {e}")
    else:
        print(f"✓ Folder '{folder}' does not exist (first run)")
    
    # Create fresh output folder
    os.makedirs(folder, exist_ok=True)
    print(f"✓ Created fresh folder '{folder}'\n")

    # ==================================================================================
    # CFLP DIFFICULTY MODEL (Capacity Tightness + Cost Ratio)
    # ==================================================================================
    # Key factors creating computational hardness in CFLP:
    # 1. CAPACITY TIGHTNESS (β): Ratio of total capacity to total demand
    #    - β < 1.2: Very tight → forces many facilities → EXTREMELY HARD
    #    - β ≈ 1.3: Moderate → interesting trade-offs → HARD
    #    - β > 1.5: Loose → capacity less binding → easier
    # 2. COST RATIO (α): Fixed costs relative to service costs
    # 3. PROBLEM SIZE: Limited by free Gurobi (2000 vars, 2000 constraints)
    #
    # Variables: n*m + m  |  Constraints: n + n*m + m
    # Must keep: (n*m + m) ≤ 2000  AND  (n + n*m + m) ≤ 2000
    size_settings = {
        "small": {
            "fac_range": (15, 21),      # 15-20 facilities
            "cli_range": (50, 81),      # 50-80 clients → Max ~1,620 vars, ~1,700 constraints
            "alpha_ranges": {
                "easy": (0.3, 0.6),      # Cheap facilities + loose capacity
                "moderate": (0.8, 1.5),  # Balanced costs + tight capacity → HARDEST!
                "expensive": (2.0, 4.0)  # Expensive facilities + moderate capacity
            },
            "beta_ranges": {
                "easy": (1.4, 1.7),      # Loose capacity (40-70% slack)
                "moderate": (1.1, 1.3),  # Tight capacity (10-30% slack) → HARDEST!
                "expensive": (1.2, 1.5)  # Moderate capacity
            }
        },
        "medium": {
            "fac_range": (18, 25),      # 18-24 facilities
            "cli_range": (60, 76),      # 60-75 clients → Max ~1,824 vars, ~1,923 constraints
            "alpha_ranges": {
                "easy": (0.3, 0.6),
                "moderate": (0.8, 1.5),  # HARDEST with tight capacity!
                "expensive": (2.0, 4.0)
            },
            "beta_ranges": {
                "easy": (1.4, 1.7),
                "moderate": (1.1, 1.3),  # Very challenging!
                "expensive": (1.2, 1.5)
            }
        },
        "large": {
            "fac_range": (20, 28),      # 20-27 facilities (reduced max to stay within limits)
            "cli_range": (65, 69),      # 65-68 clients → Max ~1,863 vars, ~1,944 constraints (safe margin!)
            "alpha_ranges": {
                "easy": (0.3, 0.6),
                "moderate": (0.8, 1.5),  # HARDEST! Near Gurobi free limit + tight capacity
                "expensive": (2.0, 4.0)
            },
            "beta_ranges": {
                "easy": (1.4, 1.7),
                "moderate": (1.1, 1.3),  # Extremely challenging!
                "expensive": (1.2, 1.5)
            }
        }
    }

    # Calculate instances per category (evenly distributed)
    num_sizes = len(size_settings)
    num_difficulties = 3
    instances_per_combination = num_total_instances // (num_sizes * num_difficulties)
    
    instances_info = []
    instance_id = 1

    print(f"Generating {num_total_instances} CFLP instances...")
    print(f"Distribution: {instances_per_combination} per size-difficulty combination")
    print(f"All instances comply with free Gurobi limits (≤2000 vars, ≤2000 constraints)\n")
    
    # Generate instances for each size-difficulty combination
    for size, params in size_settings.items():
        for difficulty, alpha_range in params["alpha_ranges"].items():
            beta_range = params["beta_ranges"][difficulty]
            
            for rep in range(instances_per_combination):
                # Randomize problem dimensions
                num_facilities = np.random.randint(*params["fac_range"])
                num_clients = np.random.randint(*params["cli_range"])
                
                # Verify free Gurobi compliance
                num_vars = num_clients * num_facilities + num_facilities
                num_constraints = num_clients + num_clients * num_facilities + num_facilities
                
                if num_vars > 2000 or num_constraints > 2000:
                    print(f"⚠️  WARNING: Instance too large! Vars={num_vars}, Constraints={num_constraints}")
                    continue
                
                # Sample α (cost ratio) and β (capacity tightness) from difficulty range
                alpha = np.random.uniform(*alpha_range)
                beta = np.random.uniform(*beta_range)

                # Generate UNIQUE random 2D coordinates using grid-based approach
                # This ensures no overlapping points and creates diverse spatial patterns
                grid_size = 1000  # Large grid for unique positions
                
                # Generate unique client positions
                clients = []
                used_positions = set()
                attempts = 0
                while len(clients) < num_clients and attempts < num_clients * 10:
                    x = np.random.uniform(0, grid_size)
                    y = np.random.uniform(0, grid_size)
                    # Round to ensure uniqueness but maintain diversity
                    pos_key = (round(x, 1), round(y, 1))
                    if pos_key not in used_positions:
                        clients.append([x, y])
                        used_positions.add(pos_key)
                    attempts += 1
                clients = np.array(clients)
                
                # Generate unique facility positions (separate from clients)
                facilities = []
                attempts = 0
                while len(facilities) < num_facilities and attempts < num_facilities * 10:
                    x = np.random.uniform(0, grid_size)
                    y = np.random.uniform(0, grid_size)
                    pos_key = (round(x, 1), round(y, 1))
                    if pos_key not in used_positions:
                        facilities.append([x, y])
                        used_positions.add(pos_key)
                    attempts += 1
                facilities = np.array(facilities)

                # Compute service costs: distance + perturbation
                # Perturbation creates problem structure, not difficulty directly
                service_costs = np.zeros((num_clients, num_facilities))
                
                # Use CONSTANT moderate noise across all difficulties
                # Difficulty comes from α (cost ratio) and problem size, NOT noise!
                base_noise = 0.15  # 15% noise is realistic for real-world problems
                
                for i in range(num_clients):
                    for j in range(num_facilities):
                        euclidean_dist = np.linalg.norm(clients[i] - facilities[j])
                        # Add proportional noise (15% of distance)
                        perturbation = euclidean_dist * np.random.uniform(0, base_noise)
                        service_costs[i, j] = euclidean_dist + perturbation

                # ===================================================================
                # GENERATE DEMANDS AND CAPACITIES FOR CFLP
                # ===================================================================
                # Client demands: uniform random in [5, 20] (realistic range)
                client_demands = np.random.uniform(5, 20, size=num_clients)
                total_demand = client_demands.sum()
                
                # Facility capacities based on β (capacity tightness ratio)
                # β = total_capacity / total_demand
                # Lower β → tighter capacity → harder problem!
                target_total_capacity = beta * total_demand
                
                # Distribute capacity across facilities with heterogeneity
                # Base capacity per facility
                base_capacity = target_total_capacity / num_facilities
                
                # Add heterogeneity: some facilities larger, some smaller
                capacity_factors = np.random.uniform(0.7, 1.3, size=num_facilities)
                facility_capacities = base_capacity * capacity_factors
                
                # Normalize to maintain exact β ratio
                facility_capacities = facility_capacities * (target_total_capacity / facility_capacities.sum())
                
                # Ensure all capacities can serve at least 2-3 clients
                min_capacity = 2 * client_demands.mean()
                facility_capacities = np.maximum(facility_capacities, min_capacity)
                
                # Recalculate actual β after adjustments
                actual_beta = facility_capacities.sum() / total_demand
                
                # ===================================================================
                # GENERATE FIXED COSTS USING α RATIO MODEL
                # ===================================================================
                fixed_costs = generate_fixed_costs_from_cost_matrix(
                    service_costs, 
                    alpha=alpha, 
                    noise_range=(0.8, 1.2)  # ±20% heterogeneity in fixed costs
                )

                # Create instance filename
                instance_name = f"{instance_id:03d}_cflp_{size}_{difficulty}_fac{num_facilities}_cli{num_clients}.json"
                instance_path = os.path.join(folder, instance_name)

                # Prepare instance data with capacities and demands
                instance_data = {
                    "num_facilities": int(num_facilities),
                    "num_clients": int(num_clients),
                    "size": size,
                    "difficulty": difficulty,
                    "alpha": round(alpha, 3),
                    "beta": round(actual_beta, 3),
                    "total_demand": round(total_demand, 2),
                    "total_capacity": round(facility_capacities.sum(), 2),
                    "fixed_costs": fixed_costs.tolist(),
                    "service_costs": service_costs.tolist(),
                    "facility_capacities": facility_capacities.tolist(),
                    "client_demands": client_demands.tolist(),
                    "client_coordinates": clients.tolist(),
                    "facility_coordinates": facilities.tolist(),
                    "num_variables": int(num_vars),
                    "num_constraints": int(num_constraints)
                }

                # Save instance to JSON
                with open(instance_path, "w") as f:
                    json.dump(instance_data, f, indent=2)

                # Store summary information
                instances_info.append({
                    "instance_name": instance_name,
                    "instance_id": instance_id,
                    "size": size,
                    "difficulty": difficulty,
                    "num_facilities": num_facilities,
                    "num_clients": num_clients,
                    "num_variables": num_vars,
                    "num_constraints": num_constraints,
                    "alpha": round(alpha, 3),
                    "beta": round(actual_beta, 3),
                    "total_demand": round(total_demand, 2),
                    "total_capacity": round(facility_capacities.sum(), 2),
                    "capacity_slack_pct": round((actual_beta - 1.0) * 100, 1),
                    "avg_fixed_cost": round(fixed_costs.mean(), 2),
                    "avg_service_cost": round(service_costs.mean(), 2),
                    "avg_demand": round(client_demands.mean(), 2),
                    "avg_capacity": round(facility_capacities.mean(), 2)
                })
                
                # Create visualization with correct coordinate bounds
                plt.figure(figsize=(10, 10))
                
                # Adjust marker sizes based on instance size for visibility
                if num_clients > 500:
                    client_size = 5
                    facility_size = 50
                elif num_clients > 200:
                    client_size = 10
                    facility_size = 80
                else:
                    client_size = 20
                    facility_size = 120
                
                plt.scatter(facilities[:, 0], facilities[:, 1], 
                           c='red', marker='s', s=facility_size, alpha=0.8, 
                           edgecolors='darkred', linewidths=2, 
                           label=f'Facilities (n={num_facilities})')
                plt.scatter(clients[:, 0], clients[:, 1], 
                           c='blue', marker='o', s=client_size, alpha=0.5, 
                           label=f'Clients (n={num_clients})')
                
                plt.title(f"CFLP Instance {instance_id:03d}: {size.capitalize()}-{difficulty.capitalize()}\n"
                         f"Facilities: {num_facilities}, Clients: {num_clients}, α={alpha:.2f}, β={actual_beta:.2f}",
                         fontsize=14, fontweight='bold')
                plt.xlabel("X Coordinate", fontsize=12)
                plt.ylabel("Y Coordinate", fontsize=12)
                
                # Set proper limits based on actual coordinate space (0-1000)
                plt.xlim(-50, 1050)
                plt.ylim(-50, 1050)
                
                plt.legend(loc='upper right', fontsize=10)
                plt.grid(alpha=0.3, linestyle='--')
                plt.tight_layout()
                
                # Save visualization
                viz_path = os.path.join(folder, instance_name.replace(".json", ".png"))
                plt.savefig(viz_path, dpi=150, bbox_inches='tight')
                plt.close()

                instance_id += 1

    # Create summary Excel file
    summary_df = pd.DataFrame(instances_info)
    summary_path = os.path.join(folder, "instances_summary.xlsx")
    
    with pd.ExcelWriter(summary_path, engine='openpyxl') as writer:
        # Full summary
        summary_df.to_excel(writer, sheet_name='All Instances', index=False)
        
        # Statistics by size
        size_stats = summary_df.groupby('size').agg({
            'num_facilities': ['min', 'max', 'mean'],
            'num_clients': ['min', 'max', 'mean'],
            'alpha': ['min', 'max', 'mean'],
            'beta': ['min', 'max', 'mean'],
            'num_variables': ['min', 'max', 'mean'],
            'num_constraints': ['min', 'max', 'mean'],
            'instance_id': 'count'
        }).round(2)
        size_stats.to_excel(writer, sheet_name='Size Statistics')
        
        # Statistics by difficulty
        diff_stats = summary_df.groupby('difficulty').agg({
            'alpha': ['min', 'max', 'mean'],
            'beta': ['min', 'max', 'mean'],
            'total_demand': ['min', 'max', 'mean'],
            'total_capacity': ['min', 'max', 'mean'],
            'instance_id': 'count'
        }).round(2)
        diff_stats.to_excel(writer, sheet_name='Difficulty Statistics')
        
        # Size-Difficulty matrix
        matrix = summary_df.pivot_table(
            values='instance_id', 
            index='size', 
            columns='difficulty', 
            aggfunc='count',
            fill_value=0
        )
        matrix.to_excel(writer, sheet_name='Distribution Matrix')
    
    print(f"\nSuccessfully generated {len(instances_info)} instances in '{folder}/'")
    print(f"   Summary saved to: {summary_path}")
    print(f"\nDistribution Summary:")
    print(summary_df.groupby(['size', 'difficulty']).size().unstack(fill_value=0))


def load_cflp_instance(file_path: str) -> Tuple[int, int, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Load a Capacitated Facility Location Problem instance from a JSON file.
    
    Parameters:
    -----------
    file_path : str
        Path to the JSON file containing the instance data
        
    Returns:
    --------
    num_facilities : int
        Number of facilities
    num_clients : int
        Number of clients
    fixed_costs : np.ndarray
        Fixed costs for each facility (shape: num_facilities)
    service_costs : np.ndarray
        Service costs matrix (shape: num_clients × num_facilities)
    facility_capacities : np.ndarray
        Capacity of each facility (shape: num_facilities)
    client_demands : np.ndarray
        Demand of each client (shape: num_clients)
    """
    with open(file_path, 'r') as f:
        instance_data = json.load(f)
    
    num_facilities = instance_data['num_facilities']
    num_clients = instance_data['num_clients']
    fixed_costs = np.array(instance_data['fixed_costs'])
    service_costs = np.array(instance_data['service_costs'])
    facility_capacities = np.array(instance_data['facility_capacities'])
    client_demands = np.array(instance_data['client_demands'])
    
    return num_facilities, num_clients, fixed_costs, service_costs, facility_capacities, client_demands

print("✅ CFLP instance generation module loaded successfully.")

# Uncomment the line below to test the instance generation
# generate_instances(folder="cflp_instances", num_total_instances=144, seed=42)

✅ CFLP instance generation module loaded successfully.


In [3]:
def generate_all_instances(output_folder: str = "cflp_instances",
                           sizes=None,
                           difficulties=None,
                           instances_per_combo: int = 16,
                           seed: int = 42):
    """
    Generate all CFLP instances across size and difficulty combinations.

    This is the entry point called by main(). It delegates to generate_instances()
    which creates instances across small/medium/large × easy/moderate/expensive.

    Parameters:
    -----------
    output_folder : str
        Folder where JSON instance files will be saved.
    sizes : list, optional
        List of size categories (default: ['small', 'medium', 'large']).
    difficulties : list, optional
        List of difficulty levels (default: ['easy', 'moderate', 'expensive']).
    instances_per_combo : int
        Number of instances per (size, difficulty) combination.
    seed : int
        Random seed for reproducibility.
    """
    if sizes is None:
        sizes = ['small', 'medium', 'large']
    if difficulties is None:
        difficulties = ['easy', 'moderate', 'expensive']

    num_total = len(sizes) * len(difficulties) * instances_per_combo
    generate_instances(folder=output_folder, num_total_instances=num_total, seed=seed)


print("✅ generate_all_instances() registered successfully.")


✅ generate_all_instances() registered successfully.


---

## 4. Solution Encoding (Representation) and Utility Functions

In this section, we define the solution encoding for the Facility Location Problem (FLP) and implement utility functions to evaluate and manipulate solutions. 

The solution is represented as a dictionary with two components:
1. `facilities`: A binary list indicating which facilities are open (1) or closed (0).
2. `assignments`: A list where each element represents the index of the facility assigned to each client.

For example, a dictionary representing a solution might look like this:
{
  "facilities": [1, 0, 1, 0, 1],
  "assignments": [0, 2, 4, 2, 0, 2, 4]
}
- Facilities: [1, 0, 1, 0, 1] (facilities 1, 3, and 5 are open)
- Assignments: [0, 2, 4, 2, 0, 2, 4] (client 1 assigned to facility 1, client 2 to facility 3, client 3 to facility 5, etc.)

The functions related to solution encoding include:
- `compute_total_cost`: Calculate total cost of a solution
- `is_feasible`: Check if a solution is feasible
- `assign_clients_to_nearest_open_facility`: Assign clients to the nearest open facility
- `copy_solution`: Create a deep copy of a solution
- `greedy_initial_solution`: Generate an initial solution using a greedy heuristic. It consists to:
    1. Start with all facilities closed.
    2. Iteratively open the facility that provides the largest cost reduction until no further improvement is possible.
    3. Assign clients to the nearest open facility.

In [4]:
def compute_total_cost(fixed_costs: np.ndarray, service_costs: np.ndarray, 
                       facility_opened: np.ndarray, client_assignments: np.ndarray) -> float:
    """
    Compute the total cost of a given solution.
    
    Parameters:
    -----------
    fixed_costs : np.ndarray
        Fixed costs for each facility (shape: num_facilities)
    service_costs : np.ndarray
        Service costs matrix (shape: num_clients × num_facilities)
    facility_opened : np.ndarray
        Binary array indicating which facilities are opened (shape: num_facilities)
    client_assignments : np.ndarray
        Array indicating the assigned facility for each client (shape: num_clients)
        
    Returns:
    --------
    total_cost : float
        Total cost of the solution
    """
    total_fixed_cost = np.sum(fixed_costs * facility_opened)
    total_service_cost = np.sum(service_costs[np.arange(len(client_assignments)), client_assignments])
    
    total_cost = total_fixed_cost + total_service_cost
    return total_cost

def is_feasible(facility_opened: np.ndarray, client_assignments: np.ndarray,
                facility_capacities: np.ndarray, client_demands: np.ndarray) -> bool:
    """
    Check if a given solution is feasible for CFLP.
    
    Parameters:
    -----------
    facility_opened : np.ndarray
        Binary array indicating which facilities are opened (shape: num_facilities)
    client_assignments : np.ndarray
        Array indicating the assigned facility for each client (shape: num_clients)
    facility_capacities : np.ndarray
        Capacity of each facility (shape: num_facilities)
    client_demands : np.ndarray
        Demand of each client (shape: num_clients)
        
    Returns:
    --------
    feasible : bool
        True if the solution is feasible, False otherwise
    """
    # Check if clients are only assigned to open facilities
    for assigned_facility in client_assignments:
        if facility_opened[assigned_facility] == 0:
            return False
    
    # Check capacity constraints
    for j in range(len(facility_opened)):
        if facility_opened[j] == 1:
            assigned_clients = np.where(client_assignments == j)[0]
            total_demand = client_demands[assigned_clients].sum()
            if total_demand > facility_capacities[j] + 1e-6:  # Small tolerance for floating point
                return False
    
    return True

def assign_clients_to_nearest_open_facility(service_costs: np.ndarray, facility_opened: np.ndarray,
                                            facility_capacities: np.ndarray, client_demands: np.ndarray) -> np.ndarray:
    """
    Assign each client to the nearest opened facility respecting capacity constraints (CFLP).
    Uses a fast capacity-aware greedy heuristic with demand-based sorting.
    
    Parameters:
    -----------
    service_costs : np.ndarray
        Service costs matrix (shape: num_clients × num_facilities)
    facility_opened : np.ndarray
        Binary array indicating which facilities are opened (shape: num_facilities)
    facility_capacities : np.ndarray
        Capacity of each facility (shape: num_facilities)
    client_demands : np.ndarray
        Demand of each client (shape: num_clients)
        
    Returns:
    --------
    client_assignments : np.ndarray
        Array indicating the assigned facility for each client (shape: num_clients)
    """
    num_clients = service_costs.shape[0]
    opened_indices = np.where(facility_opened == 1)[0]
    
    if len(opened_indices) == 0:
        raise ValueError("No facilities are open!")
    
    # Initialize assignments and track remaining capacity
    client_assignments = np.full(num_clients, -1, dtype=int)
    remaining_capacity = facility_capacities.copy()
    
    # Get costs only for opened facilities
    costs_to_open = service_costs[:, opened_indices]
    
    # Assign each client to nearest facility with remaining capacity
    # Process clients in natural order (no sorting) for flexibility
    for client_idx in range(num_clients):
        costs = costs_to_open[client_idx]
        sorted_fac_positions = np.argsort(costs)
        
        assigned = False
        for pos in sorted_fac_positions:
            fac_idx = opened_indices[pos]
            if remaining_capacity[fac_idx] >= client_demands[client_idx]:
                client_assignments[client_idx] = fac_idx
                remaining_capacity[fac_idx] -= client_demands[client_idx]
                assigned = True
                break
        
        if not assigned:
            # Infeasible - assign to nearest anyway (will be caught by is_feasible)
            nearest_fac = opened_indices[sorted_fac_positions[0]]
            client_assignments[client_idx] = nearest_fac
    
    return client_assignments

def copy_solution(facility_opened: np.ndarray, client_assignments: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Create a deep copy of a solution.
    
    Parameters:
    -----------
    facility_opened : np.ndarray
        Binary array indicating which facilities are opened (shape: num_facilities)
    client_assignments : np.ndarray
        Array indicating the assigned facility for each client (shape: num_clients)
        
    Returns:
    --------
    facility_opened_copy : np.ndarray
        Copy of the facility opened array
    client_assignments_copy : np.ndarray
        Copy of the client assignments array
    """
    facility_opened_copy = np.copy(facility_opened)
    client_assignments_copy = np.copy(client_assignments)
    
    return facility_opened_copy, client_assignments_copy

def greedy_initial_solution(num_clients: int, num_facilities: int, service_costs: np.ndarray,
                           facility_capacities: np.ndarray, client_demands: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Generate an initial solution using a greedy heuristic for CFLP.
    
    Parameters:
    -----------
    num_clients : int
        Number of clients
    num_facilities : int
        Number of facilities
    service_costs : np.ndarray
        Service costs matrix (shape: num_clients × num_facilities)
    facility_capacities : np.ndarray
        Capacity of each facility (shape: num_facilities)
    client_demands : np.ndarray
        Demand of each client (shape: num_clients)

    Returns:
    --------
    facility_opened : np.ndarray
        Binary array indicating which facilities are opened (shape: num_facilities)
    client_assignments : np.ndarray
        Array indicating the assigned facility for each client (shape: num_clients)
    """
    # Simple greedy approach: open facilities with best average service cost
    # until we have enough capacity to serve all clients
    
    # Calculate average service cost for each facility
    avg_service_cost_per_facility = service_costs.mean(axis=0)
    
    # Sort facilities by average service cost (ascending - lower is better)
    sorted_by_cost = np.argsort(avg_service_cost_per_facility)
    
    # Start with all facilities closed
    facility_opened = np.zeros(num_facilities, dtype=int)
    total_demand = np.sum(client_demands)
    
    # Open facilities in order of lowest average service cost until feasible
    for fac_idx in sorted_by_cost:
        facility_opened[fac_idx] = 1
        
        # Try assigning clients to open facilities
        client_assignments = assign_clients_to_nearest_open_facility(
            service_costs, facility_opened, facility_capacities, client_demands
        )
        
        # Check if feasible
        if is_feasible(facility_opened, client_assignments, facility_capacities, client_demands):
            # We have a feasible solution - stop here!
            break
    
    # If still not feasible (rare case), open all facilities
    if not is_feasible(facility_opened, client_assignments, facility_capacities, client_demands):
        facility_opened = np.ones(num_facilities, dtype=int)
        client_assignments = assign_clients_to_nearest_open_facility(
            service_costs, facility_opened, facility_capacities, client_demands
        )
    
    return facility_opened, client_assignments

print("✅ CFLP utility functions module loaded successfully.")

# Example usage: Uncomment the the following lines to load an instance and test utility functions
# file_path = "cflp_instances/001_cflp_small_easy_fac15_cli50.json"
# num_facilities, num_clients, fixed_costs, service_costs, facility_capacities, client_demands = load_cflp_instance(file_path)
# # combine to a dictionary
# instance = {
#     "num_facilities": num_facilities,
#     "num_clients": num_clients,
#     "fixed_costs": fixed_costs,
#     "service_costs": service_costs,
#     "facility_capacities": facility_capacities,
#     "client_demands": client_demands
# }
# # Generate initial solution
# facility_opened, client_assignments = greedy_initial_solution(
#     num_clients=instance["num_clients"], 
#     num_facilities=instance["num_facilities"], 
#     service_costs=instance["service_costs"],
#     facility_capacities=instance["facility_capacities"],
#     client_demands=instance["client_demands"]
# )
# # Display solution summary (just the first and last ten assignments)
# print(f"Facility opened: {facility_opened}")
# print(f"Client assignments: {client_assignments[:10]} ... {client_assignments[-10:]}")

# # Test total cost computation
# total_cost = compute_total_cost(
#     fixed_costs=instance["fixed_costs"], 
#     service_costs=instance["service_costs"], 
#     facility_opened=facility_opened, 
#     client_assignments=client_assignments
# )
# print(f"Total cost of the initial solution: {total_cost}")

# # Test feasibility check
# feasible = is_feasible(
#     facility_opened=facility_opened, 
#     client_assignments=client_assignments,
#     facility_capacities=instance["facility_capacities"],
#     client_demands=instance["client_demands"]
# )
# print(f"Is the initial solution feasible? {'Yes' if feasible else 'No'}")


✅ CFLP utility functions module loaded successfully.


---

## 5. Tabu Search (TS) Implementation

In this section, we implement the **Tabu Search (TS)** algorithm to solve the Capacitated Facility Location Problem (CFLP). Tabu Search is a trajectory-based metaheuristic that enhances local search by using a **tabu list** — a memory structure that records recently visited moves to prevent cycling and force the search to explore new areas of the solution space.

The TS algorithm consists of the following key components:
1. **Move Structures**: Define different ways to modify the current solution (open/close facilities, swap facilities, reassign clients).
2. **Tabu List**: A short-term memory that forbids reversing recent moves for a specified number of iterations (the **tabu tenure**).
3. **Aspiration Criterion**: Allows a tabu move to be accepted if it leads to a solution better than the current best known solution.
4. **Candidate Evaluation**: At each iteration, all admissible moves (non-tabu or satisfying aspiration) are evaluated and the best is selected — even if it does not improve the current solution.
5. **Stopping Criterion**: Maximum iterations or no improvement for a defined number of consecutive iterations.


### 5.1 Move Generation Structures

We define four types of moves to explore the solution space. Each move modifies the current solution and is identified by a unique key used in the tabu list:

a) **Move Type 1 — Open Facility** (`generate_open_moves`):
- Select a closed facility and open it; reassign clients to the nearest feasible open facility.
- Move key: `('open', j)` where `j` is the index of the newly opened facility.
- Reversal: `('close', j)` — closing the same facility is tabu for `tabu_tenure` iterations.

b) **Move Type 2 — Close Facility** (`generate_close_moves`):
- Select an open facility and close it (requires ≥ 2 open facilities); reassign clients.
- Move key: `('close', j)` where `j` is the index of the closed facility.
- Reversal: `('open', j)` — reopening the same facility is tabu.

c) **Move Type 3 — Swap Facilities** (`generate_swap_moves`):
- Close one open facility and simultaneously open one closed facility; reassign clients.
- Move key: `('swap', j_open, j_close)` where `j_open` is closed and `j_close` is opened.
- Reversal: `('swap', j_close, j_open)` — reversing the swap is tabu.

d) **Move Type 4 — Reassign Client** (`generate_reassign_moves`):
- Move a client from its current facility to a different open facility (capacity check required).
- Move key: `('reassign', client, old_facility, new_facility)`.
- Reversal: `('reassign', client, new_facility, old_facility)` — moving the client back is tabu.

**Key principle**: The tabu list stores the *inverse* (reversal) of each applied move, preventing it from being undone for `tabu_tenure` iterations. The **aspiration criterion** overrides the tabu status if the move leads to a solution strictly better than the best known.


In [5]:
def generate_open_moves(facility_opened: np.ndarray, service_costs: np.ndarray,
                        facility_capacities: np.ndarray, client_demands: np.ndarray,
                        max_moves: int = 20) -> List[Tuple]:
    """
    Generate move candidates by opening a closed facility (Tabu Search move type 1).

    Parameters:
    -----------
    facility_opened : np.ndarray
        Binary array indicating which facilities are opened (shape: num_facilities)
    service_costs : np.ndarray
        Service costs matrix (shape: num_clients × num_facilities)
    facility_capacities : np.ndarray
        Capacity of each facility (shape: num_facilities)
    client_demands : np.ndarray
        Demand of each client (shape: num_clients)
    max_moves : int
        Maximum number of candidate moves to generate

    Returns:
    --------
    candidates : List[Tuple]
        List of (new_facility_opened, new_client_assignments, move_key) tuples
    """
    candidates = []
    closed_indices = np.where(facility_opened == 0)[0]
    np.random.shuffle(closed_indices)

    for j in closed_indices:
        if len(candidates) >= max_moves:
            break
        new_opened = facility_opened.copy()
        new_opened[j] = 1
        new_assignments = assign_clients_to_nearest_open_facility(
            service_costs, new_opened, facility_capacities, client_demands
        )
        if is_feasible(new_opened, new_assignments, facility_capacities, client_demands):
            move_key = ('open', int(j))
            candidates.append((new_opened, new_assignments, move_key))

    return candidates


def generate_close_moves(facility_opened: np.ndarray, service_costs: np.ndarray,
                         facility_capacities: np.ndarray, client_demands: np.ndarray,
                         max_moves: int = 20) -> List[Tuple]:
    """
    Generate move candidates by closing an open facility (Tabu Search move type 2).

    Parameters:
    -----------
    facility_opened : np.ndarray
        Binary array indicating which facilities are opened (shape: num_facilities)
    service_costs : np.ndarray
        Service costs matrix (shape: num_clients × num_facilities)
    facility_capacities : np.ndarray
        Capacity of each facility (shape: num_facilities)
    client_demands : np.ndarray
        Demand of each client (shape: num_clients)
    max_moves : int
        Maximum number of candidate moves to generate

    Returns:
    --------
    candidates : List[Tuple]
        List of (new_facility_opened, new_client_assignments, move_key) tuples
    """
    candidates = []
    opened_indices = np.where(facility_opened == 1)[0]

    # Must keep at least one facility open
    if len(opened_indices) <= 1:
        return candidates

    np.random.shuffle(opened_indices)
    for j in opened_indices:
        if len(candidates) >= max_moves:
            break
        new_opened = facility_opened.copy()
        new_opened[j] = 0
        new_assignments = assign_clients_to_nearest_open_facility(
            service_costs, new_opened, facility_capacities, client_demands
        )
        if is_feasible(new_opened, new_assignments, facility_capacities, client_demands):
            move_key = ('close', int(j))
            candidates.append((new_opened, new_assignments, move_key))

    return candidates


def generate_swap_moves(facility_opened: np.ndarray, service_costs: np.ndarray,
                        facility_capacities: np.ndarray, client_demands: np.ndarray,
                        max_moves: int = 20) -> List[Tuple]:
    """
    Generate move candidates by swapping an open and a closed facility (Tabu Search move type 3).

    Parameters:
    -----------
    facility_opened : np.ndarray
        Binary array indicating which facilities are opened (shape: num_facilities)
    service_costs : np.ndarray
        Service costs matrix (shape: num_clients × num_facilities)
    facility_capacities : np.ndarray
        Capacity of each facility (shape: num_facilities)
    client_demands : np.ndarray
        Demand of each client (shape: num_clients)
    max_moves : int
        Maximum number of candidate moves to generate

    Returns:
    --------
    candidates : List[Tuple]
        List of (new_facility_opened, new_client_assignments, move_key) tuples
    """
    candidates = []
    opened_indices = np.where(facility_opened == 1)[0]
    closed_indices = np.where(facility_opened == 0)[0]

    if len(opened_indices) == 0 or len(closed_indices) == 0:
        return candidates

    np.random.shuffle(opened_indices)
    np.random.shuffle(closed_indices)

    count = 0
    for j_open in opened_indices:
        for j_close in closed_indices[:4]:
            if count >= max_moves:
                break
            new_opened = facility_opened.copy()
            new_opened[j_open] = 0
            new_opened[j_close] = 1
            new_assignments = assign_clients_to_nearest_open_facility(
                service_costs, new_opened, facility_capacities, client_demands
            )
            if is_feasible(new_opened, new_assignments, facility_capacities, client_demands):
                move_key = ('swap', int(j_open), int(j_close))
                candidates.append((new_opened, new_assignments, move_key))
                count += 1
        if count >= max_moves:
            break

    return candidates


def generate_reassign_moves(facility_opened: np.ndarray, client_assignments: np.ndarray,
                             service_costs: np.ndarray, facility_capacities: np.ndarray,
                             client_demands: np.ndarray, max_moves: int = 20) -> List[Tuple]:
    """
    Generate move candidates by reassigning a client to a different open facility
    (Tabu Search move type 4).

    Parameters:
    -----------
    facility_opened : np.ndarray
        Binary array indicating which facilities are opened (shape: num_facilities)
    client_assignments : np.ndarray
        Array indicating the assigned facility for each client (shape: num_clients)
    service_costs : np.ndarray
        Service costs matrix (shape: num_clients × num_facilities)
    facility_capacities : np.ndarray
        Capacity of each facility (shape: num_facilities)
    client_demands : np.ndarray
        Demand of each client (shape: num_clients)
    max_moves : int
        Maximum number of candidate moves to generate

    Returns:
    --------
    candidates : List[Tuple]
        List of (new_facility_opened, new_client_assignments, move_key) tuples
    """
    candidates = []
    opened_indices = np.where(facility_opened == 1)[0]

    if len(opened_indices) <= 1:
        return candidates

    client_indices = list(range(len(client_assignments)))
    np.random.shuffle(client_indices)

    for i in client_indices:
        if len(candidates) >= max_moves:
            break
        current_fac = client_assignments[i]
        alternatives = opened_indices[opened_indices != current_fac]
        np.random.shuffle(alternatives)

        for j in alternatives[:3]:
            new_assignments = client_assignments.copy()
            new_assignments[i] = j
            if is_feasible(facility_opened, new_assignments, facility_capacities, client_demands):
                move_key = ('reassign', int(i), int(current_fac), int(j))
                candidates.append((facility_opened.copy(), new_assignments, move_key))
                break  # One candidate per client

    return candidates


def get_inverse_move(move_key: tuple) -> tuple:
    """
    Compute the inverse (reversal) of a Tabu Search move.
    The inverse is added to the tabu list to prevent undoing the move.

    Parameters:
    -----------
    move_key : tuple
        The move key (type, indices...)

    Returns:
    --------
    inverse_key : tuple
        The inverse move key
    """
    move_type = move_key[0]
    if move_type == 'open':
        return ('close', move_key[1])
    elif move_type == 'close':
        return ('open', move_key[1])
    elif move_type == 'swap':
        return ('swap', move_key[2], move_key[1])    # reverse the swap
    elif move_type == 'reassign':
        return ('reassign', move_key[1], move_key[3], move_key[2])   # swap old/new facility
    return move_key


print("✅ Tabu Search move generation modules loaded successfully.")


✅ Tabu Search move generation modules loaded successfully.


### 5.2 Tabu List and Candidate Selection

The **tabu list** is the core memory structure of Tabu Search. It stores the *inverse* of recently applied moves and forbids those inverse moves for a fixed number of iterations, called the **tabu tenure**.

- **Tabu tenure**: Controls how long a move stays forbidden (typically 5–15 iterations). A longer tenure increases diversification but may prevent useful moves.
- **Aspiration criterion**: A tabu move can be accepted if its resulting cost is strictly better than the best solution found so far (`cost < best_cost`). This prevents the tabu list from blocking globally improving moves.
- **Move selection**: At each iteration, *all* candidate moves (from all four move types) are evaluated. The best admissible move is selected (non-tabu, or tabu but satisfying aspiration).

The function `select_best_candidate` implements this logic by iterating over all generated candidates, checking tabu status, and applying the aspiration criterion.


In [6]:
def generate_all_candidates(current_solution: Tuple[np.ndarray, np.ndarray],
                            service_costs: np.ndarray, facility_capacities: np.ndarray,
                            client_demands: np.ndarray,
                            max_candidates: int = 20) -> List[Tuple]:
    """
    Generate all candidate moves for the current solution by applying all four move types.

    Parameters:
    -----------
    current_solution : Tuple[np.ndarray, np.ndarray]
        Current solution as (facility_opened, client_assignments)
    service_costs : np.ndarray
        Service costs matrix (shape: num_clients × num_facilities)
    facility_capacities : np.ndarray
        Capacity of each facility (shape: num_facilities)
    client_demands : np.ndarray
        Demand of each client (shape: num_clients)
    max_candidates : int
        Maximum number of candidates per move type

    Returns:
    --------
    all_candidates : List[Tuple]
        Combined list of (new_facility_opened, new_client_assignments, move_key) tuples
    """
    facility_opened, client_assignments = current_solution

    all_candidates = []
    all_candidates.extend(generate_open_moves(
        facility_opened, service_costs, facility_capacities, client_demands, max_candidates))
    all_candidates.extend(generate_close_moves(
        facility_opened, service_costs, facility_capacities, client_demands, max_candidates))
    all_candidates.extend(generate_swap_moves(
        facility_opened, service_costs, facility_capacities, client_demands, max_candidates))
    all_candidates.extend(generate_reassign_moves(
        facility_opened, client_assignments, service_costs, facility_capacities,
        client_demands, max_candidates))

    return all_candidates


def select_best_candidate(candidates: List[Tuple], tabu_list: dict, iteration: int,
                          best_cost: float, fixed_costs: np.ndarray,
                          service_costs: np.ndarray) -> Tuple:
    """
    Select the best admissible candidate move using tabu status and aspiration criterion.

    A candidate is admissible if:
    - Its move key is NOT in the tabu list (non-tabu), OR
    - Its resulting cost is strictly less than the current best known cost (aspiration).

    Parameters:
    -----------
    candidates : List[Tuple]
        List of (new_facility_opened, new_client_assignments, move_key) tuples
    tabu_list : dict
        Dictionary mapping inverse_move_key -> expiry_iteration
    iteration : int
        Current iteration number
    best_cost : float
        Best cost found so far (used for aspiration criterion)
    fixed_costs : np.ndarray
        Fixed costs for each facility
    service_costs : np.ndarray
        Service costs matrix

    Returns:
    --------
    best_candidate : Tuple or None
        (new_facility_opened, new_client_assignments, move_key, cost) or None if no admissible move
    """
    best_selected = None
    best_selected_cost = float('inf')

    for new_opened, new_assignments, move_key in candidates:
        cost = compute_total_cost(fixed_costs, service_costs, new_opened, new_assignments)

        # Check tabu status using the inverse move key
        inverse_key = get_inverse_move(move_key)
        is_tabu = (inverse_key in tabu_list) and (tabu_list[inverse_key] > iteration)

        # Accept if non-tabu OR aspiration criterion satisfied
        if not is_tabu or cost < best_cost:
            if cost < best_selected_cost:
                best_selected_cost = cost
                best_selected = (new_opened, new_assignments, move_key, cost)

    return best_selected


print("✅ Tabu List candidate selection module loaded successfully.")


✅ Tabu List candidate selection module loaded successfully.


### 5.3 Initial Solution Construction

Tabu Search starts from an initial solution constructed by a **greedy heuristic**:
1. Begin with all facilities closed.
2. Iteratively open the facility that provides the largest reduction in total cost.
3. Assign all clients to their nearest feasible open facility.

This greedy solution is then passed directly to the Tabu Search main loop as the starting point. Unlike VNS, Tabu Search does not require a separate local search phase after initialization — it performs guided neighborhood exploration from the very first iteration using the tabu mechanism.


In [7]:
def tabu_initial_solution(instance: dict) -> Tuple[np.ndarray, np.ndarray]:
    """
    Construct an initial solution using the greedy heuristic for Tabu Search.
    
    Delegates to the existing greedy_initial_solution function and returns
    the solution tuple (facility_opened, client_assignments).
    
    Parameters:
    -----------
    instance : dict
        CFLP instance dictionary with keys: num_clients, num_facilities,
        service_costs, facility_capacities, client_demands
    
    Returns:
    --------
    initial_solution : Tuple[np.ndarray, np.ndarray]
        (facility_opened, client_assignments)
    """
    return greedy_initial_solution(
        num_clients=instance["num_clients"],
        num_facilities=instance["num_facilities"],
        service_costs=instance["service_costs"],
        facility_capacities=instance["facility_capacities"],
        client_demands=instance["client_demands"]
    )

print("✅ Tabu Search initialization module loaded successfully.")


✅ Tabu Search initialization module loaded successfully.


---

### 5.4 Tabu Search — MAIN ALGORITHM

The complete Tabu Search algorithm for CFLP orchestrates all components:

**Pseudocode:**
```
1. s ← GreedyInitialSolution()
2. s* ← s  (best solution found)
3. TabuList ← {}
4. iteration ← 0, no_improve ← 0

5. WHILE iteration < max_iterations AND no_improve < max_no_improve:
   a. Generate all candidate moves from s (open, close, swap, reassign)
   b. Evaluate cost of each candidate
   c. Select best admissible candidate s' (non-tabu OR aspiration satisfied)
   d. Apply the move: s ← s'
   e. Add inverse(move) to TabuList with expiry = iteration + tabu_tenure
   f. IF cost(s') < cost(s*):
          s* ← s'   (new best found)
          no_improve ← 0
      ELSE:
          no_improve ← no_improve + 1
   g. iteration ← iteration + 1

6. RETURN s*, trajectory
```

**Key parameters:**
- `max_iterations`: Maximum total iterations (default: 200)
- `tabu_tenure`: How many iterations a move stays forbidden (default: 10)
- `max_candidates`: Number of candidates generated per move type per iteration (default: 20)
- `max_no_improve`: Stop early if no improvement for this many consecutive iterations (default: 40)

**Important**: Tabu Search always moves to the best admissible neighbor, even if it is worse than the current solution. This allows escaping local optima. The tabu list prevents cycling.


In [8]:
def tabu_search_solve_cflp(instance: dict,
                           max_iterations: int = 200,
                           tabu_tenure: int = 10,
                           max_candidates: int = 20,
                           max_no_improve: int = 40) -> Tuple[Tuple[np.ndarray, np.ndarray], Tuple[List[float], List[int]]]:
    """
    Tabu Search for the Capacitated Facility Location Problem (CFLP).

    At each iteration, all feasible move candidates (open, close, swap, reassign) are
    generated from the current solution. The best admissible move is selected — either
    non-tabu, or tabu but satisfying the aspiration criterion (improves best known cost).
    The inverse of the applied move is added to the tabu list to prevent immediate reversal.

    Parameters:
    -----------
    instance : dict
        CFLP instance dictionary with keys:
            num_facilities, num_clients, fixed_costs, service_costs,
            facility_capacities, client_demands
    max_iterations : int
        Maximum number of Tabu Search iterations (default: 200)
    tabu_tenure : int
        Number of iterations a move's inverse remains forbidden (default: 10)
    max_candidates : int
        Maximum candidates generated per move type per iteration (default: 20)
    max_no_improve : int
        Stop early if no improvement for this many consecutive iterations (default: 40)

    Returns:
    --------
    best_solution : Tuple[np.ndarray, np.ndarray]
        Best solution found as (facility_opened, client_assignments)
    trajectory_data : Tuple[List[float], List[int]]
        (trajectory, trajectory_iterations) — costs and iterations where improvements occurred
    """
    fixed_costs = instance["fixed_costs"]
    service_costs = instance["service_costs"]
    facility_capacities = instance["facility_capacities"]
    client_demands = instance["client_demands"]

    # ── Step 1: Greedy initial solution ──────────────────────────────────────
    current_solution = tabu_initial_solution(instance)
    current_cost = compute_total_cost(
        fixed_costs, service_costs, current_solution[0], current_solution[1]
    )

    best_solution = current_solution
    best_cost = current_cost
    trajectory = [best_cost]
    trajectory_iterations = [0]

    # ── Step 2: Tabu list (inverse_move_key → expiry iteration) ──────────────
    tabu_list: dict = {}

    no_improve_count = 0
    print(f"Starting Tabu Search for CFLP (tenure={tabu_tenure}, max_iter={max_iterations})...")

    # ── Step 3: Main loop ─────────────────────────────────────────────────────
    for iteration in range(1, max_iterations + 1):

        # 3a. Generate all candidate moves
        all_candidates = generate_all_candidates(
            current_solution, service_costs, facility_capacities, client_demands, max_candidates
        )

        if not all_candidates:
            print(f"  No candidates generated at iteration {iteration}. Stopping.")
            break

        # 3b. Select best admissible candidate (tabu status + aspiration)
        selected = select_best_candidate(
            all_candidates, tabu_list, iteration, best_cost, fixed_costs, service_costs
        )

        if selected is None:
            # All moves are tabu — extremely rare; skip iteration
            no_improve_count += 1
            if no_improve_count >= max_no_improve:
                break
            continue

        new_opened, new_assignments, move_key, new_cost = selected

        # 3c. Apply move: update current solution
        current_solution = (new_opened, new_assignments)
        current_cost = new_cost

        # 3d. Add inverse of the applied move to tabu list
        inverse_key = get_inverse_move(move_key)
        tabu_list[inverse_key] = iteration + tabu_tenure

        # Optional: clean up expired tabu entries to keep memory bounded
        tabu_list = {k: v for k, v in tabu_list.items() if v > iteration}

        # 3e. Update best solution
        if new_cost < best_cost:
            best_solution = current_solution
            best_cost = new_cost
            trajectory.append(best_cost)
            trajectory_iterations.append(iteration)
            no_improve_count = 0
        else:
            no_improve_count += 1

        if no_improve_count >= max_no_improve:
            break

    print(f"Tabu Search completed after {iteration} iterations.")
    print(f"Best cost: {best_cost:.2f}")

    return best_solution, (trajectory, trajectory_iterations)


print("✅ Tabu Search (TS) module loaded successfully.")

# Test Tabu Search (uncomment to run)
# best_solution, trajectory = tabu_search_solve_cflp(
#     instance=instance,
#     max_iterations=200,
#     tabu_tenure=10,
#     max_candidates=20,
#     max_no_improve=40
# )
# print(f"Final solution cost: {trajectory[0][-1]:.2f}")


✅ Tabu Search (TS) module loaded successfully.


---

## 6. Genetic Algorithm (GA) Implementation

In this section, we implement a **Genetic Algorithm (GA)** to solve the CFLP. GAs are population-based metaheuristics inspired by Darwinian natural evolution. Instead of a single trajectory, the GA evolves a *population* of candidate solutions through generations using biologically-inspired operators.

### 6.1 Chromosome Encoding and Population

**Encoding**: Each individual (chromosome) is a binary vector of length `num_facilities`:
- `chromosome[j] = 1` if facility `j` is open, `0` otherwise.
- Client assignments are derived deterministically from the chromosome using the greedy nearest-feasible-facility assignment.

**Population initialization**: Combines one greedy solution with `population_size - 1` random feasible solutions. Random individuals are repaired if infeasible.

### 6.2 Selection — Binary Tournament

Two individuals are drawn at random; the one with lower cost (higher fitness) wins and becomes a parent. This is repeated twice to select two parents per crossover.

### 6.3 Crossover — Uniform Crossover

Each gene position is independently swapped between parents with probability 0.5:
- For each facility index `j`, if `random() < 0.5`: child1[j] = parent2[j], child2[j] = parent1[j].
- This produces two children with mixed facility decisions from both parents.
- Applied with probability `crossover_rate` (default: 0.85); otherwise children are clones.

### 6.4 Mutation — Bit-Flip

Each gene in a chromosome is independently flipped (0→1 or 1→0) with probability `mutation_rate` (default: 0.1). This introduces random variation and prevents premature convergence.

### 6.5 Repair Operator

After crossover and mutation, a chromosome may be infeasible (e.g., all facilities closed, or total capacity insufficient). The **repair operator**:
1. If no facility is open → open the facility with the highest capacity.
2. Iteratively open the highest-capacity closed facility until the assignment becomes feasible.

### 6.6 Survivor Selection — Elitism + Generational Replacement

- **Elitism**: The top `elitism` individuals (default: 2) are copied unchanged to the next generation.
- **Generational replacement**: The rest of the next generation is filled from offspring, sorted by cost.

### 6.7 Stopping Criterion

The GA stops after `max_generations` generations or when no improvement is found for `max_no_improve` consecutive generations.


In [9]:
# ==============================================================================
# 6.1 Chromosome Repair and Population Initialization
# ==============================================================================

def repair_chromosome(facility_opened: np.ndarray, service_costs: np.ndarray,
                      facility_capacities: np.ndarray, client_demands: np.ndarray
                      ) -> Tuple[np.ndarray, np.ndarray]:
    """
    Repair an infeasible chromosome by opening additional facilities until feasible.

    Parameters:
    -----------
    facility_opened : np.ndarray
        Binary facility decision vector (may be infeasible)
    service_costs : np.ndarray
        Service costs matrix (shape: num_clients × num_facilities)
    facility_capacities : np.ndarray
        Capacity of each facility
    client_demands : np.ndarray
        Demand of each client

    Returns:
    --------
    repaired_opened : np.ndarray
        Repaired binary facility decision vector (feasible)
    assignments : np.ndarray
        Client assignments derived from repaired_opened
    """
    repaired = facility_opened.copy()

    # Ensure at least one facility is open
    if repaired.sum() == 0:
        j = int(np.argmax(facility_capacities))
        repaired[j] = 1

    assignments = assign_clients_to_nearest_open_facility(
        service_costs, repaired, facility_capacities, client_demands
    )

    # Open more facilities if still infeasible (capacity insufficient)
    max_attempts = len(repaired)
    for _ in range(max_attempts):
        if is_feasible(repaired, assignments, facility_capacities, client_demands):
            break
        closed = np.where(repaired == 0)[0]
        if len(closed) == 0:
            break
        # Open the closed facility with the largest capacity
        j = closed[int(np.argmax(facility_capacities[closed]))]
        repaired[j] = 1
        assignments = assign_clients_to_nearest_open_facility(
            service_costs, repaired, facility_capacities, client_demands
        )

    return repaired, assignments


def initialize_ga_population(population_size: int, num_facilities: int,
                             service_costs: np.ndarray, facility_capacities: np.ndarray,
                             client_demands: np.ndarray, fixed_costs: np.ndarray,
                             num_clients: int) -> List[np.ndarray]:
    """
    Initialize a diverse population of chromosomes for the Genetic Algorithm.

    Includes one greedy solution and (population_size - 1) randomly generated,
    repaired individuals.

    Parameters:
    -----------
    population_size : int
        Number of individuals in the population
    num_facilities : int
        Number of potential facilities
    service_costs, facility_capacities, client_demands, fixed_costs : np.ndarray
        CFLP instance data arrays
    num_clients : int
        Number of clients

    Returns:
    --------
    population : List[np.ndarray]
        List of binary chromosome arrays (facility decision vectors)
    """
    population = []

    # Seed population with the greedy solution
    greedy_sol = greedy_initial_solution(
        num_clients=num_clients,
        num_facilities=num_facilities,
        service_costs=service_costs,
        facility_capacities=facility_capacities,
        client_demands=client_demands
    )
    population.append(greedy_sol[0].copy())

    # Add random feasible individuals
    attempts = 0
    while len(population) < population_size and attempts < population_size * 10:
        attempts += 1
        num_open = np.random.randint(1, num_facilities + 1)
        indices = np.random.choice(num_facilities, num_open, replace=False)
        chromosome = np.zeros(num_facilities, dtype=int)
        chromosome[indices] = 1
        repaired, _ = repair_chromosome(chromosome, service_costs, facility_capacities, client_demands)
        population.append(repaired)

    # Fill remaining slots with greedy clones if random generation failed
    while len(population) < population_size:
        population.append(greedy_sol[0].copy())

    return population


def evaluate_chromosome(chromosome: np.ndarray, fixed_costs: np.ndarray,
                         service_costs: np.ndarray, facility_capacities: np.ndarray,
                         client_demands: np.ndarray) -> float:
    """
    Evaluate the fitness (total cost) of a chromosome.

    Parameters:
    -----------
    chromosome : np.ndarray
        Binary facility decision vector
    fixed_costs, service_costs, facility_capacities, client_demands : np.ndarray
        CFLP instance data

    Returns:
    --------
    cost : float
        Total cost (inf if infeasible)
    """
    assignments = assign_clients_to_nearest_open_facility(
        service_costs, chromosome, facility_capacities, client_demands
    )
    if not is_feasible(chromosome, assignments, facility_capacities, client_demands):
        return float('inf')
    return compute_total_cost(fixed_costs, service_costs, chromosome, assignments)


# ==============================================================================
# 6.2 Selection, Crossover, and Mutation Operators
# ==============================================================================

def tournament_selection(population: List[np.ndarray], fitness: List[float],
                         tournament_size: int = 3) -> np.ndarray:
    """
    Select one individual using binary tournament selection.

    Randomly samples `tournament_size` individuals and returns the one with
    the lowest cost (best fitness).

    Parameters:
    -----------
    population : List[np.ndarray]
        Current population of chromosomes
    fitness : List[float]
        Corresponding fitness (cost) values
    tournament_size : int
        Number of individuals in each tournament (default: 3)

    Returns:
    --------
    winner : np.ndarray
        Copy of the winning chromosome
    """
    indices = np.random.choice(len(population), min(tournament_size, len(population)), replace=False)
    winner_idx = int(indices[np.argmin([fitness[i] for i in indices])])
    return population[winner_idx].copy()


def uniform_crossover(parent1: np.ndarray, parent2: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Apply uniform crossover on two parent chromosomes.

    Each gene is independently inherited from either parent with equal probability (0.5).

    Parameters:
    -----------
    parent1, parent2 : np.ndarray
        Parent binary chromosomes

    Returns:
    --------
    child1, child2 : Tuple[np.ndarray, np.ndarray]
        Two offspring chromosomes
    """
    mask = np.random.random(len(parent1)) < 0.5
    child1 = parent1.copy()
    child2 = parent2.copy()
    child1[mask] = parent2[mask]
    child2[mask] = parent1[mask]
    return child1, child2


def mutate_chromosome(chromosome: np.ndarray, mutation_rate: float) -> np.ndarray:
    """
    Apply bit-flip mutation to a chromosome.

    Each gene is independently flipped (0→1 or 1→0) with probability `mutation_rate`.

    Parameters:
    -----------
    chromosome : np.ndarray
        Binary chromosome
    mutation_rate : float
        Probability of flipping each gene (default: 0.1)

    Returns:
    --------
    mutated : np.ndarray
        Mutated binary chromosome
    """
    mutated = chromosome.copy()
    flip_mask = np.random.random(len(mutated)) < mutation_rate
    mutated[flip_mask] = 1 - mutated[flip_mask]
    return mutated


# ==============================================================================
# 6.3 Genetic Algorithm — Main Algorithm
# ==============================================================================

def genetic_algorithm_solve_cflp(instance: dict,
                                  population_size: int = 30,
                                  max_generations: int = 100,
                                  crossover_rate: float = 0.85,
                                  mutation_rate: float = 0.10,
                                  tournament_size: int = 3,
                                  elitism: int = 2,
                                  max_no_improve: int = 30
                                  ) -> Tuple[Tuple[np.ndarray, np.ndarray], Tuple[List[float], List[int]]]:
    """
    Genetic Algorithm for the Capacitated Facility Location Problem (CFLP).

    Evolves a population of binary chromosomes (facility open/close decisions) through
    selection, uniform crossover, bit-flip mutation, feasibility repair, and elitist
    generational replacement.

    Parameters:
    -----------
    instance : dict
        CFLP instance dictionary with keys:
            num_facilities, num_clients, fixed_costs, service_costs,
            facility_capacities, client_demands
    population_size : int
        Number of individuals per generation (default: 30)
    max_generations : int
        Maximum number of generations (default: 100)
    crossover_rate : float
        Probability of applying crossover to a pair of parents (default: 0.85)
    mutation_rate : float
        Per-gene flip probability during mutation (default: 0.10)
    tournament_size : int
        Number of contestants in each tournament selection (default: 3)
    elitism : int
        Number of best individuals carried unchanged to the next generation (default: 2)
    max_no_improve : int
        Stop early if no improvement found for this many consecutive generations (default: 30)

    Returns:
    --------
    best_solution : Tuple[np.ndarray, np.ndarray]
        Best solution found as (facility_opened, client_assignments)
    trajectory_data : Tuple[List[float], List[int]]
        (trajectory, trajectory_generations) — costs and generations where improvements occurred
    """
    num_facilities = instance["num_facilities"]
    num_clients = instance["num_clients"]
    fixed_costs = instance["fixed_costs"]
    service_costs = instance["service_costs"]
    facility_capacities = instance["facility_capacities"]
    client_demands = instance["client_demands"]

    # ── Step 1: Initialize population ────────────────────────────────────────
    population = initialize_ga_population(
        population_size, num_facilities, service_costs,
        facility_capacities, client_demands, fixed_costs, num_clients
    )
    fitness = [evaluate_chromosome(ind, fixed_costs, service_costs,
                                   facility_capacities, client_demands)
               for ind in population]

    best_idx = int(np.argmin(fitness))
    best_fitness = fitness[best_idx]
    best_chromosome = population[best_idx].copy()

    trajectory = [best_fitness]
    trajectory_generations = [0]
    no_improve_count = 0

    print(f"Starting Genetic Algorithm for CFLP "
          f"(pop={population_size}, max_gen={max_generations})...")

    # ── Step 2: Evolutionary loop ─────────────────────────────────────────────
    for generation in range(1, max_generations + 1):

        # Elitism: preserve top individuals unchanged
        sorted_indices = np.argsort(fitness)
        elites = [population[i].copy() for i in sorted_indices[:elitism]]
        elite_fitness = [fitness[i] for i in sorted_indices[:elitism]]

        # Generate offspring to fill remaining slots
        offspring = []
        offspring_fitness = []

        while len(offspring) < population_size - elitism:
            # Selection
            p1 = tournament_selection(population, fitness, tournament_size)
            p2 = tournament_selection(population, fitness, tournament_size)

            # Crossover
            if np.random.random() < crossover_rate:
                c1, c2 = uniform_crossover(p1, p2)
            else:
                c1, c2 = p1.copy(), p2.copy()

            # Mutation
            c1 = mutate_chromosome(c1, mutation_rate)
            c2 = mutate_chromosome(c2, mutation_rate)

            # Repair infeasible offspring
            c1, _ = repair_chromosome(c1, service_costs, facility_capacities, client_demands)
            c2, _ = repair_chromosome(c2, service_costs, facility_capacities, client_demands)

            # Evaluate
            f1 = evaluate_chromosome(c1, fixed_costs, service_costs,
                                     facility_capacities, client_demands)
            f2 = evaluate_chromosome(c2, fixed_costs, service_costs,
                                     facility_capacities, client_demands)

            offspring.extend([c1, c2])
            offspring_fitness.extend([f1, f2])

        # Trim to exact size needed
        offspring = offspring[:population_size - elitism]
        offspring_fitness = offspring_fitness[:population_size - elitism]

        # New population: elites + offspring
        population = elites + offspring
        fitness = elite_fitness + offspring_fitness

        # Update best solution
        current_best_idx = int(np.argmin(fitness))
        if fitness[current_best_idx] < best_fitness:
            best_fitness = fitness[current_best_idx]
            best_chromosome = population[current_best_idx].copy()
            trajectory.append(best_fitness)
            trajectory_generations.append(generation)
            no_improve_count = 0
        else:
            no_improve_count += 1

        if no_improve_count >= max_no_improve:
            break

    # ── Step 3: Derive final solution from best chromosome ────────────────────
    best_assignments = assign_clients_to_nearest_open_facility(
        service_costs, best_chromosome, facility_capacities, client_demands
    )
    best_solution = (best_chromosome, best_assignments)

    print(f"Genetic Algorithm completed after {generation} generations.")
    print(f"Best cost: {best_fitness:.2f}")

    return best_solution, (trajectory, trajectory_generations)


print("✅ Genetic Algorithm (GA) module loaded successfully.")

# Test GA (uncomment to run)
# best_solution, (trajectory, traj_gen) = genetic_algorithm_solve_cflp(
#     instance=instance,
#     population_size=30,
#     max_generations=100
# )
# print(f"GA best cost: {trajectory[-1]:.2f} after {traj_gen[-1]} generations")


✅ Genetic Algorithm (GA) module loaded successfully.


---

## 7. Exact Solution using Gurobi

In this section, we implement the exact solution for the Capacitated Facility Location Problem (CFLP) using the Gurobi optimization solver. Gurobi solves the MIP formulation to **guaranteed optimality** (within the given time limit) using Branch-and-Bound with cutting planes.

**Role in this assignment**: Gurobi provides the **optimal reference solution** against which both Tabu Search and Genetic Algorithm are benchmarked. The **optimality gap** for each metaheuristic is computed as:

$$\text{Gap}(\%) = \frac{\text{TS/GA cost} - \text{Gurobi cost}}{\text{Gurobi cost}} \times 100$$


In [10]:
def gurobi_solve_cflp(instance: dict, time_limit: int = 300, check_size_limit: bool = False) -> Tuple[np.ndarray, np.ndarray, float]:
    """
    Solve the Capacitated Facility Location Problem (CFLP) using Gurobi optimization solver.
    
    Parameters:
    -----------
    instance : dict
        Dictionary containing the CFLP instance data with keys:
            - num_facilities
            - num_clients
            - fixed_costs
            - service_costs
            - facility_capacities
            - client_demands
    time_limit : int
        Time limit for the solver in seconds (default: 300 seconds)
    check_size_limit : bool
        If True, check if instance exceeds free Gurobi limits (2000 vars, 2000 constraints)
        and skip solving if it does (default: False for academic/unlimited version)
        
    Returns:
    --------
    facility_opened : np.ndarray
        Binary array indicating which facilities are opened (shape: num_facilities)
    client_assignments : np.ndarray
        Array indicating the assigned facility for each client (shape: num_clients)
    optimal_cost : float
        Optimal total cost of the solution
        
    Raises:
    -------
    ValueError
        If instance exceeds free Gurobi limits (when check_size_limit=True)
    """
    from gurobipy import Model, GRB, quicksum

    num_facilities = instance["num_facilities"]
    num_clients = instance["num_clients"]
    fixed_costs = instance["fixed_costs"]
    service_costs = instance["service_costs"]
    facility_capacities = instance["facility_capacities"]
    client_demands = instance["client_demands"]

    # Check size limits for free Gurobi version
    if check_size_limit:
        num_variables = num_facilities + (num_clients * num_facilities)  # y + x variables
        num_constraints = num_clients + (num_clients * num_facilities) + num_facilities  # assignment + linking + capacity
        
        if num_variables > 2000 or num_constraints > 2000:
            raise ValueError(
                f"Instance exceeds free Gurobi limits: "
                f"{num_variables} variables (max 2000), "
                f"{num_constraints} constraints (max 2000). "
                f"Skipping this instance."
            )

    # Create Gurobi model
    model = Model("Capacitated_Facility_Location_Problem")
    model.setParam('OutputFlag', 0)  # Suppress output
    model.setParam('TimeLimit', time_limit)
    
    # Decision variables
    y = model.addVars(num_facilities, vtype=GRB.BINARY, name="y")  # Facility open variables
    x = model.addVars(num_clients, num_facilities, vtype=GRB.BINARY, name="x")  # Client assignment variables
    
    # Objective function: Minimize total cost
    model.setObjective(
        quicksum(fixed_costs[j] * y[j] for j in range(num_facilities)) +
        quicksum(service_costs[i][j] * x[i, j] for i in range(num_clients) for j in range(num_facilities)),
        GRB.MINIMIZE
    )
    
    # Constraints: Each client must be assigned to exactly one facility
    for i in range(num_clients):
        model.addConstr(quicksum(x[i, j] for j in range(num_facilities)) == 1, name=f"Client_{i}_assignment")
    
    # Constraints: Clients can only be assigned to opened facilities
    for i in range(num_clients):
        for j in range(num_facilities):
            model.addConstr(x[i, j] <= y[j], name=f"Client_{i}_to_Facility_{j}_open")
    
    # NEW: Capacity constraints for CFLP
    for j in range(num_facilities):
        model.addConstr(
            quicksum(client_demands[i] * x[i, j] for i in range(num_clients)) <= facility_capacities[j] * y[j],
            name=f"Facility_{j}_capacity"
        )
    
    # Optimize
    model.optimize()

    # Check if solution was found
    if model.status == GRB.OPTIMAL or model.status == GRB.TIME_LIMIT:
        # Extract solution
        facility_opened = np.array([int(round(y[j].X)) for j in range(num_facilities)])
        client_assignments = np.array([int(round(sum(j * x[i, j].X for j in range(num_facilities)))) 
                                        for i in range(num_clients)])
        optimal_cost = model.objVal
        
        return facility_opened, client_assignments, optimal_cost
    else:
        raise Exception(f"Gurobi optimization failed with status: {model.status}")

print("✅ Gurobi module loaded successfully.")

# Test Gurobi (uncomment the following lines to run a test)
# gurobi_facilities, gurobi_assignments, gurobi_cost = gurobi_solve_cflp(instance, time_limit=60)
# print(f"Gurobi optimal cost: {gurobi_cost:.2f}")
# print(f"Facilities opened: {gurobi_facilities}")
# print(f"Number of facilities opened: {gurobi_facilities.sum()}")

✅ Gurobi module loaded successfully.


In [11]:
def compare_solutions(instance: dict,
                      gurobi_solution: Tuple[np.ndarray, np.ndarray, float],
                      ts_solution: Tuple[Tuple[np.ndarray, np.ndarray], Tuple[List[float], List[int]]],
                      ga_solution: Tuple[Tuple[np.ndarray, np.ndarray], Tuple[List[float], List[int]]]):
    """
    Compare Gurobi, Tabu Search, and Genetic Algorithm solutions side by side.

    Parameters:
    -----------
    instance : dict
        CFLP instance dictionary
    gurobi_solution : Tuple[np.ndarray, np.ndarray, float]
        Gurobi result as (facility_opened, client_assignments, optimal_cost)
    ts_solution : Tuple[Tuple[np.ndarray, np.ndarray], Tuple[List[float], List[int]]]
        TS result as ((facility_opened, client_assignments), (trajectory, trajectory_iterations))
    ga_solution : Tuple[Tuple[np.ndarray, np.ndarray], Tuple[List[float], List[int]]]
        GA result as ((facility_opened, client_assignments), (trajectory, trajectory_generations))
    """
    fixed_costs = instance['fixed_costs']
    service_costs = instance['service_costs']

    gurobi_facilities, gurobi_assignments, gurobi_cost = gurobi_solution

    (ts_facilities, ts_assignments), (ts_traj, ts_traj_iter) = ts_solution
    ts_cost = compute_total_cost(fixed_costs, service_costs, ts_facilities, ts_assignments)
    ts_gap = ((ts_cost - gurobi_cost) / gurobi_cost) * 100
    ts_total_iters = ts_traj_iter[-1] if ts_traj_iter else 0
    ts_improvements = len(ts_traj) - 1

    (ga_facilities, ga_assignments), (ga_traj, ga_traj_gen) = ga_solution
    ga_cost = compute_total_cost(fixed_costs, service_costs, ga_facilities, ga_assignments)
    ga_gap = ((ga_cost - gurobi_cost) / gurobi_cost) * 100
    ga_total_gens = ga_traj_gen[-1] if ga_traj_gen else 0
    ga_improvements = len(ga_traj) - 1

    print("=" * 70)
    print("SOLUTION COMPARISON: Gurobi vs Tabu Search vs Genetic Algorithm")
    print("=" * 70)

    print(f"\n{'Method':<20} {'Cost':>14} {'Facilities':>10} {'Gap (%)':>10}")
    print("-" * 56)
    print(f"{'Gurobi (Exact)':<20} {gurobi_cost:>14.2f} {int(gurobi_facilities.sum()):>10} {'0.00':>10}")
    print(f"{'Tabu Search':<20} {ts_cost:>14.2f} {int(ts_facilities.sum()):>10} {ts_gap:>10.2f}")
    print(f"{'Genetic Algorithm':<20} {ga_cost:>14.2f} {int(ga_facilities.sum()):>10} {ga_gap:>10.2f}")

    print(f"\nTabu Search details: {ts_total_iters} iterations, {ts_improvements} improvements found")
    print(f"Genetic Algorithm details: {ga_total_gens} generations, {ga_improvements} improvements found")
    print("=" * 70)


print("✅ Comparison module loaded successfully.")


✅ Comparison module loaded successfully.


---

## 8. Experiments and Comparison

In this section, we conduct a series of experiments to evaluate and compare the performance of three approaches for solving the CFLP:

1. **Gurobi** (exact MIP solver) — provides optimal solutions as ground truth.
2. **Tabu Search (TS)** — trajectory-based metaheuristic using a tabu list and aspiration criterion.
3. **Genetic Algorithm (GA)** — population-based metaheuristic using selection, crossover, mutation, and repair.

Experiments assess:
- **Solution quality**: Optimality gap (%) relative to Gurobi for both TS and GA.
- **Computational time**: Solver time for each of the three methods.
- **Convergence behavior**: How quickly TS and GA improve over iterations/generations.
- **Scalability**: How each method's performance changes across small, medium, and large instances.
- **Robustness**: Consistency across different difficulty levels (easy, moderate, expensive).


In [12]:
def plot_convergence_ts_ga(ts_trajectory: List[float], ts_iterations: List[int],
                           ga_trajectory: List[float], ga_generations: List[int],
                           optimal_cost: float, instance_name: str,
                           output_folder: str, size: str, difficulty: str):
    """
    Plot and save the convergence trajectories of Tabu Search and Genetic Algorithm
    for a single instance, with the Gurobi optimal cost as reference.

    Parameters:
    -----------
    ts_trajectory, ts_iterations : Lists
        TS best costs and the iterations where improvements occurred
    ga_trajectory, ga_generations : Lists
        GA best costs and the generations where improvements occurred
    optimal_cost : float
        Gurobi optimal cost (reference line)
    instance_name, output_folder, size, difficulty : str
        Instance metadata and output path
    """
    plt.figure(figsize=(14, 8))
    plt.style.use('seaborn-v0_8-darkgrid')

    def make_step_curve(iterations, costs):
        """Build a step-wise (staircase) representation of the trajectory."""
        xi, yi = [], []
        for i in range(len(iterations)):
            xi.append(iterations[i])
            yi.append(costs[i])
            if i < len(iterations) - 1:
                xi.append(iterations[i + 1])
                yi.append(costs[i])
        return np.array(xi), np.array(yi)

    # ── TS trajectory ────────────────────────────────────────────────────────
    ts_xi, ts_yi = make_step_curve(ts_iterations, ts_trajectory)
    plt.plot(ts_xi, ts_yi, color='#2E86AB', linewidth=2.5, alpha=0.9,
             label='Tabu Search (TS)', zorder=2)
    plt.scatter(ts_iterations, ts_trajectory, s=80, color='#2E86AB',
                marker='o', edgecolors='white', linewidths=1.5, zorder=3)

    # ── GA trajectory ────────────────────────────────────────────────────────
    ga_xi, ga_yi = make_step_curve(ga_generations, ga_trajectory)
    plt.plot(ga_xi, ga_yi, color='#E07B39', linewidth=2.5, alpha=0.9,
             label='Genetic Algorithm (GA)', zorder=2)
    plt.scatter(ga_generations, ga_trajectory, s=80, color='#E07B39',
                marker='s', edgecolors='white', linewidths=1.5, zorder=3)

    # ── Gurobi reference ─────────────────────────────────────────────────────
    plt.axhline(y=optimal_cost, color='#C73E1D', linestyle='--', linewidth=2.5,
                alpha=0.85, label=f'Gurobi Optimal: {optimal_cost:.2f}', zorder=1)

    # Final gaps
    ts_final_gap = ((ts_trajectory[-1] - optimal_cost) / optimal_cost) * 100
    ga_final_gap = ((ga_trajectory[-1] - optimal_cost) / optimal_cost) * 100

    plt.title(f'Convergence: {instance_name}\n'
              f'Size: {size.capitalize()} | Difficulty: {difficulty.capitalize()} | '
              f'TS Gap: {ts_final_gap:.2f}% | GA Gap: {ga_final_gap:.2f}%',
              fontsize=13, fontweight='bold', pad=15)
    plt.xlabel('Iteration / Generation', fontsize=12, fontweight='bold')
    plt.ylabel('Total Cost', fontsize=12, fontweight='bold')
    plt.legend(loc='upper right', fontsize=11, framealpha=0.95, shadow=True,
               fancybox=True, borderpad=1)
    plt.grid(True, alpha=0.3)

    ax = plt.gca()
    ax.set_facecolor('#FAFAFA')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Stats box
    textstr = (f'TS:  {ts_trajectory[0]:>10,.2f} → {ts_trajectory[-1]:>10,.2f}  '
               f'(gap {ts_final_gap:.2f}%)\n'
               f'GA:  {ga_trajectory[0]:>10,.2f} → {ga_trajectory[-1]:>10,.2f}  '
               f'(gap {ga_final_gap:.2f}%)')
    props = dict(boxstyle='round,pad=0.6', facecolor='#F0F0F0',
                 edgecolor='#2E86AB', linewidth=1.5, alpha=0.95)
    plt.text(0.02, 0.05, textstr, transform=ax.transAxes, fontsize=9.5,
             verticalalignment='bottom', bbox=props, family='monospace')

    plt.tight_layout()
    plot_path = os.path.join(output_folder,
                             instance_name.replace('.json', '_convergence.png'))
    plt.savefig(plot_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()


def run_experiments(instances_folder: str = "cflp_instances",
                    results_file: str = "results_comparison.csv",
                    ts_max_iterations: int = 200,
                    ts_tabu_tenure: int = 10,
                    ts_max_candidates: int = 20,
                    ts_max_no_improve: int = 40,
                    ga_population_size: int = 30,
                    ga_max_generations: int = 100,
                    ga_crossover_rate: float = 0.85,
                    ga_mutation_rate: float = 0.10,
                    ga_max_no_improve: int = 30,
                    gurobi_time_limit: int = 300,
                    save_convergence_plots: bool = True,
                    convergence_plots_folder: str = "convergence_plots",
                    gurobi_version: str = "academic"):
    """
    Run experiments comparing Gurobi, Tabu Search, and Genetic Algorithm
    on all CFLP instances in the specified folder.

    Parameters:
    -----------
    instances_folder : str
        Folder containing JSON instance files
    results_file : str
        CSV file for saving results
    ts_* : Tabu Search hyperparameters
    ga_* : Genetic Algorithm hyperparameters
    gurobi_time_limit : int
        Time limit (seconds) for Gurobi
    save_convergence_plots : bool
        Whether to save convergence plots (TS + GA vs Gurobi)
    convergence_plots_folder : str
        Folder for convergence plot output
    gurobi_version : str
        "academic" (unlimited) or "free" (max 2000 vars/constraints)

    Returns:
    --------
    results_df : pd.DataFrame
        DataFrame with results for all instances
    """
    if save_convergence_plots:
        os.makedirs(convergence_plots_folder, exist_ok=True)

    check_size_limit = (gurobi_version.lower() == "free")
    instance_files = sorted([f for f in os.listdir(instances_folder) if f.endswith('.json')])

    results = []
    skipped_instances = 0

    print(f"Running experiments on {len(instance_files)} instances...")
    print(f"Gurobi version: {gurobi_version.upper()}")
    if check_size_limit:
        print("⚠️  Free version: instances exceeding 2000 vars/constraints skipped for Gurobi.\n")
    else:
        print("✓ Academic version: no size limits.\n")

    for idx, instance_file in enumerate(instance_files, 1):
        instance_path = os.path.join(instances_folder, instance_file)
        print(f"[{idx}/{len(instance_files)}] Processing {instance_file}")

        # Load instance
        num_facilities, num_clients, fixed_costs, service_costs, \
            facility_capacities, client_demands = load_cflp_instance(instance_path)
        instance = {
            "num_facilities": num_facilities,
            "num_clients": num_clients,
            "fixed_costs": fixed_costs,
            "service_costs": service_costs,
            "facility_capacities": facility_capacities,
            "client_demands": client_demands
        }

        # Extract metadata from filename
        parts = instance_file.replace('.json', '').split('_')
        instance_id = int(parts[0])
        size = parts[2]
        difficulty = parts[3]

        # ── Gurobi ───────────────────────────────────────────────────────────
        print("  - Running Gurobi...")
        try:
            g_start = time.time()
            gurobi_facilities, gurobi_assignments, gurobi_cost = gurobi_solve_cflp(
                instance, time_limit=gurobi_time_limit, check_size_limit=check_size_limit
            )
            gurobi_time = time.time() - g_start
            gurobi_status = "Optimal"
        except ValueError:
            print("    ⚠️  Skipped (exceeds free version limits)")
            gurobi_cost = np.nan
            gurobi_time = np.nan
            gurobi_status = "Skipped (Size Limit)"
            skipped_instances += 1
        except Exception as e:
            print(f"    Gurobi failed: {e}")
            gurobi_cost = np.inf
            gurobi_time = gurobi_time_limit
            gurobi_status = "Failed"

        # ── Tabu Search ──────────────────────────────────────────────────────
        print("  - Running Tabu Search...")
        try:
            ts_start = time.time()
            ts_solution, (ts_traj, ts_traj_iter) = tabu_search_solve_cflp(
                instance,
                max_iterations=ts_max_iterations,
                tabu_tenure=ts_tabu_tenure,
                max_candidates=ts_max_candidates,
                max_no_improve=ts_max_no_improve
            )
            ts_time = time.time() - ts_start
            ts_facilities, ts_assignments = ts_solution
            ts_cost = compute_total_cost(fixed_costs, service_costs, ts_facilities, ts_assignments)
            ts_status = "Completed"
            ts_iterations = ts_traj_iter[-1] if ts_traj_iter else 0
        except Exception as e:
            print(f"    TS failed: {e}")
            ts_cost = np.inf
            ts_time = 0.0
            ts_status = "Failed"
            ts_iterations = 0
            ts_traj, ts_traj_iter = [], []

        # ── Genetic Algorithm ────────────────────────────────────────────────
        print("  - Running Genetic Algorithm...")
        try:
            ga_start = time.time()
            ga_solution, (ga_traj, ga_traj_gen) = genetic_algorithm_solve_cflp(
                instance,
                population_size=ga_population_size,
                max_generations=ga_max_generations,
                crossover_rate=ga_crossover_rate,
                mutation_rate=ga_mutation_rate,
                max_no_improve=ga_max_no_improve
            )
            ga_time = time.time() - ga_start
            ga_facilities, ga_assignments = ga_solution
            ga_cost = compute_total_cost(fixed_costs, service_costs, ga_facilities, ga_assignments)
            ga_status = "Completed"
            ga_generations_run = ga_traj_gen[-1] if ga_traj_gen else 0
        except Exception as e:
            print(f"    GA failed: {e}")
            ga_cost = np.inf
            ga_time = 0.0
            ga_status = "Failed"
            ga_generations_run = 0
            ga_traj, ga_traj_gen = [], []

        # ── Compute gaps ─────────────────────────────────────────────────────
        if gurobi_status == "Optimal" and gurobi_cost > 0 and not np.isnan(gurobi_cost):
            ts_gap = ((ts_cost - gurobi_cost) / gurobi_cost) * 100
            ga_gap = ((ga_cost - gurobi_cost) / gurobi_cost) * 100
        else:
            ts_gap = np.nan
            ga_gap = np.nan

        ts_speedup = gurobi_time / ts_time if ts_time > 0 and not np.isnan(gurobi_time) else np.nan
        ga_speedup = gurobi_time / ga_time if ga_time > 0 and not np.isnan(gurobi_time) else np.nan

        # Save convergence plot
        if (save_convergence_plots and gurobi_status == "Optimal"
                and ts_status == "Completed" and ga_status == "Completed"
                and ts_traj and ga_traj):
            try:
                plot_convergence_ts_ga(
                    ts_traj, ts_traj_iter, ga_traj, ga_traj_gen,
                    gurobi_cost, instance_file, convergence_plots_folder, size, difficulty
                )
            except Exception:
                pass

        # ── Store results ─────────────────────────────────────────────────────
        results.append({
            'instance_id': instance_id,
            'instance_name': instance_file,
            'size': size,
            'difficulty': difficulty,
            'num_facilities': num_facilities,
            'num_clients': num_clients,
            # Gurobi
            'gurobi_cost': gurobi_cost,
            'gurobi_time': gurobi_time,
            'gurobi_status': gurobi_status,
            # Tabu Search
            'ts_cost': ts_cost,
            'ts_time': ts_time,
            'ts_iterations': ts_iterations,
            'ts_status': ts_status,
            'ts_gap_percent': ts_gap,
            'ts_speedup_factor': ts_speedup,
            # Genetic Algorithm
            'ga_cost': ga_cost,
            'ga_time': ga_time,
            'ga_generations': ga_generations_run,
            'ga_status': ga_status,
            'ga_gap_percent': ga_gap,
            'ga_speedup_factor': ga_speedup,
        })

        print(f"  - Gurobi: {gurobi_cost:.2f} ({gurobi_time:.2f}s) | "
              f"TS: gap={ts_gap:.2f}% ({ts_time:.2f}s) | "
              f"GA: gap={ga_gap:.2f}% ({ga_time:.2f}s)\n")

    # ── Save results ──────────────────────────────────────────────────────────
    results_df = pd.DataFrame(results)
    results_df.to_csv(results_file, index=False)
    print(f"\nResults saved to {results_file}")

    xlsx_file = results_file.replace('.csv', '.xlsx')
    with pd.ExcelWriter(xlsx_file, engine='openpyxl') as writer:
        results_df.to_excel(writer, sheet_name='All Results', index=False)

        for col_prefix, label in [('ts', 'Tabu Search'), ('ga', 'Genetic Algorithm')]:
            for group_col, sheet_suffix in [('size', 'by Size'), ('difficulty', 'by Difficulty')]:
                summary = results_df.groupby(group_col).agg({
                    'gurobi_cost': ['mean', 'std'],
                    f'{col_prefix}_cost': ['mean', 'std'],
                    f'{col_prefix}_gap_percent': ['mean', 'min', 'max'],
                    'gurobi_time': ['mean', 'std'],
                    f'{col_prefix}_time': ['mean', 'std'],
                    f'{col_prefix}_speedup_factor': ['mean', 'std'],
                    'instance_id': 'count'
                }).round(2)
                summary.to_excel(writer, sheet_name=f'{label[:2]} {sheet_suffix}')

        for col_prefix, label in [('ts', 'TS'), ('ga', 'GA')]:
            pivot = results_df.pivot_table(
                values=f'{col_prefix}_gap_percent', index='size',
                columns='difficulty', aggfunc='mean'
            ).round(2)
            pivot.to_excel(writer, sheet_name=f'{label} Gap Matrix')

    print(f"Excel results saved to {xlsx_file}")

    # Summary
    valid = results_df[
        (results_df['gurobi_status'] == 'Optimal') &
        (results_df['ts_status'] == 'Completed') &
        (results_df['ga_status'] == 'Completed')
    ]
    print("\n" + "=" * 70)
    print("EXPERIMENTS SUMMARY")
    print("=" * 70)
    print(f"Total instances: {len(instance_files)}")
    print(f"Solved by all three methods: {len(valid)}")
    if len(valid) > 0:
        print(f"\nTabu Search  — avg gap: {valid['ts_gap_percent'].mean():.2f}%, "
              f"avg speedup: {valid['ts_speedup_factor'].mean():.2f}x")
        print(f"Genetic Algo — avg gap: {valid['ga_gap_percent'].mean():.2f}%, "
              f"avg speedup: {valid['ga_speedup_factor'].mean():.2f}x")
    print("=" * 70)

    return results_df


print("✅ Experiment runner (TS + GA + Gurobi) loaded successfully.")


✅ Experiment runner (TS + GA + Gurobi) loaded successfully.


---

## 9. Visualization and Analysis

This section provides comprehensive visualization tools for comparing Gurobi, Tabu Search, and Genetic Algorithm results across all instances.

**Plots generated:**
- **Optimality Gap (%)** — Boxplots of TS and GA gap by instance size and difficulty.
- **Computation Time** — Bar charts comparing Gurobi, TS, and GA runtimes.
- **Speedup Factor** — TS speedup and GA speedup over Gurobi (× faster).
- **Solution Cost Heatmaps** — Mean gap organized in a size × difficulty matrix for both TS and GA.
- **TS vs GA head-to-head** — Scatter plot comparing TS gap against GA gap per instance.


In [13]:
def visualize_results(results_df: pd.DataFrame,
                      output_folder: str = "visualization_plots"):
    """
    Generate comprehensive comparison plots for Gurobi, Tabu Search, and Genetic Algorithm.

    Parameters:
    -----------
    results_df : pd.DataFrame
        DataFrame produced by run_experiments()
    output_folder : str
        Folder for saving plots
    """
    os.makedirs(output_folder, exist_ok=True)

    # Filter to instances solved by all three methods
    valid = results_df[
        (results_df['gurobi_status'] == 'Optimal') &
        (results_df['ts_status'] == 'Completed') &
        (results_df['ga_status'] == 'Completed')
    ].copy()

    if valid.empty:
        print("⚠️  No valid results to visualize.")
        return

    plt.style.use('seaborn-v0_8-darkgrid')
    SIZE_ORDER = ['small', 'medium', 'large']
    DIFF_ORDER = ['easy', 'moderate', 'expensive']
    TS_COLOR   = '#2E86AB'
    GA_COLOR   = '#E07B39'
    GRB_COLOR  = '#C73E1D'

    # ── 1. Optimality Gap by Instance Size ───────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)
    for ax, col, label, color in zip(
            axes,
            ['ts_gap_percent', 'ga_gap_percent'],
            ['Tabu Search', 'Genetic Algorithm'],
            [TS_COLOR, GA_COLOR]):
        sizes_present = [s for s in SIZE_ORDER if s in valid['size'].values]
        data_by_size = [valid[valid['size'] == s][col].dropna().values for s in sizes_present]
        bp = ax.boxplot(data_by_size, labels=[s.capitalize() for s in sizes_present],
                        patch_artist=True, widths=0.5, notch=False)
        for patch in bp['boxes']:
            patch.set_facecolor(color)
            patch.set_alpha(0.6)
        for median in bp['medians']:
            median.set_color('black')
            median.set_linewidth(2)
        ax.set_title(f'{label}: Optimality Gap by Size', fontsize=13, fontweight='bold')
        ax.set_xlabel('Instance Size', fontsize=11)
        ax.set_ylabel('Optimality Gap (%)', fontsize=11)
        ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(os.path.join(output_folder, 'gap_by_size.png'), dpi=150, bbox_inches='tight')
    plt.close()

    # ── 2. Optimality Gap by Difficulty ──────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)
    for ax, col, label, color in zip(
            axes,
            ['ts_gap_percent', 'ga_gap_percent'],
            ['Tabu Search', 'Genetic Algorithm'],
            [TS_COLOR, GA_COLOR]):
        diffs_present = [d for d in DIFF_ORDER if d in valid['difficulty'].values]
        data_by_diff = [valid[valid['difficulty'] == d][col].dropna().values for d in diffs_present]
        bp = ax.boxplot(data_by_diff, labels=[d.capitalize() for d in diffs_present],
                        patch_artist=True, widths=0.5)
        for patch in bp['boxes']:
            patch.set_facecolor(color)
            patch.set_alpha(0.6)
        for median in bp['medians']:
            median.set_color('black')
            median.set_linewidth(2)
        ax.set_title(f'{label}: Optimality Gap by Difficulty', fontsize=13, fontweight='bold')
        ax.set_xlabel('Difficulty', fontsize=11)
        ax.set_ylabel('Optimality Gap (%)', fontsize=11)
        ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(os.path.join(output_folder, 'gap_by_difficulty.png'), dpi=150, bbox_inches='tight')
    plt.close()

    # ── 3. Computation Time Comparison ───────────────────────────────────────
    fig, ax = plt.subplots(figsize=(14, 6))
    sizes_present = [s for s in SIZE_ORDER if s in valid['size'].values]
    x = np.arange(len(sizes_present))
    width = 0.25
    for i, (col, label, color) in enumerate(zip(
            ['gurobi_time', 'ts_time', 'ga_time'],
            ['Gurobi', 'Tabu Search', 'Genetic Algorithm'],
            [GRB_COLOR, TS_COLOR, GA_COLOR])):
        means = [valid[valid['size'] == s][col].mean() for s in sizes_present]
        stds  = [valid[valid['size'] == s][col].std()  for s in sizes_present]
        bars = ax.bar(x + i * width, means, width,
                      label=label, color=color, alpha=0.8, edgecolor='white',
                      linewidth=1.2, yerr=stds, capsize=5, error_kw={'linewidth': 1.5})
    ax.set_xlabel('Instance Size', fontsize=12, fontweight='bold')
    ax.set_ylabel('Computation Time (seconds)', fontsize=12, fontweight='bold')
    ax.set_title('Mean Computation Time: Gurobi vs TS vs GA', fontsize=13, fontweight='bold')
    ax.set_xticks(x + width)
    ax.set_xticklabels([s.capitalize() for s in sizes_present], fontsize=11)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    fig.savefig(os.path.join(output_folder, 'computation_time.png'), dpi=150, bbox_inches='tight')
    plt.close()

    # ── 4. Speedup Factor ────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(14, 6))
    for i, (col, label, color) in enumerate(zip(
            ['ts_speedup_factor', 'ga_speedup_factor'],
            ['Tabu Search Speedup', 'GA Speedup'],
            [TS_COLOR, GA_COLOR])):
        means = [valid[valid['size'] == s][col].mean() for s in sizes_present]
        stds  = [valid[valid['size'] == s][col].std()  for s in sizes_present]
        bars = ax.bar(x + i * width, means, width,
                      label=label, color=color, alpha=0.8, edgecolor='white',
                      linewidth=1.2, yerr=stds, capsize=5, error_kw={'linewidth': 1.5})
    ax.axhline(y=1, color='gray', linestyle='--', alpha=0.6, label='Baseline (1×)')
    ax.set_xlabel('Instance Size', fontsize=12, fontweight='bold')
    ax.set_ylabel('Speedup over Gurobi (×)', fontsize=12, fontweight='bold')
    ax.set_title('Speedup Factor: TS and GA vs Gurobi', fontsize=13, fontweight='bold')
    ax.set_xticks(x + width / 2)
    ax.set_xticklabels([s.capitalize() for s in sizes_present], fontsize=11)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    fig.savefig(os.path.join(output_folder, 'speedup_factor.png'), dpi=150, bbox_inches='tight')
    plt.close()

    # ── 5. TS vs GA Head-to-Head Scatter ─────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 8))
    size_markers = {'small': 'o', 'medium': 's', 'large': '^'}
    diff_alphas  = {'easy': 0.4, 'moderate': 0.7, 'expensive': 1.0}
    for _, row in valid.iterrows():
        marker = size_markers.get(row['size'], 'o')
        alpha  = diff_alphas.get(row['difficulty'], 0.7)
        ax.scatter(row['ts_gap_percent'], row['ga_gap_percent'],
                   s=80, marker=marker, alpha=alpha,
                   c=GA_COLOR, edgecolors='black', linewidths=0.8)
    max_gap = max(valid[['ts_gap_percent', 'ga_gap_percent']].max())
    diag = np.linspace(0, max_gap * 1.05, 100)
    ax.plot(diag, diag, 'k--', alpha=0.5, label='TS = GA')
    ax.set_xlabel('Tabu Search Gap (%)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Genetic Algorithm Gap (%)', fontsize=12, fontweight='bold')
    ax.set_title('TS vs GA: Optimality Gap per Instance\n'
                 '(points above diagonal: TS outperforms GA)', fontsize=13, fontweight='bold')
    # Legend patches
    import matplotlib.patches as mpatches
    legend_elements = [
        mpatches.Patch(color='none', label='Size: o=Small  s=Medium  ^=Large'),
        mpatches.Patch(color='none', label='Difficulty: lighter=easier  darker=expensive'),
        plt.Line2D([0], [0], linestyle='--', color='black', alpha=0.5, label='TS = GA'),
    ]
    ax.legend(handles=legend_elements, fontsize=10, loc='upper left')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(os.path.join(output_folder, 'ts_vs_ga_scatter.png'), dpi=150, bbox_inches='tight')
    plt.close()

    # ── 6. Gap Heatmaps ──────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    for ax, col, label, cmap in zip(
            axes,
            ['ts_gap_percent', 'ga_gap_percent'],
            ['Tabu Search', 'Genetic Algorithm'],
            ['Blues', 'Oranges']):
        pivot = valid.pivot_table(values=col, index='size', columns='difficulty',
                                   aggfunc='mean')
        pivot = pivot.reindex(index=[s for s in SIZE_ORDER if s in pivot.index],
                              columns=[d for d in DIFF_ORDER if d in pivot.columns])
        sns.heatmap(pivot, ax=ax, annot=True, fmt='.2f', cmap=cmap,
                    linewidths=0.5, linecolor='white',
                    cbar_kws={'label': 'Mean Gap (%)'}, annot_kws={'size': 12})
        ax.set_title(f'{label}: Mean Optimality Gap (%) — Size × Difficulty',
                     fontsize=12, fontweight='bold')
        ax.set_xlabel('Difficulty', fontsize=11)
        ax.set_ylabel('Instance Size', fontsize=11)
    plt.tight_layout()
    fig.savefig(os.path.join(output_folder, 'gap_heatmap.png'), dpi=150, bbox_inches='tight')
    plt.close()

    # ── 7. Summary table ─────────────────────────────────────────────────────
    print("\n" + "=" * 75)
    print("VISUALIZATION SUMMARY — Mean Results by Instance Size")
    print("=" * 75)
    print(f"\n{'Size':<12} {'Gurobi (s)':>12} {'TS Gap%':>10} {'TS (s)':>10} "
          f"{'GA Gap%':>10} {'GA (s)':>10}")
    print("-" * 75)
    for s in sizes_present:
        sub = valid[valid['size'] == s]
        print(f"{s.capitalize():<12} {sub['gurobi_time'].mean():>12.2f} "
              f"{sub['ts_gap_percent'].mean():>10.2f} {sub['ts_time'].mean():>10.2f} "
              f"{sub['ga_gap_percent'].mean():>10.2f} {sub['ga_time'].mean():>10.2f}")

    print("\n" + "=" * 75)
    print(f"Plots saved to: {output_folder}/")
    print("=" * 75)


print("✅ Visualization module (3-way comparison: TS, GA, Gurobi) loaded successfully.")


✅ Visualization module (3-way comparison: TS, GA, Gurobi) loaded successfully.


---

## 10. Student Configuration

**Set your parameters in the cell below, then run it before running Section 11 (Main Execution).**

### 10.0  Your Student ID (REQUIRED)

| Setting | What to do |
|---------|------------|
| `STUDENT_ID` | Replace `12345` with **your own student ID**. It is used as the random seed of the instance generator, so **your instances and results are unique**, and it appears in the names of all your output files. The cell stops with an error if you leave the default value. |

### 10.1  Gurobi Version

| Setting | When to use |
|---------|-------------|
| `"academic"` | You have an Academic or Full Gurobi license (no size limits). |
| `"free"`     | You are using the Gurobi free/trial version (≤ 2,000 variables). Large instances will be skipped. |

### 10.2  Tabu Search Hyperparameters

| Parameter | Default | Effect |
|-----------|---------|--------|
| `TS_MAX_ITERATIONS` | 200 | Total number of TS moves attempted |
| `TS_TABU_TENURE` | 10 | How many iterations a move stays on the tabu list |
| `TS_MAX_CANDIDATES` | 20 | Number of neighbor moves evaluated per iteration |
| `TS_MAX_NO_IMPROVE` | 40 | Stop after this many non-improving iterations |

### 10.3  Genetic Algorithm Hyperparameters

| Parameter | Default | Effect |
|-----------|---------|--------|
| `GA_POPULATION_SIZE` | 30 | Number of chromosomes per generation |
| `GA_MAX_GENERATIONS` | 100 | Maximum number of generations to evolve |
| `GA_CROSSOVER_RATE` | 0.85 | Probability that two parents produce offspring |
| `GA_MUTATION_RATE` | 0.10 | Per-gene mutation probability |
| `GA_MAX_NO_IMPROVE` | 30 | Early stopping if best cost does not improve |

### 10.4  Expected Runtimes (rough estimates)

| Instance size | Gurobi | Tabu Search | Genetic Algorithm |
|---------------|--------|-------------|-------------------|
| Small | < 5 s | < 3 s | < 5 s |
| Medium | 10–120 s | < 10 s | < 15 s |
| Large | 60–300 s | < 30 s | < 40 s |

> **Note**: These are approximate. Actual runtimes vary with instance difficulty and your machine's speed.


In [14]:
# ════════════════════════════════════════════════════════════════
# STUDENT CONFIGURATION — Edit these values before running main()
# ════════════════════════════════════════════════════════════════

# ── Your identity (REQUIRED) ─────────────────────────────────────
# TODO: replace 12345 with YOUR actual student ID (it seeds YOUR unique instances)
STUDENT_ID = 447540190

# ── Gurobi version ───────────────────────────────────────────────
GUROBI_VERSION = "academic"   # "academic" or "free"

# ── Instance generation ───────────────────────────────────────────
INSTANCES_FOLDER       = f"cflp_instances_student_{STUDENT_ID}"
INSTANCE_SIZES         = ['small', 'medium', 'large']
INSTANCE_DIFFICULTIES  = ['easy', 'moderate', 'expensive']
INSTANCES_PER_COMBO    = 16    # 3 × 3 × 16 = 144 instances total
RANDOM_SEED            = STUDENT_ID   # your ID -> your own unique instances

# ── Tabu Search hyperparameters ───────────────────────────────────
TS_MAX_ITERATIONS  = 200   # Total TS iterations
TS_TABU_TENURE     = 10    # Tabu list tenure (iterations)
TS_MAX_CANDIDATES  = 20    # Candidate moves per iteration
TS_MAX_NO_IMPROVE  = 40    # Early stopping after no improvement

# ── Genetic Algorithm hyperparameters ─────────────────────────────
GA_POPULATION_SIZE = 30    # Number of chromosomes
GA_MAX_GENERATIONS = 100   # Maximum generations
GA_CROSSOVER_RATE  = 0.85  # Probability of crossover
GA_MUTATION_RATE   = 0.10  # Per-gene mutation probability
GA_MAX_NO_IMPROVE  = 30    # Early stopping after no improvement

# ── Gurobi time limit ─────────────────────────────────────────────
GUROBI_TIME_LIMIT  = 300   # seconds per instance

# ── Output settings ───────────────────────────────────────────────
RESULTS_FILE              = f"results_student_{STUDENT_ID}.csv"
SAVE_CONVERGENCE_PLOTS    = True
CONVERGENCE_PLOTS_FOLDER  = f"convergence_plots_student_{STUDENT_ID}"
VISUALIZATION_FOLDER      = f"visualization_plots_student_{STUDENT_ID}"

# ════════════════════════════════════════════════════════════════
if STUDENT_ID == 12345:
    raise ValueError("Please set STUDENT_ID to YOUR actual student ID (replace 12345) before running.")

print("✅ Configuration loaded:")
print(f"   Student ID     : {STUDENT_ID} (random seed -> unique instances and results)")
print(f"   Gurobi version : {GUROBI_VERSION}")
print(f"   Instances      : {INSTANCES_PER_COMBO} per combo "
      f"({len(INSTANCE_SIZES)}×{len(INSTANCE_DIFFICULTIES)}×{INSTANCES_PER_COMBO} = "
      f"{len(INSTANCE_SIZES)*len(INSTANCE_DIFFICULTIES)*INSTANCES_PER_COMBO} total)")
print(f"   TS params      : max_iter={TS_MAX_ITERATIONS}, tenure={TS_TABU_TENURE}, "
      f"max_cand={TS_MAX_CANDIDATES}, max_no_improve={TS_MAX_NO_IMPROVE}")
print(f"   GA params      : pop={GA_POPULATION_SIZE}, gen={GA_MAX_GENERATIONS}, "
      f"cx={GA_CROSSOVER_RATE}, mut={GA_MUTATION_RATE}, max_no_improve={GA_MAX_NO_IMPROVE}")
print(f"   Gurobi limit   : {GUROBI_TIME_LIMIT}s")


✅ Configuration loaded:
   Student ID     : 447540190 (random seed -> unique instances and results)
   Gurobi version : academic
   Instances      : 16 per combo (3×3×16 = 144 total)
   TS params      : max_iter=200, tenure=10, max_cand=20, max_no_improve=40
   GA params      : pop=30, gen=100, cx=0.85, mut=0.1, max_no_improve=30
   Gurobi limit   : 300s


---

## 11. Main Execution

This cell orchestrates the full pipeline:
1. Generate all 144 CFLP instances (3 sizes × 3 difficulties × 16 each).
2. Solve each instance with **Gurobi**, **Tabu Search**, and **Genetic Algorithm**.
3. Save results to CSV + Excel.
4. Generate all visualization plots.

> **Before running**: Make sure you have run **Section 10 (Student Configuration)** first to set your **student ID**, your Gurobi version and the hyperparameters.


In [15]:
def main():
    """
    Main execution pipeline: generate instances, run Gurobi + TS + GA, visualize results.
    """
    print("=" * 70)
    print("CFLP — Gurobi vs Tabu Search vs Genetic Algorithm")
    print("=" * 70)

    # ── Step 1: Generate instances ────────────────────────────────────────────
    print("\nStep 1: Generating CFLP instances...")
    generate_all_instances(
        output_folder=INSTANCES_FOLDER,
        sizes=INSTANCE_SIZES,
        difficulties=INSTANCE_DIFFICULTIES,
        instances_per_combo=INSTANCES_PER_COMBO,
        seed=RANDOM_SEED
    )

    # ── Step 2: Run experiments ───────────────────────────────────────────────
    print("\nStep 2: Running experiments (Gurobi + Tabu Search + Genetic Algorithm)...")
    results_df = run_experiments(
        instances_folder=INSTANCES_FOLDER,
        results_file=RESULTS_FILE,
        ts_max_iterations=TS_MAX_ITERATIONS,
        ts_tabu_tenure=TS_TABU_TENURE,
        ts_max_candidates=TS_MAX_CANDIDATES,
        ts_max_no_improve=TS_MAX_NO_IMPROVE,
        ga_population_size=GA_POPULATION_SIZE,
        ga_max_generations=GA_MAX_GENERATIONS,
        ga_crossover_rate=GA_CROSSOVER_RATE,
        ga_mutation_rate=GA_MUTATION_RATE,
        ga_max_no_improve=GA_MAX_NO_IMPROVE,
        gurobi_time_limit=GUROBI_TIME_LIMIT,
        save_convergence_plots=SAVE_CONVERGENCE_PLOTS,
        convergence_plots_folder=CONVERGENCE_PLOTS_FOLDER,
        gurobi_version=GUROBI_VERSION
    )

    # ── Step 3: Visualize results ─────────────────────────────────────────────
    print("\nStep 3: Generating visualization plots...")
    visualize_results(results_df, output_folder=VISUALIZATION_FOLDER)

    # ── Step 4: Print final summary ───────────────────────────────────────────
    valid = results_df[
        (results_df['gurobi_status'] == 'Optimal') &
        (results_df['ts_status'] == 'Completed') &
        (results_df['ga_status'] == 'Completed')
    ]

    print("\n" + "=" * 70)
    print("FINAL SUMMARY")
    print("=" * 70)
    print(f"Total instances: {len(results_df)}")
    print(f"Solved by all three methods: {len(valid)}")
    if len(valid) > 0:
        print(f"\n{'Metric':<35} {'Tabu Search':>15} {'Gen. Algorithm':>16}")
        print("-" * 70)
        print(f"{'Mean optimality gap (%):':<35} "
              f"{valid['ts_gap_percent'].mean():>14.2f}% "
              f"{valid['ga_gap_percent'].mean():>15.2f}%")
        print(f"{'Mean computation time (s):':<35} "
              f"{valid['ts_time'].mean():>15.2f} "
              f"{valid['ga_time'].mean():>16.2f}")
        print(f"{'Mean speedup over Gurobi (×):':<35} "
              f"{valid['ts_speedup_factor'].mean():>15.2f} "
              f"{valid['ga_speedup_factor'].mean():>16.2f}")
        print(f"\n{'Gurobi mean time (s):':<35} {valid['gurobi_time'].mean():>15.2f}")

    print("\n" + "=" * 70)
    print("Output files:")
    print(f"  Results CSV  : {RESULTS_FILE}")
    print(f"  Results XLSX : {RESULTS_FILE.replace('.csv', '.xlsx')}")
    print(f"  Convergence  : {CONVERGENCE_PLOTS_FOLDER}/")
    print(f"  Visualizations: {VISUALIZATION_FOLDER}/")
    print("=" * 70)
    print("\n✅ Done! Review the output files and answer the analysis questions.")

    return results_df


# Run the pipeline
results = main()


CFLP — Gurobi vs Tabu Search vs Genetic Algorithm

Step 1: Generating CFLP instances...

CLEANING UP OLD INSTANCES
✓ Folder 'cflp_instances_student_447540190' does not exist (first run)
✓ Created fresh folder 'cflp_instances_student_447540190'

Generating 144 CFLP instances...
Distribution: 16 per size-difficulty combination
All instances comply with free Gurobi limits (≤2000 vars, ≤2000 constraints)


Successfully generated 144 instances in 'cflp_instances_student_447540190/'
   Summary saved to: cflp_instances_student_447540190\instances_summary.xlsx

Distribution Summary:
difficulty  easy  expensive  moderate
size                                 
large         16         16        16
medium        16         16        16
small         16         16        16

Step 2: Running experiments (Gurobi + Tabu Search + Genetic Algorithm)...
Running experiments on 144 instances...
Gurobi version: ACADEMIC
✓ Academic version: no size limits.

[1/144] Processing 001_cflp_small_easy_fac16_cli75.

---

## 12. Bonus Challenges (Optional — Up to +10% Extra Credit)

These optional challenges require you to **systematically evaluate all proposed enhancements**, then **select and fully implement the single best one** for each algorithm. Simply picking one without comparing the others will not receive full marks.

---

### Bonus A — Improve Tabu Search (+5%)

**Step 1 — Explore all options.** Run a quick experiment (e.g., on the small/moderate subset only) for **each** of the four enhancements below. Record the mean gap improvement per option.

| Enhancement | Description |
|-------------|-------------|
| **Adaptive tenure** | Dynamically increase tenure when the search stagnates; decrease it when improvements are frequent. |
| **Aspiration+ criterion** | Allow a tabu move if it improves the **global** best by a margin δ (stronger than the standard criterion). |
| **Frequency-based memory** | Track long-term move frequencies and add a penalty to discourage frequently revisited configurations. |
| **Diversification restart** | After a stagnation threshold, apply a strong multi-swap perturbation to escape local optima. |

**Step 2 — Choose ONE.** Select the enhancement with the best gap reduction in Step 1 and justify your choice in the student guide (Section 12, Bonus A).

**Step 3 — Full implementation.** Apply your chosen enhancement to the full 144-instance experiment. Compare mean gap and runtime before vs after and report in Section 12 of the student guide.

---

### Bonus B — Improve Genetic Algorithm (+5%)

**Step 1 — Explore all options.** Run a quick experiment on the small/moderate subset for **each** of the four enhancements below. Record the mean gap improvement per option.

| Enhancement | Description |
|-------------|-------------|
| **Adaptive mutation** | Decrease mutation rate as population converges; increase it if diversity drops below a threshold. |
| **Two-point crossover** | Replace uniform crossover with two-point crossover; compare convergence speed and final gap. |
| **Memetic GA (TS + GA)** | Apply a short Tabu Search local refinement to the best chromosome each generation. |
| **Diversity preservation** | Track Hamming distance between chromosomes; actively reintroduce diversity when population converges. |

**Step 2 — Choose ONE.** Select the enhancement with the best gap reduction in Step 1 and justify your choice in the student guide (Section 12, Bonus B).

**Step 3 — Full implementation.** Apply your chosen enhancement to the full 144-instance experiment. Compare mean gap and runtime before vs after and report in Section 12 of the student guide.


In [1]:

# ==================================================
# BONUS A — TABU SEARCH ENHANCEMENTS
# Step 1: Experiment Setup
# ==================================================

import numpy as np
import pandas as pd
import time
import random

BONUS_SEED = 447540190

random.seed(BONUS_SEED)
np.random.seed(BONUS_SEED)

TS_ENHANCEMENTS = [
    "adaptive_tenure",
    "aspiration",
    "frequency_memory",
    "diversification_restart"
]

print("Bonus A setup completed successfully!")
print("Student ID:", BONUS_SEED)
print("Enhancements to evaluate:", len(TS_ENHANCEMENTS))


Bonus A setup completed successfully!
Student ID: 447540190
Enhancements to evaluate: 4


In [5]:

# BONUS A - Load original Tabu Search functions

import json
import ast
import numpy as np
from typing import Tuple, List

with open("Assignment_2.ipynb", encoding="utf-8") as f:
    original_notebook = json.load(f)

for cell_number in [5, 8, 11, 13, 15, 17]:
    source = "".join(
        original_notebook["cells"][cell_number]["source"]
    )

    tree = ast.parse(source)

    functions = [
        node for node in tree.body
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
    ]

    module = ast.Module(body=functions, type_ignores=[])
    exec(compile(module, "<original_notebook>", "exec"), globals())

print("Original Tabu Search functions loaded!")
print("Ready for Adaptive Tenure experiment.")


Original Tabu Search functions loaded!
Ready for Adaptive Tenure experiment.


In [43]:

# BONUS A - Repair Infeasible Initial Assignments

import gurobipy as gp
import numpy as np

def repair_tabu_initial_solution(inst):
    opened, assignments = tabu_initial_solution(inst)

    capacities = np.asarray(inst["facility_capacities"])
    demands = np.asarray(inst["client_demands"])
    service = np.asarray(inst["service_costs"])

    if is_feasible(opened, assignments, capacities, demands):
        return opened, assignments

    n_clients = len(demands)
    open_facilities = np.flatnonzero(opened).tolist()

    model = gp.Model("Repair_Assignments")
    model.Params.OutputFlag = 0
    model.Params.TimeLimit = 60

    x = model.addVars(
        range(n_clients),
        open_facilities,
        vtype=gp.GRB.BINARY
    )

    for i in range(n_clients):
        model.addConstr(
            gp.quicksum(x[i, j] for j in open_facilities) == 1
        )

    for j in open_facilities:
        model.addConstr(
            gp.quicksum(
                demands[i] * x[i, j]
                for i in range(n_clients)
            ) <= capacities[j]
        )

    model.setObjective(
        gp.quicksum(
            (
                1000 * int(j != assignments[i])
                + 0.0001 * float(service[i, j])
            ) * x[i, j]
            for i in range(n_clients)
            for j in open_facilities
        ),
        gp.GRB.MINIMIZE
    )

    model.optimize()

    if model.SolCount == 0:
        raise RuntimeError(
            "No feasible repair found. Check solver status."
        )

    repaired = np.array([
        max(
            open_facilities,
            key=lambda j: x[i, j].X
        )
        for i in range(n_clients)
    ], dtype=int)

    if not is_feasible(opened, repaired, capacities, demands):
        raise RuntimeError("Repaired solution is not feasible")

    return opened.copy(), repaired

In [44]:

# BONUS A1 - Adaptive Tabu Tenure

def tabu_search_adaptive_tenure(
    instance,
    max_iterations=200,
    tabu_tenure=10,
    max_candidates=20,
    max_no_improve=40
):
    fixed_costs = instance["fixed_costs"]
    service_costs = instance["service_costs"]
    capacities = instance["facility_capacities"]
    demands = instance["client_demands"]
    current_solution = repair_tabu_initial_solution(instance)

    best_solution = copy_solution(current_solution[0], current_solution[1])
    best_cost = compute_total_cost(
        fixed_costs, service_costs,
        best_solution[0], best_solution[1]
    )

    trajectory = [best_cost]
    trajectory_iterations = [0]

    tabu_list = {}
    no_improve = 0

    current_tenure = tabu_tenure
    min_tenure = max(2, tabu_tenure // 2)
    max_tenure = tabu_tenure * 2

    for iteration in range(1, max_iterations + 1):

        candidates = generate_all_candidates(
            current_solution,
            service_costs,
            capacities,
            demands,
            max_candidates
        )

        if not candidates:
            break

        selected = select_best_candidate(
            candidates,
            tabu_list,
            iteration,
            best_cost,
            fixed_costs,
            service_costs
        )

        if selected is None:
            no_improve += 1
            if no_improve >= max_no_improve:
                break
            continue

        new_opened, new_assignments, move_key, new_cost = selected

        current_solution = (new_opened, new_assignments)

        inverse_key = get_inverse_move(move_key)
        tabu_list[inverse_key] = iteration + current_tenure

        tabu_list = {
            k: v for k, v in tabu_list.items()
            if v > iteration
        }

        if new_cost < best_cost:
            best_solution = copy_solution(current_solution[0], current_solution[1])
            best_cost = new_cost
            trajectory.append(best_cost)
            trajectory_iterations.append(iteration)
            no_improve = 0

            current_tenure = max(
                min_tenure, current_tenure - 1
            )
        else:
            no_improve += 1

            if no_improve % 5 == 0:
                current_tenure = min(
                    max_tenure, current_tenure + 2
                )

        if no_improve >= max_no_improve:
            break

    return best_solution, (
        trajectory, trajectory_iterations
    )


print("Adaptive Tenure implementation loaded!")
print("Original Tabu Search remains unchanged.")


Adaptive Tenure implementation loaded!
Original Tabu Search remains unchanged.


In [45]:

# BONUS A - Verify Repaired Adaptive TS on Instance 116

import glob
import json
import numpy as np
import random

files_116 = glob.glob(
    "cflp_instances_student_447540190/116_*.json"
)

assert len(files_116) == 1, "Instance 116 not found"

with open(files_116[0], encoding="utf-8") as f:
    inst = json.load(f)

for key in [
    "fixed_costs",
    "service_costs",
    "facility_capacities",
    "client_demands"
]:
    inst[key] = np.asarray(inst[key], dtype=float)

random.seed(447540190 + 116)
np.random.seed(447540190 + 116)

solution, trajectory = tabu_search_adaptive_tenure(
    inst,
    max_iterations=200,
    tabu_tenure=10,
    max_candidates=20,
    max_no_improve=40
)

feasible = is_feasible(
    solution[0],
    solution[1],
    inst["facility_capacities"],
    inst["client_demands"]
)

print("Instance 116 feasible:", feasible)
assert feasible, "Repair verification failed"
print("BONUS A REPAIR TEST PASSED!")


Restricted license - for non-production use only - expires 2027-11-29
Instance 116 feasible: True
BONUS A REPAIR TEST PASSED!


In [10]:

# BONUS A1 - Test Adaptive Tenure

import os
import json
import numpy as np
import time

test_file = sorted(instance_files)[0]

with open(test_file, "r", encoding="utf-8") as f:
    raw_instance = json.load(f)

instance = {
    "num_facilities": int(raw_instance["num_facilities"]),
    "num_clients": int(raw_instance["num_clients"]),
    "fixed_costs": np.array(raw_instance["fixed_costs"]),
    "service_costs": np.array(raw_instance["service_costs"]),
    "facility_capacities": np.array(raw_instance["facility_capacities"]),
    "client_demands": np.array(raw_instance["client_demands"])
}

np.random.seed(BONUS_SEED)
start = time.perf_counter()

solution, trajectory_data = tabu_search_adaptive_tenure(
    instance,
    max_iterations=200,
    tabu_tenure=10,
    max_candidates=20,
    max_no_improve=40
)

runtime = time.perf_counter() - start

print("Instance:", os.path.basename(test_file))
print("Best cost:", trajectory_data[0][-1])
print("Runtime:", round(runtime, 3), "seconds")
print("Feasible:", is_feasible(
    solution[0],
    solution[1],
    instance["facility_capacities"],
    instance["client_demands"]
))


Instance: 001_cflp_small_easy_fac16_cli75.json
Best cost: 18007.527403107208
Runtime: 0.54 seconds
Feasible: True


In [11]:

# BONUS A1 - Baseline vs Adaptive Tenure

import time
import random
import numpy as np

# Original Tabu Search
random.seed(BONUS_SEED)
np.random.seed(BONUS_SEED)

start = time.perf_counter()

original_solution, original_trajectory = tabu_search_solve_cflp(
    instance,
    max_iterations=200,
    tabu_tenure=10,
    max_candidates=20,
    max_no_improve=40
)

original_time = time.perf_counter() - start
original_cost = original_trajectory[0][-1]

# Improved Tabu Search
random.seed(BONUS_SEED)
np.random.seed(BONUS_SEED)

start = time.perf_counter()

adaptive_solution, adaptive_trajectory = tabu_search_adaptive_tenure(
    instance,
    max_iterations=200,
    tabu_tenure=10,
    max_candidates=20,
    max_no_improve=40
)

adaptive_time = time.perf_counter() - start
adaptive_cost = adaptive_trajectory[0][-1]

# Comparison
improvement = (
    (original_cost - adaptive_cost) / original_cost
) * 100

print("\n===== BONUS A1 COMPARISON =====")
print(f"Original TS cost: {original_cost:.2f}")
print(f"Adaptive TS cost: {adaptive_cost:.2f}")
print(f"Original runtime: {original_time:.3f} s")
print(f"Adaptive runtime: {adaptive_time:.3f} s")
print(f"Cost improvement: {improvement:.2f}%")
print("Original feasible:", is_feasible(
    original_solution[0], original_solution[1],
    instance["facility_capacities"], instance["client_demands"]
))
print("Adaptive feasible:", is_feasible(
    adaptive_solution[0], adaptive_solution[1],
    instance["facility_capacities"], instance["client_demands"]
))


Starting Tabu Search for CFLP (tenure=10, max_iter=200)...
Tabu Search completed after 43 iterations.
Best cost: 18007.53

===== BONUS A1 COMPARISON =====
Original TS cost: 18007.53
Adaptive TS cost: 18007.53
Original runtime: 0.548 s
Adaptive runtime: 0.517 s
Cost improvement: 0.00%
Original feasible: True
Adaptive feasible: True


In [12]:

# BONUS A1 - Pilot Experiment (6 Instances)

import json
import os
import time
import random
import numpy as np
import pandas as pd
import contextlib
import io

small_files = sorted(
    [f for f in instance_files if "small" in os.path.basename(f)]
)[:3]

medium_files = sorted(
    [f for f in instance_files if "medium" in os.path.basename(f)]
)[:3]

pilot_files = small_files + medium_files
assert len(pilot_files) == 6, "Check pilot instance files"

pilot_results = []

for file_path in pilot_files:
    with open(file_path, encoding="utf-8") as f:
        raw = json.load(f)

    inst = {
        "num_facilities": int(raw["num_facilities"]),
        "num_clients": int(raw["num_clients"]),
        "fixed_costs": np.array(raw["fixed_costs"]),
        "service_costs": np.array(raw["service_costs"]),
        "facility_capacities": np.array(raw["facility_capacities"]),
        "client_demands": np.array(raw["client_demands"])
    }

    costs = []
    runtimes = []

    for solver in [
        tabu_search_solve_cflp,
        tabu_search_adaptive_tenure
    ]:
        random.seed(BONUS_SEED)
        np.random.seed(BONUS_SEED)

        start = time.perf_counter()

        with contextlib.redirect_stdout(io.StringIO()):
            sol, trajectory = solver(
                inst,
                max_iterations=200,
                tabu_tenure=10,
                max_candidates=20,
                max_no_improve=40
            )

        elapsed = time.perf_counter() - start

        assert is_feasible(
            sol[0], sol[1],
            inst["facility_capacities"],
            inst["client_demands"]
        ), "Infeasible solution"

        costs.append(trajectory[0][-1])
        runtimes.append(elapsed)

    pilot_results.append({
        "Instance": os.path.basename(file_path),
        "Original_Cost": costs[0],
        "Adaptive_Cost": costs[1],
        "Improvement_%":
            (costs[0] - costs[1]) / costs[0] * 100,
        "Original_Time": runtimes[0],
        "Adaptive_Time": runtimes[1]
    })

pilot_df = pd.DataFrame(pilot_results)

print(pilot_df.round(3).to_string(index=False))
print("\nMean cost improvement (%):",
      round(pilot_df["Improvement_%"].mean(), 3))
print("Mean original time:",
      round(pilot_df["Original_Time"].mean(), 3))
print("Mean adaptive time:",
      round(pilot_df["Adaptive_Time"].mean(), 3))


                             Instance  Original_Cost  Adaptive_Cost  Improvement_%  Original_Time  Adaptive_Time
 001_cflp_small_easy_fac16_cli75.json      18007.527      18007.527          0.000          0.736          0.702
 002_cflp_small_easy_fac20_cli65.json      11102.691      11102.691          0.000          0.593          0.575
 003_cflp_small_easy_fac18_cli62.json      11950.818      11950.818          0.000          0.458          0.530
049_cflp_medium_easy_fac22_cli65.json      10655.492      10655.492          0.000          0.825          0.834
050_cflp_medium_easy_fac19_cli72.json      12146.898      12146.898          0.000          0.713          0.770
051_cflp_medium_easy_fac24_cli74.json      13111.886      13105.234          0.051          1.070          1.292

Mean cost improvement (%): 0.008
Mean original time: 0.732
Mean adaptive time: 0.784


In [13]:

# BONUS A2 - Aspiration+ Criterion

import ast
import copy

ASPIRATION_MARGIN = 0.002  # Require 0.2% improvement

def select_best_candidate_aspiration_plus(
    candidates, tabu_list, iteration, best_cost,
    fixed_costs, service_costs
):
    selected = None
    selected_cost = float("inf")

    for opened, assignments, move_key in candidates:
        cost = compute_total_cost(
            fixed_costs, service_costs, opened, assignments
        )

        inverse_key = get_inverse_move(move_key)
        is_tabu = (
            tabu_list.get(inverse_key, 0) > iteration
        )

        strong_improvement = (
            cost < best_cost * (1 - ASPIRATION_MARGIN)
        )

        if not is_tabu or strong_improvement:
            if cost < selected_cost:
                selected_cost = cost
                selected = (
                    opened, assignments, move_key, cost
                )

    return selected


# Reuse the original TS algorithm with the new criterion
original_source = "".join(
    original_notebook["cells"][17]["source"]
)

tree = ast.parse(original_source)

original_function = next(
    node for node in tree.body
    if isinstance(node, ast.FunctionDef)
    and node.name == "tabu_search_solve_cflp"
)

new_function = copy.deepcopy(original_function)
new_function.name = "tabu_search_aspiration_plus"

class ReplaceSelector(ast.NodeTransformer):
    def visit_Call(self, node):
        self.generic_visit(node)
        if (
            isinstance(node.func, ast.Name)
            and node.func.id == "select_best_candidate"
        ):
            node.func.id = "select_best_candidate_aspiration_plus"
        return node

new_function = ReplaceSelector().visit(new_function)
module = ast.fix_missing_locations(
    ast.Module(body=[new_function], type_ignores=[])
)

exec(
    compile(module, "<aspiration_plus>", "exec"),
    globals()
)

print("Aspiration+ implementation loaded successfully!")
print("Aspiration margin:", ASPIRATION_MARGIN)


Aspiration+ implementation loaded successfully!
Aspiration margin: 0.002


In [14]:

# BONUS A2 - Aspiration+ Pilot Experiment

import json
import time
import random
import numpy as np
import pandas as pd
import contextlib
import io
import os

aspiration_results = []

for file_path in pilot_files:

    with open(file_path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    inst = {
        "num_facilities": int(raw["num_facilities"]),
        "num_clients": int(raw["num_clients"]),
        "fixed_costs": np.array(raw["fixed_costs"]),
        "service_costs": np.array(raw["service_costs"]),
        "facility_capacities": np.array(raw["facility_capacities"]),
        "client_demands": np.array(raw["client_demands"])
    }

    results = []

    for solver in [
        tabu_search_solve_cflp,
        tabu_search_aspiration_plus
    ]:

        random.seed(BONUS_SEED)
        np.random.seed(BONUS_SEED)

        start = time.perf_counter()

        with contextlib.redirect_stdout(io.StringIO()):
            solution, trajectory = solver(
                inst,
                max_iterations=200,
                tabu_tenure=10,
                max_candidates=20,
                max_no_improve=40
            )

        elapsed = time.perf_counter() - start

        assert is_feasible(
            solution[0], solution[1],
            inst["facility_capacities"],
            inst["client_demands"]
        ), "Infeasible solution"

        results.append((trajectory[0][-1], elapsed))

    original_cost, original_time = results[0]
    improved_cost, improved_time = results[1]

    aspiration_results.append({
        "Instance": os.path.basename(file_path),
        "Original_Cost": original_cost,
        "Aspiration_Cost": improved_cost,
        "Improvement_%": (
            (original_cost - improved_cost) / original_cost
        ) * 100,
        "Original_Time": original_time,
        "Aspiration_Time": improved_time
    })

aspiration_df = pd.DataFrame(aspiration_results)

print(aspiration_df.round(3).to_string(index=False))

print("\nMean cost improvement (%):",
      round(aspiration_df["Improvement_%"].mean(), 3))

print("Mean original runtime:",
      round(aspiration_df["Original_Time"].mean(), 3))

print("Mean Aspiration+ runtime:",
      round(aspiration_df["Aspiration_Time"].mean(), 3))


                             Instance  Original_Cost  Aspiration_Cost  Improvement_%  Original_Time  Aspiration_Time
 001_cflp_small_easy_fac16_cli75.json      18007.527        18007.527            0.0          0.627            0.487
 002_cflp_small_easy_fac20_cli65.json      11102.691        11102.691            0.0          0.573            0.542
 003_cflp_small_easy_fac18_cli62.json      11950.818        11950.818            0.0          0.501            0.504
049_cflp_medium_easy_fac22_cli65.json      10655.492        10655.492            0.0          0.644            0.627
050_cflp_medium_easy_fac19_cli72.json      12146.898        12146.898            0.0          0.606            0.602
051_cflp_medium_easy_fac24_cli74.json      13111.886        13111.886            0.0          0.836            0.880

Mean cost improvement (%): 0.0
Mean original runtime: 0.631
Mean Aspiration+ runtime: 0.607


In [15]:

# BONUS A3 - Frequency-based Memory

import ast
import copy

FREQUENCY_PENALTY = 0.001
frequency_counts = {}

def select_best_candidate_frequency(
    candidates, tabu_list, iteration, best_cost,
    fixed_costs, service_costs
):
    selected = None
    best_score = float("inf")

    for opened, assignments, move_key in candidates:

        cost = compute_total_cost(
            fixed_costs, service_costs,
            opened, assignments
        )

        inverse_key = get_inverse_move(move_key)
        is_tabu = tabu_list.get(inverse_key, 0) > iteration

        # Preserve original aspiration rule
        if is_tabu and cost >= best_cost:
            continue

        frequency = frequency_counts.get(move_key, 0)

        penalty = (
            FREQUENCY_PENALTY
            * best_cost
            * frequency
        )

        score = cost + penalty

        if score < best_score:
            best_score = score
            selected = (
                opened, assignments, move_key, cost
            )

    if selected is not None:
        chosen_move = selected[2]
        frequency_counts[chosen_move] = (
            frequency_counts.get(chosen_move, 0) + 1
        )

    return selected


# Create an independent version of original TS

source = "".join(
    original_notebook["cells"][17]["source"]
)

tree = ast.parse(source)

original_function = next(
    node for node in tree.body
    if isinstance(node, ast.FunctionDef)
    and node.name == "tabu_search_solve_cflp"
)

new_function = copy.deepcopy(original_function)
new_function.name = "_tabu_search_frequency_core"

class ReplaceFrequencySelector(ast.NodeTransformer):
    def visit_Call(self, node):
        self.generic_visit(node)

        if (
            isinstance(node.func, ast.Name)
            and node.func.id == "select_best_candidate"
        ):
            node.func.id = "select_best_candidate_frequency"

        return node


new_function = ReplaceFrequencySelector().visit(new_function)

module = ast.fix_missing_locations(
    ast.Module(
        body=[new_function],
        type_ignores=[]
    )
)

exec(
    compile(module, "<frequency_memory>", "exec"),
    globals()
)

def tabu_search_frequency_memory(
    instance,
    max_iterations=200,
    tabu_tenure=10,
    max_candidates=20,
    max_no_improve=40
):
    frequency_counts.clear()

    return _tabu_search_frequency_core(
        instance,
        max_iterations=max_iterations,
        tabu_tenure=tabu_tenure,
        max_candidates=max_candidates,
        max_no_improve=max_no_improve
    )


assert (
    "select_best_candidate_frequency"
    in _tabu_search_frequency_core.__code__.co_names
), "Frequency selector was not installed"

print("Frequency-based Memory loaded successfully!")
print("Frequency penalty:", FREQUENCY_PENALTY)
print("Original Tabu Search remains unchanged.")


Frequency-based Memory loaded successfully!
Frequency penalty: 0.001
Original Tabu Search remains unchanged.


In [16]:

# BONUS A3 - Frequency Memory Pilot Experiment

frequency_results = []

for file_path in pilot_files:

    with open(file_path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    inst = {
        "num_facilities": int(raw["num_facilities"]),
        "num_clients": int(raw["num_clients"]),
        "fixed_costs": np.array(raw["fixed_costs"]),
        "service_costs": np.array(raw["service_costs"]),
        "facility_capacities": np.array(raw["facility_capacities"]),
        "client_demands": np.array(raw["client_demands"])
    }

    random.seed(BONUS_SEED)
    np.random.seed(BONUS_SEED)

    start = time.perf_counter()

    with contextlib.redirect_stdout(io.StringIO()):
        solution, trajectory = tabu_search_frequency_memory(
            inst,
            max_iterations=200,
            tabu_tenure=10,
            max_candidates=20,
            max_no_improve=40
        )

    elapsed = time.perf_counter() - start

    assert is_feasible(
        solution[0], solution[1],
        inst["facility_capacities"],
        inst["client_demands"]
    ), "Infeasible solution"

    original_row = pilot_df.loc[
        pilot_df["Instance"] == os.path.basename(file_path)
    ].iloc[0]

    original_cost = original_row["Original_Cost"]
    frequency_cost = trajectory[0][-1]

    frequency_results.append({
        "Instance": os.path.basename(file_path),
        "Original_Cost": original_cost,
        "Frequency_Cost": frequency_cost,
        "Improvement_%": (
            (original_cost - frequency_cost) / original_cost
        ) * 100,
        "Original_Time": original_row["Original_Time"],
        "Frequency_Time": elapsed
    })

frequency_df = pd.DataFrame(frequency_results)

print(frequency_df.round(3).to_string(index=False))

print("\nMean cost improvement (%):",
      round(frequency_df["Improvement_%"].mean(), 3))

print("Mean original runtime:",
      round(frequency_df["Original_Time"].mean(), 3))

print("Mean Frequency Memory runtime:",
      round(frequency_df["Frequency_Time"].mean(), 3))


                             Instance  Original_Cost  Frequency_Cost  Improvement_%  Original_Time  Frequency_Time
 001_cflp_small_easy_fac16_cli75.json      18007.527       18007.527            0.0          0.736           0.613
 002_cflp_small_easy_fac20_cli65.json      11102.691       11102.691            0.0          0.593           0.585
 003_cflp_small_easy_fac18_cli62.json      11950.818       11950.818            0.0          0.458           0.518
049_cflp_medium_easy_fac22_cli65.json      10655.492       10655.492            0.0          0.825           0.807
050_cflp_medium_easy_fac19_cli72.json      12146.898       12146.898            0.0          0.713           0.583
051_cflp_medium_easy_fac24_cli74.json      13111.886       13111.886            0.0          1.070           0.872

Mean cost improvement (%): 0.0
Mean original runtime: 0.732
Mean Frequency Memory runtime: 0.663


In [17]:

# BONUS A4 - Diversification Restart

def tabu_search_diversification_restart(
    instance,
    max_iterations=200,
    tabu_tenure=10,
    max_candidates=20,
    max_no_improve=40
):
    fixed = instance["fixed_costs"]
    service = instance["service_costs"]
    capacities = instance["facility_capacities"]
    demands = instance["client_demands"]

    current = tabu_initial_solution(instance)
    best = copy_solution(current[0], current[1])

    best_cost = compute_total_cost(
        fixed, service, best[0], best[1]
    )

    trajectory = [best_cost]
    trajectory_iterations = [0]
    tabu_list = {}

    stagnation = 0
    restarts = 0
    max_restarts = 2
    restart_threshold = max(5, max_no_improve // 2)

    for iteration in range(1, max_iterations + 1):

        candidates = generate_all_candidates(
            current, service, capacities,
            demands, max_candidates
        )

        selected = select_best_candidate(
            candidates, tabu_list, iteration,
            best_cost, fixed, service
        ) if candidates else None

        if selected is not None:
            opened, assignments, move_key, cost = selected

            current = (opened, assignments)
            inverse = get_inverse_move(move_key)
            tabu_list[inverse] = iteration + tabu_tenure
            tabu_list = {
                k: v for k, v in tabu_list.items()
                if v > iteration
            }

            if cost < best_cost:
                best = copy_solution(opened, assignments)
                best_cost = cost
                trajectory.append(best_cost)
                trajectory_iterations.append(iteration)
                stagnation = 0
            else:
                stagnation += 1
        else:
            stagnation += 1

        if stagnation >= restart_threshold:
            if restarts >= max_restarts:
                break

            # Restart from best solution
            new_current = copy_solution(best[0], best[1])
            perturbations = 0

            # Apply up to 3 feasible facility swaps
            for _ in range(3):
                swaps = generate_swap_moves(
                    new_current[0], service,
                    capacities, demands, 30
                )

                if not swaps:
                    break

                chosen = swaps[
                    np.random.randint(len(swaps))
                ]

                new_current = (
                    chosen[0].copy(),
                    chosen[1].copy()
                )
                perturbations += 1

            if perturbations == 0:
                break

            current = new_current
            tabu_list.clear()
            stagnation = 0
            restarts += 1

    print("Diversification restarts:", restarts)

    return best, (trajectory, trajectory_iterations)


print("Diversification Restart implementation loaded!")
print("Maximum restarts: 2")
print("Original Tabu Search remains unchanged.")


Diversification Restart implementation loaded!
Maximum restarts: 2
Original Tabu Search remains unchanged.


In [18]:

# BONUS A4 - Diversification Restart Pilot Experiment

diversification_results = []

for file_path in pilot_files:

    with open(file_path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    inst = {
        "num_facilities": int(raw["num_facilities"]),
        "num_clients": int(raw["num_clients"]),
        "fixed_costs": np.array(raw["fixed_costs"]),
        "service_costs": np.array(raw["service_costs"]),
        "facility_capacities": np.array(raw["facility_capacities"]),
        "client_demands": np.array(raw["client_demands"])
    }

    random.seed(BONUS_SEED)
    np.random.seed(BONUS_SEED)

    start = time.perf_counter()

    with contextlib.redirect_stdout(io.StringIO()) as output:
        solution, trajectory = tabu_search_diversification_restart(
            inst,
            max_iterations=200,
            tabu_tenure=10,
            max_candidates=20,
            max_no_improve=40
        )

    elapsed = time.perf_counter() - start

    assert is_feasible(
        solution[0],
        solution[1],
        inst["facility_capacities"],
        inst["client_demands"]
    ), "Infeasible solution"

    baseline_row = pilot_df.loc[
        pilot_df["Instance"] == os.path.basename(file_path)
    ].iloc[0]

    original_cost = baseline_row["Original_Cost"]
    improved_cost = trajectory[0][-1]

    diversification_results.append({
        "Instance": os.path.basename(file_path),
        "Original_Cost": original_cost,
        "Diversification_Cost": improved_cost,
        "Improvement_%": (
            (original_cost - improved_cost) / original_cost
        ) * 100,
        "Original_Time": baseline_row["Original_Time"],
        "Diversification_Time": elapsed
    })

diversification_df = pd.DataFrame(diversification_results)

print(diversification_df.round(3).to_string(index=False))

print("\nMean cost improvement (%):",
      round(diversification_df["Improvement_%"].mean(), 3))

print("Mean original runtime:",
      round(diversification_df["Original_Time"].mean(), 3))

print("Mean Diversification runtime:",
      round(diversification_df["Diversification_Time"].mean(), 3))


                             Instance  Original_Cost  Diversification_Cost  Improvement_%  Original_Time  Diversification_Time
 001_cflp_small_easy_fac16_cli75.json      18007.527             18007.527            0.0          0.736                 1.157
 002_cflp_small_easy_fac20_cli65.json      11102.691             11102.691            0.0          0.593                 1.063
 003_cflp_small_easy_fac18_cli62.json      11950.818             11950.818            0.0          0.458                 0.732
049_cflp_medium_easy_fac22_cli65.json      10655.492             10655.492            0.0          0.825                 0.951
050_cflp_medium_easy_fac19_cli72.json      12146.898             12146.898            0.0          0.713                 0.911
051_cflp_medium_easy_fac24_cli74.json      13111.886             13111.886            0.0          1.070                 1.186

Mean cost improvement (%): 0.0
Mean original runtime: 0.732
Mean Diversification runtime: 1.0


In [19]:

# BONUS A4 - Verify Diversification Restarts

import json
import numpy as np
import random

with open(pilot_files[0], "r", encoding="utf-8") as f:
    raw = json.load(f)

test_instance = {
    "num_facilities": int(raw["num_facilities"]),
    "num_clients": int(raw["num_clients"]),
    "fixed_costs": np.array(raw["fixed_costs"]),
    "service_costs": np.array(raw["service_costs"]),
    "facility_capacities": np.array(raw["facility_capacities"]),
    "client_demands": np.array(raw["client_demands"])
}

random.seed(BONUS_SEED)
np.random.seed(BONUS_SEED)

solution, trajectory = tabu_search_diversification_restart(
    test_instance,
    max_iterations=200,
    tabu_tenure=10,
    max_candidates=20,
    max_no_improve=40
)

print("Best Cost:", round(trajectory[0][-1], 3))
print("Feasible:", is_feasible(
    solution[0],
    solution[1],
    test_instance["facility_capacities"],
    test_instance["client_demands"]
))


Diversification restarts: 2
Best Cost: 18007.527
Feasible: True


In [20]:

# BONUS A - Summary of Four Enhancements

bonus_a_summary = pd.DataFrame({
    "Enhancement": [
        "Adaptive Tenure",
        "Aspiration+",
        "Frequency-based Memory",
        "Diversification Restart"
    ],
    "Mean_Cost_Improvement_%": [
        pilot_df["Improvement_%"].mean(),
        aspiration_df["Improvement_%"].mean(),
        frequency_df["Improvement_%"].mean(),
        diversification_df["Improvement_%"].mean()
    ],
    "Mean_Runtime_s": [
        pilot_df["Adaptive_Time"].mean(),
        aspiration_df["Aspiration_Time"].mean(),
        frequency_df["Frequency_Time"].mean(),
        diversification_df["Diversification_Time"].mean()
    ]
})

print("===== BONUS A SUMMARY =====")
print(bonus_a_summary.round(4).to_string(index=False))

print("\n===== ORIGINAL RESULTS COLUMNS =====")
original_results = pd.read_csv(
    "results_student_447540190.csv"
)

print(original_results.columns.tolist())
print("\nFirst two rows:")
print(original_results.head(2).to_string(index=False))


===== BONUS A SUMMARY =====
            Enhancement  Mean_Cost_Improvement_%  Mean_Runtime_s
        Adaptive Tenure                   0.0085          0.7839
            Aspiration+                   0.0000          0.6071
 Frequency-based Memory                   0.0000          0.6630
Diversification Restart                   0.0000          1.0001

===== ORIGINAL RESULTS COLUMNS =====
['instance_id', 'instance_name', 'size', 'difficulty', 'num_facilities', 'num_clients', 'gurobi_cost', 'gurobi_time', 'gurobi_status', 'ts_cost', 'ts_time', 'ts_iterations', 'ts_status', 'ts_gap_percent', 'ts_speedup_factor', 'ga_cost', 'ga_time', 'ga_generations', 'ga_status', 'ga_gap_percent', 'ga_speedup_factor']

First two rows:
 instance_id                        instance_name  size difficulty  num_facilities  num_clients  gurobi_cost  gurobi_time gurobi_status      ts_cost  ts_time  ts_iterations ts_status  ts_gap_percent  ts_speedup_factor      ga_cost  ga_time  ga_generations ga_status  ga_gap_

In [21]:

# BONUS A - Final Pilot Gap Comparison

import pandas as pd
import numpy as np

# Gurobi reference costs from original results
reference = original_results[
    ["instance_name", "gurobi_cost"]
].copy()

reference["gurobi_cost"] = pd.to_numeric(
    reference["gurobi_cost"], errors="coerce"
)

# Collect results from the four enhancements
comparison = pilot_df[
    ["Instance", "Original_Cost", "Adaptive_Cost"]
].copy()

comparison = comparison.merge(
    aspiration_df[["Instance", "Aspiration_Cost"]],
    on="Instance",
    validate="one_to_one"
)

comparison = comparison.merge(
    frequency_df[["Instance", "Frequency_Cost"]],
    on="Instance",
    validate="one_to_one"
)

comparison = comparison.merge(
    diversification_df[
        ["Instance", "Diversification_Cost"]
    ],
    on="Instance",
    validate="one_to_one"
)

comparison = comparison.merge(
    reference,
    left_on="Instance",
    right_on="instance_name",
    how="left",
    validate="one_to_one"
)

assert comparison["gurobi_cost"].notna().all(), \
    "Some Gurobi reference costs are missing"

assert (comparison["gurobi_cost"] > 0).all(), \
    "Invalid Gurobi reference cost"

methods = {
    "Original TS": "Original_Cost",
    "Adaptive Tenure": "Adaptive_Cost",
    "Aspiration+": "Aspiration_Cost",
    "Frequency Memory": "Frequency_Cost",
    "Diversification": "Diversification_Cost"
}

gap_summary = []

for method, cost_column in methods.items():
    gaps = (
        (comparison[cost_column] - comparison["gurobi_cost"])
        / comparison["gurobi_cost"]
    ) * 100

    gap_summary.append({
        "Method": method,
        "Mean_Gap_%": gaps.mean()
    })

gap_df = pd.DataFrame(gap_summary)

baseline_gap = gap_df.loc[
    gap_df["Method"] == "Original TS",
    "Mean_Gap_%"
].iloc[0]

gap_df["Gap_Reduction_pp"] = (
    baseline_gap - gap_df["Mean_Gap_%"]
)

print("===== BONUS A GAP COMPARISON =====")
print(gap_df.round(4).to_string(index=False))

print("\nBest pilot enhancement:")
best = gap_df[gap_df["Method"] != "Original TS"]
best = best.sort_values("Mean_Gap_%")
print(best.iloc[0]["Method"])


===== BONUS A GAP COMPARISON =====
          Method  Mean_Gap_%  Gap_Reduction_pp
     Original TS      6.8877            0.0000
 Adaptive Tenure      6.8783            0.0094
     Aspiration+      6.8877            0.0000
Frequency Memory      6.8877            0.0000
 Diversification      6.8877            0.0000

Best pilot enhancement:
Adaptive Tenure


In [ ]:

# BONUS A - Full 144 Instance Experiment
# Selected enhancement: Adaptive Tenure


import os
import json
import time
import random
import numpy as np
import pandas as pd

bonus_a_full_results = []

baseline = pd.read_csv(
    "results_student_447540190.csv"
).copy()

baseline["instance_key"] = baseline["instance_name"].apply(
    lambda x: os.path.basename(str(x))
)

assert baseline["instance_key"].is_unique
assert len(instance_files) == 144

for index, file_path in enumerate(sorted(instance_files), 1):

    with open(file_path, encoding="utf-8") as f:
        raw = json.load(f)

    inst = {
        "num_facilities": int(raw["num_facilities"]),
        "num_clients": int(raw["num_clients"]),
        "fixed_costs": np.array(raw["fixed_costs"]),
        "service_costs": np.array(raw["service_costs"]),
        "facility_capacities": np.array(raw["facility_capacities"]),
        "client_demands": np.array(raw["client_demands"])
    }

    name = os.path.basename(file_path)

    original = baseline.loc[
        baseline["instance_key"] == name
    ]

    assert len(original) == 1, f"Missing baseline: {name}"
    original = original.iloc[0]

    random.seed(BONUS_SEED + index)
    np.random.seed(BONUS_SEED + index)

    start = time.perf_counter()

    solution, trajectory = tabu_search_adaptive_tenure(
        inst,
        max_iterations=200,
        tabu_tenure=10,
        max_candidates=20,
        max_no_improve=40
    )

    elapsed = time.perf_counter() - start

    feasible = is_feasible(
        solution[0], solution[1],
        inst["facility_capacities"],
        inst["client_demands"]
    )

    assert feasible, f"Infeasible solution: {name}"

    improved_cost = compute_total_cost(
        inst["fixed_costs"],
        inst["service_costs"],
        solution[0],
        solution[1]
    )

    reference_cost = float(original["gurobi_cost"])
    original_cost = float(original["ts_cost"])

    valid_reference = (
        original["gurobi_status"] == "Optimal"
        and np.isfinite(reference_cost)
        and reference_cost > 0
    )

    original_gap = (
        (original_cost - reference_cost)
        / reference_cost * 100
        if valid_reference else np.nan
    )

    improved_gap = (
        (improved_cost - reference_cost)
        / reference_cost * 100
        if valid_reference else np.nan
    )

    bonus_a_full_results.append({
        "instance_name": name,
        "size": original["size"],
        "difficulty": original["difficulty"],
        "gurobi_cost": reference_cost,
        "original_ts_cost": original_cost,
        "improved_ts_cost": improved_cost,
        "original_ts_time": original["ts_time"],
        "improved_ts_time": elapsed,
        "original_gap_percent": original_gap,
        "improved_gap_percent": improved_gap,
        "gap_reduction_pp": original_gap - improved_gap,
        "feasible": feasible
    })

    if index % 12 == 0 or index == 144:
        print(f"Completed {index}/144 instances")

bonus_a_full_df = pd.DataFrame(bonus_a_full_results)

bonus_a_full_df.to_csv(
    "bonus_A_adaptive_student_447540190.csv",
    index=False
)

bonus_a_full_df.to_excel(
    "bonus_A_adaptive_student_447540190.xlsx",
    index=False
)

valid = bonus_a_full_df.dropna(
    subset=["original_gap_percent", "improved_gap_percent"]
)

print("\n===== BONUS A FULL RESULTS =====")
print("Instances:", len(bonus_a_full_df))
print("Valid gap comparisons:", len(valid))
print(
    "Original mean gap:",
    round(valid["original_gap_percent"].mean(), 4)
)
print(
    "Improved mean gap:",
    round(valid["improved_gap_percent"].mean(), 4)
)
print(
    "Mean gap reduction (percentage points):",
    round(valid["gap_reduction_pp"].mean(), 4)
)
print(
    "Mean original runtime:",
    round(bonus_a_full_df["original_ts_time"].mean(), 4)
)
print(
    "Mean improved runtime:",
    round(bonus_a_full_df["improved_ts_time"].mean(), 4)
)
print("Bonus A results saved successfully!")


In [28]:

# BONUS B - Inspect original Genetic Algorithm functions

import inspect

ga_functions = [
    name for name, obj in globals().items()
    if callable(obj)
    and ("genetic" in name.lower()
         or name.lower().startswith("ga_"))
]

print("===== GENETIC ALGORITHM FUNCTIONS =====")

for name in sorted(ga_functions):
    try:
        print(name, inspect.signature(globals()[name]))
    except (TypeError, ValueError):
        print(name, "(signature unavailable)")

print("\nBonus B inspection completed!")


===== GENETIC ALGORITHM FUNCTIONS =====

Bonus B inspection completed!


In [2]:

# BONUS B - Load Original Genetic Algorithm Functions

import json
import ast
import inspect
import numpy as np
from typing import Tuple, List

with open("Assignment_2.ipynb", encoding="utf-8") as f:
    original_notebook = json.load(f)

source = "".join(
    original_notebook["cells"][19]["source"]
)

tree = ast.parse(source)

ga_nodes = [
    node for node in tree.body
    if isinstance(node, ast.FunctionDef)
]

module = ast.fix_missing_locations(
    ast.Module(body=ga_nodes, type_ignores=[])
)

exec(
    compile(module, "<original_ga>", "exec"),
    globals()
)

print("===== GENETIC ALGORITHM FUNCTIONS =====")

for node in ga_nodes:
    function = globals()[node.name]
    print(node.name, inspect.signature(function))

print("\nOriginal GA functions loaded successfully!")


===== GENETIC ALGORITHM FUNCTIONS =====
repair_chromosome (facility_opened: numpy.ndarray, service_costs: numpy.ndarray, facility_capacities: numpy.ndarray, client_demands: numpy.ndarray) -> Tuple[numpy.ndarray, numpy.ndarray]
initialize_ga_population (population_size: int, num_facilities: int, service_costs: numpy.ndarray, facility_capacities: numpy.ndarray, client_demands: numpy.ndarray, fixed_costs: numpy.ndarray, num_clients: int) -> List[numpy.ndarray]
evaluate_chromosome (chromosome: numpy.ndarray, fixed_costs: numpy.ndarray, service_costs: numpy.ndarray, facility_capacities: numpy.ndarray, client_demands: numpy.ndarray) -> float
tournament_selection (population: List[numpy.ndarray], fitness: List[float], tournament_size: int = 3) -> numpy.ndarray
uniform_crossover (parent1: numpy.ndarray, parent2: numpy.ndarray) -> Tuple[numpy.ndarray, numpy.ndarray]
mutate_chromosome (chromosome: numpy.ndarray, mutation_rate: float) -> numpy.ndarray
genetic_algorithm_solve_cflp (instance: dict,

In [52]:

# BONUS B1 - Adaptive Mutation

import ast
import copy
import numpy as np

def ga_adaptive_mutation_rate(population, base_rate):
    # Measure unique chromosome ratio
    unique = len({
        tuple(chromosome.tolist())
        for chromosome in population
    })

    diversity = unique / len(population)

    if diversity < 0.35:
        rate = min(0.30, base_rate * 2.5)
    elif diversity > 0.75:
        rate = max(0.02, base_rate * 0.5)
    else:
        rate = base_rate

    return rate


# Copy the original GA function
source = "".join(
    original_notebook["cells"][19]["source"]
)

tree = ast.parse(source)

original_ga_node = next(
    node for node in tree.body
    if isinstance(node, ast.FunctionDef)
    and node.name == "genetic_algorithm_solve_cflp"
)

adaptive_ga_node = copy.deepcopy(original_ga_node)
adaptive_ga_node.name = "genetic_algorithm_adaptive_mutation"


class AdaptiveMutationTransformer(ast.NodeTransformer):

    def visit_For(self, node):
        self.generic_visit(node)

        if (
            isinstance(node.target, ast.Name)
            and node.target.id == "generation"
        ):
            setup = ast.parse(
                "adaptive_rate = ga_adaptive_mutation_rate("
                "population, mutation_rate)"
            ).body[0]

            node.body.insert(0, setup)

        return node

    def visit_Call(self, node):
        self.generic_visit(node)

        if (
            isinstance(node.func, ast.Name)
            and node.func.id == "mutate_chromosome"
            and len(node.args) == 2
        ):
            node.args[1] = ast.Name(
                id="adaptive_rate",
                ctx=ast.Load()
            )

        return node


adaptive_ga_node = AdaptiveMutationTransformer().visit(
    adaptive_ga_node
)

module = ast.fix_missing_locations(
    ast.Module(
        body=[adaptive_ga_node],
        type_ignores=[]
    )
)

exec(
    compile(module, "<adaptive_mutation_ga>", "exec"),
    globals()
)

# Verify the adaptive rate is used in mutation
assert (
    "ga_adaptive_mutation_rate"
    in genetic_algorithm_adaptive_mutation.__code__.co_names
)

print("Adaptive Mutation GA loaded successfully!")
print("Original Genetic Algorithm remains unchanged.")
print("Ready for pilot testing.")


Adaptive Mutation GA loaded successfully!
Original Genetic Algorithm remains unchanged.
Ready for pilot testing.


In [31]:

# BONUS B1 - Verify Adaptive Mutation Setup

import inspect

print("===== ORIGINAL GA =====")
print(inspect.signature(genetic_algorithm_solve_cflp))

print("\n===== ADAPTIVE GA =====")
print(inspect.signature(genetic_algorithm_adaptive_mutation))

print("\n===== MUTATION FUNCTION =====")
print(inspect.signature(mutate_chromosome))

print("\n===== ADAPTIVE RATE TEST =====")

test_population = [
    np.array([1, 0, 1, 0])
    for _ in range(20)
]

test_rate = ga_adaptive_mutation_rate(
    test_population, 0.10
)

print("Original mutation rate:", 0.10)
print("Adaptive mutation rate:", test_rate)
print("Verification completed!")


===== ORIGINAL GA =====
(instance: dict, population_size: int = 30, max_generations: int = 100, crossover_rate: float = 0.85, mutation_rate: float = 0.1, tournament_size: int = 3, elitism: int = 2, max_no_improve: int = 30) -> Tuple[Tuple[numpy.ndarray, numpy.ndarray], Tuple[List[float], List[int]]]

===== ADAPTIVE GA =====
(instance: dict, population_size: int = 30, max_generations: int = 100, crossover_rate: float = 0.85, mutation_rate: float = 0.1, tournament_size: int = 3, elitism: int = 2, max_no_improve: int = 30) -> Tuple[Tuple[numpy.ndarray, numpy.ndarray], Tuple[List[float], List[int]]]

===== MUTATION FUNCTION =====
(chromosome: numpy.ndarray, mutation_rate: float) -> numpy.ndarray

===== ADAPTIVE RATE TEST =====
Original mutation rate: 0.1
Adaptive mutation rate: 0.25
Verification completed!


In [32]:

# BONUS B1 - Verify mutation calls

import ast

print("===== ADAPTIVE MUTATION CHECK =====")

mutation_calls = [
    node for node in ast.walk(adaptive_ga_node)
    if isinstance(node, ast.Call)
    and isinstance(node.func, ast.Name)
    and node.func.id == "mutate_chromosome"
]

print("Mutation calls found:", len(mutation_calls))

for i, call in enumerate(mutation_calls, 1):
    print(f"\nMutation call {i}:")
    print(ast.unparse(call))

    uses_adaptive = any(
        isinstance(node, ast.Name)
        and node.id == "adaptive_rate"
        for node in ast.walk(call)
    )

    print("Uses adaptive rate:", uses_adaptive)

print("\n===== CHECK FINISHED =====")


===== ADAPTIVE MUTATION CHECK =====
Mutation calls found: 2

Mutation call 1:
mutate_chromosome(c1, adaptive_rate)
Uses adaptive rate: True

Mutation call 2:
mutate_chromosome(c2, adaptive_rate)
Uses adaptive rate: True

===== CHECK FINISHED =====


In [33]:

# BONUS B1 - First Real Test: Adaptive Mutation

import json
import os
import time
import random
import numpy as np
import contextlib
import io

# Use the first small instance
test_file = sorted(pilot_files)[0]

with open(test_file, "r", encoding="utf-8") as f:
    raw = json.load(f)

ga_instance = {
    "num_facilities": int(raw["num_facilities"]),
    "num_clients": int(raw["num_clients"]),
    "fixed_costs": np.array(raw["fixed_costs"]),
    "service_costs": np.array(raw["service_costs"]),
    "facility_capacities": np.array(raw["facility_capacities"]),
    "client_demands": np.array(raw["client_demands"])
}

# Same random seed for both methods
random.seed(BONUS_SEED)
np.random.seed(BONUS_SEED)

start = time.perf_counter()

with contextlib.redirect_stdout(io.StringIO()):
    original_ga_result = genetic_algorithm_solve_cflp(
        ga_instance,
        population_size=30,
        max_generations=100,
        crossover_rate=0.85,
        mutation_rate=0.10
    )

original_time = time.perf_counter() - start

random.seed(BONUS_SEED)
np.random.seed(BONUS_SEED)

start = time.perf_counter()

with contextlib.redirect_stdout(io.StringIO()):
    adaptive_ga_result = genetic_algorithm_adaptive_mutation(
        ga_instance,
        population_size=30,
        max_generations=100,
        crossover_rate=0.85,
        mutation_rate=0.10
    )

adaptive_time = time.perf_counter() - start

print("===== BONUS B1 FIRST TEST =====")
print("Instance:", os.path.basename(test_file))
print("Original result type:", type(original_ga_result))
print("Adaptive result type:", type(adaptive_ga_result))
print("Original runtime:", round(original_time, 3), "s")
print("Adaptive runtime:", round(adaptive_time, 3), "s")
print("First test completed!")


===== BONUS B1 FIRST TEST =====
Instance: 001_cflp_small_easy_fac16_cli75.json
Original result type: <class 'tuple'>
Adaptive result type: <class 'tuple'>
Original runtime: 1.111 s
Adaptive runtime: 0.635 s
First test completed!


In [34]:

# BONUS B1 - Inspect GA Results

import numpy as np

def inspect_ga_result(name, result):
    print(f"\n===== {name} =====")
    print("Number of returned items:", len(result))

    for i, item in enumerate(result):
        print(f"\nItem {i}:")
        print("Type:", type(item).__name__)

        if isinstance(item, np.ndarray):
            print("Shape:", item.shape)
            print("First values:", item.ravel()[:5])

        elif isinstance(item, (list, tuple)):
            print("Length:", len(item))
            print("First values:", str(item[:3])[:200])

        elif isinstance(item, dict):
            print("Keys:", list(item.keys()))

        else:
            print("Value:", str(item)[:200])

inspect_ga_result("Original GA", original_ga_result)
inspect_ga_result("Adaptive GA", adaptive_ga_result)



===== Original GA =====
Number of returned items: 2

Item 0:
Type: tuple
Length: 2
First values: (array([1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1]), array([ 1,  6,  5,  5, 11,  3,  1,  5,  1,  6,  0, 11,  3,  5,  3,  9,  1,
       13,  9,  1,  9,  6,  0,  1,  9, 12, 12,  5,  6,  9,  3,  1, 

Item 1:
Type: tuple
Length: 2
First values: ([np.float64(18648.677482217092), np.float64(18417.579480925928), np.float64(18300.276562287967), np.float64(18130.801189225862), np.float64(18029.546164512132), np.float64(18007.527403107208)], [0, 1

===== Adaptive GA =====
Number of returned items: 2

Item 0:
Type: tuple
Length: 2
First values: (array([1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1]), array([ 1,  6,  5,  5, 11,  3,  1,  5,  1,  6,  0, 11,  3,  5,  3,  9,  1,
       13,  9,  1,  9,  6,  0,  1,  9, 12, 12,  5,  6,  9,  3,  1, 

Item 1:
Type: tuple
Length: 2
First values: ([np.float64(18648.677482217092), np.float64(18547.42245750336), np.float64(18305.862148018226), np.float64(183

In [35]:

# BONUS B1 - Compare Original GA vs Adaptive GA

def evaluate_ga_result(result, instance):
    opened, assignments = result[0]

    feasible = is_feasible(
        opened,
        assignments,
        instance["facility_capacities"],
        instance["client_demands"]
    )

    cost = compute_total_cost(
        instance["fixed_costs"],
        instance["service_costs"],
        opened,
        assignments
    )

    return float(cost), bool(feasible)


original_cost, original_feasible = evaluate_ga_result(
    original_ga_result, ga_instance
)

adaptive_cost, adaptive_feasible = evaluate_ga_result(
    adaptive_ga_result, ga_instance
)

improvement = (
    (original_cost - adaptive_cost) / original_cost
) * 100

print("===== BONUS B1 COMPARISON =====")
print(f"Original GA Cost: {original_cost:.3f}")
print(f"Adaptive GA Cost: {adaptive_cost:.3f}")
print(f"Cost Improvement: {improvement:.3f}%")
print(f"Original GA Feasible: {original_feasible}")
print(f"Adaptive GA Feasible: {adaptive_feasible}")
print(f"Original Runtime: {original_time:.3f} s")
print(f"Adaptive Runtime: {adaptive_time:.3f} s")

assert original_feasible, "Original GA solution is infeasible"
assert adaptive_feasible, "Adaptive GA solution is infeasible"


===== BONUS B1 COMPARISON =====
Original GA Cost: 18007.527
Adaptive GA Cost: 18007.527
Cost Improvement: 0.000%
Original GA Feasible: True
Adaptive GA Feasible: True
Original Runtime: 1.111 s
Adaptive Runtime: 0.635 s


In [36]:

# BONUS B1 - Adaptive Mutation Pilot Experiment

import json
import os
import time
import random
import contextlib
import io
import numpy as np
import pandas as pd

ga_pilot_results = []

for file_path in pilot_files:

    with open(file_path, encoding="utf-8") as f:
        raw = json.load(f)

    inst = {
        "num_facilities": int(raw["num_facilities"]),
        "num_clients": int(raw["num_clients"]),
        "fixed_costs": np.array(raw["fixed_costs"]),
        "service_costs": np.array(raw["service_costs"]),
        "facility_capacities": np.array(
            raw["facility_capacities"]
        ),
        "client_demands": np.array(
            raw["client_demands"]
        )
    }

    results_pair = []

    for solver in [
        genetic_algorithm_solve_cflp,
        genetic_algorithm_adaptive_mutation
    ]:

        random.seed(BONUS_SEED)
        np.random.seed(BONUS_SEED)

        start = time.perf_counter()

        with contextlib.redirect_stdout(io.StringIO()):
            result = solver(
                inst,
                population_size=30,
                max_generations=100,
                crossover_rate=0.85,
                mutation_rate=0.10
            )

        elapsed = time.perf_counter() - start

        cost, feasible = evaluate_ga_result(
            result, inst
        )

        assert feasible, "Infeasible GA solution"

        results_pair.append((cost, elapsed))

    original_cost, original_time = results_pair[0]
    adaptive_cost, adaptive_time = results_pair[1]

    ga_pilot_results.append({
        "Instance": os.path.basename(file_path),
        "Original_GA_Cost": original_cost,
        "Adaptive_GA_Cost": adaptive_cost,
        "Improvement_%": (
            (original_cost - adaptive_cost)
            / original_cost * 100
        ),
        "Original_Time": original_time,
        "Adaptive_Time": adaptive_time
    })

    print("Completed:", os.path.basename(file_path))

ga_pilot_df = pd.DataFrame(ga_pilot_results)

print("\n===== BONUS B1 RESULTS =====")
print(ga_pilot_df.round(3).to_string(index=False))

print(
    "\nMean cost improvement (%):",
    round(ga_pilot_df["Improvement_%"].mean(), 4)
)

print(
    "Mean original runtime:",
    round(ga_pilot_df["Original_Time"].mean(), 3)
)

print(
    "Mean adaptive runtime:",
    round(ga_pilot_df["Adaptive_Time"].mean(), 3)
)

ga_pilot_df.to_csv(
    "bonus_B_adaptive_pilot_447540190.csv",
    index=False
)

print("Pilot results saved!")


Completed: 001_cflp_small_easy_fac16_cli75.json
Completed: 002_cflp_small_easy_fac20_cli65.json
Completed: 003_cflp_small_easy_fac18_cli62.json
Completed: 049_cflp_medium_easy_fac22_cli65.json
Completed: 050_cflp_medium_easy_fac19_cli72.json
Completed: 051_cflp_medium_easy_fac24_cli74.json

===== BONUS B1 RESULTS =====
                             Instance  Original_GA_Cost  Adaptive_GA_Cost  Improvement_%  Original_Time  Adaptive_Time
 001_cflp_small_easy_fac16_cli75.json         18007.527         18007.527          0.000          0.888          0.670
 002_cflp_small_easy_fac20_cli65.json         11178.576         11102.691          0.679          0.935          0.786
 003_cflp_small_easy_fac18_cli62.json         11950.818         11950.818          0.000          0.852          1.175
049_cflp_medium_easy_fac22_cli65.json         10655.492         10655.492          0.000          0.939          0.794
050_cflp_medium_easy_fac19_cli72.json         12146.898         12146.898          0

In [5]:

# Restore original GA function node

import ast
import json
import copy

with open("Assignment_2.ipynb", encoding="utf-8") as f:
    original_notebook = json.load(f)

source = "".join(
    original_notebook["cells"][19]["source"]
)

tree = ast.parse(source)

original_ga_node = next(
    node for node in tree.body
    if isinstance(node, ast.FunctionDef)
    and node.name == "genetic_algorithm_solve_cflp"
)

print("Original GA node restored successfully!")
print("Function:", original_ga_node.name)


Original GA node restored successfully!
Function: genetic_algorithm_solve_cflp


In [6]:

# BONUS B2 - Two-point Crossover

import numpy as np
import ast
import copy

def two_point_crossover(parent1, parent2):
    p1 = np.asarray(parent1)
    p2 = np.asarray(parent2)

    assert p1.shape == p2.shape

    n = len(p1)

    if n < 3:
        return p1.copy(), p2.copy()

    # Select two different crossover points
    point1, point2 = sorted(
        np.random.choice(
            np.arange(1, n),
            size=2,
            replace=False
        )
    )

    child1 = p1.copy()
    child2 = p2.copy()

    # Exchange the segment
    child1[point1:point2] = p2[point1:point2]
    child2[point1:point2] = p1[point1:point2]

    return child1, child2


# Copy the original Genetic Algorithm
new_ga_node = copy.deepcopy(original_ga_node)

new_ga_node.name = (
    "genetic_algorithm_two_point_crossover"
)

class TwoPointTransformer(ast.NodeTransformer):

    def __init__(self):
        self.replacements = 0

    def visit_Call(self, node):
        self.generic_visit(node)

        if (
            isinstance(node.func, ast.Name)
            and node.func.id == "uniform_crossover"
        ):
            node.func.id = "two_point_crossover"
            self.replacements += 1

        return node


transformer = TwoPointTransformer()

new_ga_node = transformer.visit(new_ga_node)

assert transformer.replacements > 0, (
    "No uniform crossover calls were found"
)

module = ast.fix_missing_locations(
    ast.Module(
        body=[new_ga_node],
        type_ignores=[]
    )
)

exec(
    compile(module, "<two_point_ga>", "exec"),
    globals()
)

print("Two-point Crossover GA loaded successfully!")
print(
    "Crossover calls replaced:",
    transformer.replacements
)
print("Original GA remains unchanged.")


Two-point Crossover GA loaded successfully!
Crossover calls replaced: 1
Original GA remains unchanged.


In [1]:

# BONUS B2 - Check Python Session

required = [
    "genetic_algorithm_solve_cflp",
    "genetic_algorithm_two_point_crossover",
    "two_point_crossover",
    "pilot_files",
    "BONUS_SEED",
    "evaluate_ga_result"
]

missing = []

for name in required:
    if name in globals():
        print(name, ": READY")
    else:
        print(name, ": NOT LOADED")
        missing.append(name)

if not missing:
    print("\nREADY FOR BONUS B2 TESTING!")
else:
    print("\nMissing variables:", missing)


genetic_algorithm_solve_cflp : NOT LOADED
genetic_algorithm_two_point_crossover : NOT LOADED
two_point_crossover : NOT LOADED
pilot_files : NOT LOADED
BONUS_SEED : NOT LOADED
evaluate_ga_result : NOT LOADED

Missing variables: ['genetic_algorithm_solve_cflp', 'genetic_algorithm_two_point_crossover', 'two_point_crossover', 'pilot_files', 'BONUS_SEED', 'evaluate_ga_result']


In [7]:

# BONUS B2 - Final readiness check

required = [
    "genetic_algorithm_solve_cflp",
    "genetic_algorithm_two_point_crossover",
    "two_point_crossover",
    "pilot_files",
    "BONUS_SEED",
    "evaluate_ga_result",
    "repair_chromosome",
    "initialize_ga_population",
    "is_feasible",
    "compute_total_cost"
]

missing = []

for name in required:
    if name in globals():
        print(name, ": READY")
    else:
        print(name, ": NOT LOADED")
        missing.append(name)

print("\n===== RESULT =====")

if not missing:
    print("READY FOR TWO-POINT CROSSOVER TEST!")
else:
    print("Missing functions or variables:")
    print(missing)


genetic_algorithm_solve_cflp : READY
genetic_algorithm_two_point_crossover : READY
two_point_crossover : READY
pilot_files : NOT LOADED
BONUS_SEED : NOT LOADED
evaluate_ga_result : NOT LOADED
repair_chromosome : READY
initialize_ga_population : READY
is_feasible : NOT LOADED
compute_total_cost : NOT LOADED

===== RESULT =====
Missing functions or variables:
['pilot_files', 'BONUS_SEED', 'evaluate_ga_result', 'is_feasible', 'compute_total_cost']


In [8]:

# BONUS B2 - Restore missing testing tools

import os
import glob
import json
import ast
import numpy as np

BONUS_SEED = 447540190

INSTANCES_FOLDER = "cflp_instances_student_447540190"

instance_files = sorted(glob.glob(
    os.path.join(INSTANCES_FOLDER, "**", "*.json"),
    recursive=True
))

pilot_files = (
    instance_files[:3] + instance_files[48:51]
)

# Load original helper functions from notebook
with open("Assignment_2.ipynb", encoding="utf-8") as f:
    notebook = json.load(f)

helper_names = {
    "is_feasible",
    "compute_total_cost"
}

nodes = []

for cell in notebook["cells"]:
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    for node in tree.body:
        if isinstance(node, ast.FunctionDef):
            if node.name in helper_names:
                nodes.append(node)

module = ast.fix_missing_locations(
    ast.Module(body=nodes, type_ignores=[])
)

exec(
    compile(module, "<original_helpers>", "exec"),
    globals()
)

def evaluate_ga_result(result, instance):
    opened, assignments = result[0]

    feasible = is_feasible(
        opened,
        assignments,
        instance["facility_capacities"],
        instance["client_demands"]
    )

    cost = compute_total_cost(
        instance["fixed_costs"],
        instance["service_costs"],
        opened,
        assignments
    )

    return float(cost), bool(feasible)

print("Instance files:", len(instance_files))
print("Pilot files:", len(pilot_files))
print("Helpers restored successfully!")


Instance files: 144
Pilot files: 6
Helpers restored successfully!


In [10]:

# BONUS B2 - Restore greedy initialization

import json
import ast

with open("Assignment_2.ipynb", encoding="utf-8") as f:
    notebook = json.load(f)

found = []

for cell in notebook["cells"]:
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    for node in tree.body:
        if (
            isinstance(node, ast.FunctionDef)
            and node.name == "greedy_initial_solution"
        ):
            found.append(node)

assert len(found) == 1, (
    f"Expected one greedy_initial_solution, found {len(found)}"
)

module = ast.fix_missing_locations(
    ast.Module(body=found, type_ignores=[])
)

exec(
    compile(module, "<ga_helpers>", "exec"),
    globals()
)

print("greedy_initial_solution loaded successfully!")


greedy_initial_solution loaded successfully!


In [16]:

# BONUS B2 - Two-point Crossover Pilot Experiment

import json
import os
import time
import random
import contextlib
import io
import numpy as np
import pandas as pd

two_point_results = []

for index, file_path in enumerate(pilot_files):

    with open(file_path, encoding="utf-8") as f:
        raw = json.load(f)

    inst = {
        "num_facilities": int(raw["num_facilities"]),
        "num_clients": int(raw["num_clients"]),
        "fixed_costs": np.array(raw["fixed_costs"]),
        "service_costs": np.array(raw["service_costs"]),
        "facility_capacities": np.array(
            raw["facility_capacities"]
        ),
        "client_demands": np.array(
            raw["client_demands"]
        )
    }

    comparison = []

    for solver in [
        genetic_algorithm_solve_cflp,
        genetic_algorithm_two_point_crossover
    ]:

        random.seed(BONUS_SEED + index)
        np.random.seed(BONUS_SEED + index)

        start = time.perf_counter()

        with contextlib.redirect_stdout(io.StringIO()):
            result = solver(
                inst,
                population_size=30,
                max_generations=100,
                crossover_rate=0.85,
                mutation_rate=0.10
            )

        elapsed = time.perf_counter() - start

        cost, feasible = evaluate_ga_result(
            result, inst
        )

        assert feasible, (
            f"Infeasible GA solution: {os.path.basename(file_path)}"
        )

        comparison.append((cost, elapsed))

    original_cost, original_time = comparison[0]
    two_point_cost, two_point_time = comparison[1]

    two_point_results.append({
        "Instance": os.path.basename(file_path),
        "Original_GA_Cost": original_cost,
        "TwoPoint_GA_Cost": two_point_cost,
        "Improvement_%": (
            (original_cost - two_point_cost)
            / original_cost * 100
        ),
        "Original_Time": original_time,
        "TwoPoint_Time": two_point_time
    })

    print(
        f"Completed {index + 1}/6:",
        os.path.basename(file_path)
    )

two_point_df = pd.DataFrame(two_point_results)

print("\n===== BONUS B2 RESULTS =====")
print(two_point_df.round(3).to_string(index=False))

print(
    "\nMean cost improvement (%):",
    round(two_point_df["Improvement_%"].mean(), 4)
)

print(
    "Mean original runtime:",
    round(two_point_df["Original_Time"].mean(), 3)
)

print(
    "Mean two-point runtime:",
    round(two_point_df["TwoPoint_Time"].mean(), 3)
)

two_point_df.to_csv(
    "bonus_B_two_point_pilot_447540190.csv",
    index=False
)

print("\nTwo-point pilot results saved!")


Completed 1/6: 001_cflp_small_easy_fac16_cli75.json
Completed 2/6: 002_cflp_small_easy_fac20_cli65.json
Completed 3/6: 003_cflp_small_easy_fac18_cli62.json
Completed 4/6: 049_cflp_medium_easy_fac22_cli65.json
Completed 5/6: 050_cflp_medium_easy_fac19_cli72.json
Completed 6/6: 051_cflp_medium_easy_fac24_cli74.json

===== BONUS B2 RESULTS =====
                             Instance  Original_GA_Cost  TwoPoint_GA_Cost  Improvement_%  Original_Time  TwoPoint_Time
 001_cflp_small_easy_fac16_cli75.json         18007.527         18007.527          0.000          0.820          0.802
 002_cflp_small_easy_fac20_cli65.json         11102.691         11158.913         -0.506          0.780          0.567
 003_cflp_small_easy_fac18_cli62.json         11950.818         11950.818          0.000          0.748          1.362
049_cflp_medium_easy_fac22_cli65.json         10655.492         10655.492          0.000          0.919          1.024
050_cflp_medium_easy_fac19_cli72.json         12146.898     

In [13]:

# BONUS B2 - Diagnose missing GA helper

import inspect

functions = [
    "greedy_initial_solution",
    "initialize_ga_population",
    "genetic_algorithm_solve_cflp",
    "genetic_algorithm_two_point_crossover"
]

for name in functions:
    func = globals().get(name)

    print("\nFunction:", name)
    print("Loaded:", callable(func))

    if callable(func):
        print(
            "Greedy helper accessible:",
            callable(
                func.__globals__.get(
                    "greedy_initial_solution"
                )
            )
        )

print("\nDiagnosis completed!")



Function: greedy_initial_solution
Loaded: True
Greedy helper accessible: True

Function: initialize_ga_population
Loaded: True
Greedy helper accessible: True

Function: genetic_algorithm_solve_cflp
Loaded: True
Greedy helper accessible: True

Function: genetic_algorithm_two_point_crossover
Loaded: True
Greedy helper accessible: True

Diagnosis completed!


In [15]:

# BONUS B2 - Restore all original helper functions

import json
import ast
import inspect

with open("Assignment_2.ipynb", encoding="utf-8") as f:
    notebook = json.load(f)

all_functions = []

for cell in notebook["cells"]:
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    for node in tree.body:
        if isinstance(
            node, (ast.FunctionDef, ast.AsyncFunctionDef)
        ):
            all_functions.append(node)

module = ast.fix_missing_locations(
    ast.Module(
        body=all_functions,
        type_ignores=[]
    )
)

exec(
    compile(module, "<original_functions>", "exec"),
    globals()
)

print("Original functions restored:", len(all_functions))

required = [
    "greedy_initial_solution",
    "assign_clients_to_nearest_open_facility",
    "initialize_ga_population",
    "genetic_algorithm_solve_cflp",
    "genetic_algorithm_two_point_crossover"
]

for name in required:
    print(
        name,
        "READY" if callable(globals().get(name))
        else "NOT LOADED"
    )


Original functions restored: 31
greedy_initial_solution READY
assign_clients_to_nearest_open_facility READY
initialize_ga_population READY
genetic_algorithm_solve_cflp READY
genetic_algorithm_two_point_crossover READY


In [17]:

# BONUS B3 - Inspect GA structure for Memetic Search

import ast
import inspect

print("===== GA MAIN FUNCTION =====")
print(inspect.signature(genetic_algorithm_solve_cflp))

print("\n===== GA OPERATORS =====")

operators = [
    "evaluate_chromosome",
    "repair_chromosome",
    "mutate_chromosome",
    "tournament_selection",
    "uniform_crossover"
]

for name in operators:
    func = globals().get(name)
    if callable(func):
        print(name, inspect.signature(func))
    else:
        print(name, "NOT LOADED")

print("\n===== GA LOOP STRUCTURE =====")

for node in ast.walk(original_ga_node):
    if isinstance(node, (ast.For, ast.While)):
        print("Loop:", ast.unparse(node)[:250])
        break

print("\nMemetic GA inspection completed!")


===== GA MAIN FUNCTION =====
(instance: dict, population_size: int = 30, max_generations: int = 100, crossover_rate: float = 0.85, mutation_rate: float = 0.1, tournament_size: int = 3, elitism: int = 2, max_no_improve: int = 30) -> Tuple[Tuple[numpy.ndarray, numpy.ndarray], Tuple[List[float], List[int]]]

===== GA OPERATORS =====
evaluate_chromosome (chromosome: numpy.ndarray, fixed_costs: numpy.ndarray, service_costs: numpy.ndarray, facility_capacities: numpy.ndarray, client_demands: numpy.ndarray) -> float
repair_chromosome (facility_opened: numpy.ndarray, service_costs: numpy.ndarray, facility_capacities: numpy.ndarray, client_demands: numpy.ndarray) -> Tuple[numpy.ndarray, numpy.ndarray]
mutate_chromosome (chromosome: numpy.ndarray, mutation_rate: float) -> numpy.ndarray
tournament_selection (population: List[numpy.ndarray], fitness: List[float], tournament_size: int = 3) -> numpy.ndarray
uniform_crossover (parent1: numpy.ndarray, parent2: numpy.ndarray) -> Tuple[numpy.ndarray, num

In [18]:

# BONUS B3 - Memetic GA Local Search

import numpy as np

def memetic_local_search(
    chromosome,
    fixed_costs,
    service_costs,
    facility_capacities,
    client_demands,
    max_neighbors=10
):
    best = np.asarray(chromosome).copy()

    def evaluate(candidate):
        cost, opened, assignments = evaluate_chromosome(
            candidate,
            fixed_costs,
            service_costs,
            facility_capacities,
            client_demands
        )
        return float(cost), opened, assignments

    best_cost, _, _ = evaluate(best)

    positions = np.random.permutation(len(best))

    for position in positions[:max_neighbors]:
        candidate = best.copy()

        # Flip one facility decision
        candidate[position] = 1 - candidate[position]

        candidate_cost, opened, assignments = evaluate(
            candidate
        )

        if not np.isfinite(candidate_cost):
            continue

        if not is_feasible(
            opened,
            assignments,
            facility_capacities,
            client_demands
        ):
            continue

        if candidate_cost < best_cost:
            best = candidate
            best_cost = candidate_cost

    return best


print("Memetic Local Search function loaded!")


Memetic Local Search function loaded!


In [20]:

# BONUS B3 - Inspect actual evaluation outputs

import inspect

print("===== EVALUATION FUNCTION =====")
print(inspect.signature(evaluate_chromosome))

print("\n===== REPAIR FUNCTION =====")
print(inspect.signature(repair_chromosome))

print("\n===== FITNESS FUNCTION =====")
print(inspect.signature(compute_total_cost))

print("\n===== RETURN VALUES =====")

import json
import ast

with open("Assignment_2.ipynb", encoding="utf-8") as f:
    notebook = json.load(f)

found = False

for cell in notebook["cells"]:
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    for node in ast.walk(tree):
        if (
            isinstance(node, ast.FunctionDef)
            and node.name == "evaluate_chromosome"
        ):
            print("Function found!")

            for child in ast.walk(node):
                if isinstance(child, ast.Return):
                    print(
                        "Return:",
                        ast.unparse(child.value)
                        if child.value is not None
                        else "None"
                    )

            found = True

if not found:
    print("Function not found in original notebook.")

print("\nInspection completed!")


===== EVALUATION FUNCTION =====
(chromosome: numpy.ndarray, fixed_costs: numpy.ndarray, service_costs: numpy.ndarray, facility_capacities: numpy.ndarray, client_demands: numpy.ndarray) -> float

===== REPAIR FUNCTION =====
(facility_opened: numpy.ndarray, service_costs: numpy.ndarray, facility_capacities: numpy.ndarray, client_demands: numpy.ndarray) -> Tuple[numpy.ndarray, numpy.ndarray]

===== FITNESS FUNCTION =====
(fixed_costs: numpy.ndarray, service_costs: numpy.ndarray, facility_opened: numpy.ndarray, client_assignments: numpy.ndarray) -> float

===== RETURN VALUES =====
Function found!
Return: compute_total_cost(fixed_costs, service_costs, chromosome, assignments)
Return: float('inf')

Inspection completed!


In [21]:

# BONUS B3 - Corrected Memetic Local Search

import numpy as np

def memetic_local_search(
    chromosome,
    fixed_costs,
    service_costs,
    facility_capacities,
    client_demands,
    max_neighbors=10
):
    best = np.asarray(chromosome).copy()

    def evaluate(candidate):
        return float(evaluate_chromosome(
            candidate,
            fixed_costs,
            service_costs,
            facility_capacities,
            client_demands
        ))

    best_cost = evaluate(best)

    positions = np.random.permutation(len(best))

    for position in positions[:max_neighbors]:
        candidate = best.copy()

        # Flip one facility decision
        candidate[position] = 1 - candidate[position]

        candidate_cost = evaluate(candidate)

        # Accept only feasible improvements
        if (
            np.isfinite(candidate_cost)
            and candidate_cost < best_cost
        ):
            best = candidate
            best_cost = candidate_cost

    return best


print("Corrected Memetic Local Search loaded!")


Corrected Memetic Local Search loaded!


In [22]:

# BONUS B3 - Test Memetic Local Search

import json
import numpy as np

with open(pilot_files[0], encoding="utf-8") as f:
    test_instance = json.load(f)

fixed_costs = np.asarray(
    test_instance["fixed_costs"]
)
service_costs = np.asarray(
    test_instance["service_costs"]
)
capacities = np.asarray(
    test_instance["facility_capacities"]
)
demands = np.asarray(
    test_instance["client_demands"]
)

# Start with all facilities open
chromosome = np.ones(
    len(fixed_costs), dtype=int
)

original_cost = float(evaluate_chromosome(
    chromosome,
    fixed_costs,
    service_costs,
    capacities,
    demands
))

improved = memetic_local_search(
    chromosome,
    fixed_costs,
    service_costs,
    capacities,
    demands,
    max_neighbors=10
)

improved_cost = float(evaluate_chromosome(
    improved,
    fixed_costs,
    service_costs,
    capacities,
    demands
))

print("===== MEMETIC LOCAL SEARCH TEST =====")
print("Original cost:", round(original_cost, 3))
print("Improved cost:", round(improved_cost, 3))
print("Original chromosome:", chromosome)
print("Improved chromosome:", improved)

assert np.isfinite(improved_cost)
assert improved_cost <= original_cost + 1e-8

print("Local Search cost test passed!")


===== MEMETIC LOCAL SEARCH TEST =====
Original cost: 18815.582
Improved cost: 18153.729
Original chromosome: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
Improved chromosome: [1 1 1 1 0 1 0 1 1 1 1 1 1 0 1 1]
Local Search cost test passed!


In [24]:

# BONUS B3 - Inspect repair and assignment functions

import inspect

names = [
    "repair_chromosome",
    "assign_clients_to_nearest_open_facility",
    "evaluate_chromosome",
    "is_feasible"
]

for name in names:
    func = globals().get(name)

    print("\nFunction:", name)

    if callable(func):
        print("Signature:", inspect.signature(func))
    else:
        print("NOT LOADED")

print("\nInspection completed!")



Function: repair_chromosome
Signature: (facility_opened: numpy.ndarray, service_costs: numpy.ndarray, facility_capacities: numpy.ndarray, client_demands: numpy.ndarray) -> Tuple[numpy.ndarray, numpy.ndarray]

Function: assign_clients_to_nearest_open_facility
Signature: (service_costs: numpy.ndarray, facility_opened: numpy.ndarray, facility_capacities: numpy.ndarray, client_demands: numpy.ndarray) -> numpy.ndarray

Function: evaluate_chromosome
Signature: (chromosome: numpy.ndarray, fixed_costs: numpy.ndarray, service_costs: numpy.ndarray, facility_capacities: numpy.ndarray, client_demands: numpy.ndarray) -> float

Function: is_feasible
Signature: (facility_opened: numpy.ndarray, client_assignments: numpy.ndarray, facility_capacities: numpy.ndarray, client_demands: numpy.ndarray) -> bool

Inspection completed!


In [25]:

# BONUS B3 - Verify Local Search Feasibility

import numpy as np

def verify_memetic_solution(chromosome):
    opened = np.asarray(chromosome).copy()

    assignments = assign_clients_to_nearest_open_facility(
        service_costs,
        opened,
        capacities,
        demands
    )

    return is_feasible(
        opened,
        assignments,
        capacities,
        demands
    )

original_feasible = verify_memetic_solution(chromosome)
improved_feasible = verify_memetic_solution(improved)

print("===== MEMETIC FEASIBILITY CHECK =====")
print("Original feasible:", original_feasible)
print("Improved feasible:", improved_feasible)


===== MEMETIC FEASIBILITY CHECK =====
Original feasible: True
Improved feasible: True


In [26]:

# BONUS B3 - Locate GA offspring generation

import ast
import json

with open("Assignment_2.ipynb", encoding="utf-8") as f:
    notebook = json.load(f)

found = False

for cell in notebook["cells"]:
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    for node in tree.body:
        if (
            isinstance(node, ast.FunctionDef)
            and node.name == "genetic_algorithm_solve_cflp"
        ):
            found = True

            for line_number, line in enumerate(
                ast.unparse(node).splitlines(), 1
            ):
                if any(word in line.lower() for word in [
                    "offspring",
                    "new_population",
                    "crossover",
                    "mutation",
                    "fitness"
                ]):
                    print(f"{line_number}: {line}")

if not found:
    print("Original GA function not found.")

print("\nGA insertion-point inspection completed!")


1: def genetic_algorithm_solve_cflp(instance: dict, population_size: int=30, max_generations: int=100, crossover_rate: float=0.85, mutation_rate: float=0.1, tournament_size: int=3, elitism: int=2, max_no_improve: int=30) -> Tuple[Tuple[np.ndarray, np.ndarray], Tuple[List[float], List[int]]]:
6:     selection, uniform crossover, bit-flip mutation, feasibility repair, and elitist
19:     crossover_rate : float
20:         Probability of applying crossover to a pair of parents (default: 0.85)
21:     mutation_rate : float
22:         Per-gene flip probability during mutation (default: 0.10)
44:     fitness = [evaluate_chromosome(ind, fixed_costs, service_costs, facility_capacities, client_demands) for ind in population]
45:     best_idx = int(np.argmin(fitness))
46:     best_fitness = fitness[best_idx]
48:     trajectory = [best_fitness]
53:         sorted_indices = np.argsort(fitness)
55:         elite_fitness = [fitness[i] for i in sorted_indices[:elitism]]
56:         offspring = []
57

In [27]:

# BONUS B3 - Inspect exact offspring insertion point

import ast
import json

with open("Assignment_2.ipynb", encoding="utf-8") as f:
    notebook = json.load(f)

found = False

for cell in notebook["cells"]:
    if cell["cell_type"] != "code":
        continue

    try:
        tree = ast.parse("".join(cell["source"]))
    except SyntaxError:
        continue

    for node in tree.body:
        if (
            isinstance(node, ast.FunctionDef)
            and node.name == "genetic_algorithm_solve_cflp"
        ):
            lines = ast.unparse(node).splitlines()

            for i, line in enumerate(lines):
                if "offspring.append" in line:
                    start = max(0, i - 18)
                    end = min(len(lines), i + 8)

                    print("===== OFFSPRING CODE =====")
                    for j in range(start, end):
                        print(f"{j + 1}: {lines[j]}")

                    found = True
                    break

if not found:
    print("Offspring insertion point not found.")

print("\nInsertion inspection completed!")


Offspring insertion point not found.

Insertion inspection completed!


In [28]:

# BONUS B3 - Find exact offspring generation code

import ast
import json

with open("Assignment_2.ipynb", encoding="utf-8") as f:
    notebook = json.load(f)

found = False

for cell in notebook["cells"]:
    if cell["cell_type"] != "code":
        continue

    try:
        tree = ast.parse("".join(cell["source"]))
    except SyntaxError:
        continue

    for node in tree.body:
        if (
            isinstance(node, ast.FunctionDef)
            and node.name == "genetic_algorithm_solve_cflp"
        ):
            lines = ast.unparse(node).splitlines()

            for i, line in enumerate(lines):
                if "offspring.extend" in line:
                    start = max(0, i - 12)
                    end = min(len(lines), i + 5)

                    print("===== OFFSPRING GENERATION =====")
                    for j in range(start, end):
                        print(f"{j + 1}: {lines[j]}")

                    found = True
                    break

if not found:
    print("Offspring generation not found.")

print("\nInspection completed!")


===== OFFSPRING GENERATION =====
59:             p1 = tournament_selection(population, fitness, tournament_size)
60:             p2 = tournament_selection(population, fitness, tournament_size)
61:             if np.random.random() < crossover_rate:
62:                 c1, c2 = uniform_crossover(p1, p2)
63:             else:
64:                 c1, c2 = (p1.copy(), p2.copy())
65:             c1 = mutate_chromosome(c1, mutation_rate)
66:             c2 = mutate_chromosome(c2, mutation_rate)
67:             c1, _ = repair_chromosome(c1, service_costs, facility_capacities, client_demands)
68:             c2, _ = repair_chromosome(c2, service_costs, facility_capacities, client_demands)
69:             f1 = evaluate_chromosome(c1, fixed_costs, service_costs, facility_capacities, client_demands)
70:             f2 = evaluate_chromosome(c2, fixed_costs, service_costs, facility_capacities, client_demands)
71:             offspring.extend([c1, c2])
72:             offspring_fitness.extend([f1, f

In [29]:
 
# BONUS B3 - Build Memetic Genetic Algorithm

import ast
import json
import copy
import numpy as np

with open("Assignment_2.ipynb", encoding="utf-8") as f:
    notebook = json.load(f)

original_node = None

for cell in notebook["cells"]:
    if cell["cell_type"] != "code":
        continue

    try:
        tree = ast.parse("".join(cell["source"]))
    except SyntaxError:
        continue

    for node in tree.body:
        if (
            isinstance(node, ast.FunctionDef)
            and node.name == "genetic_algorithm_solve_cflp"
        ):
            original_node = copy.deepcopy(node)
            break

    if original_node is not None:
        break

assert original_node is not None, "Original GA not found"

original_node.name = "genetic_algorithm_memetic"

class MemeticTransformer(ast.NodeTransformer):

    def __init__(self):
        self.replacements = 0

    def visit_Expr(self, node):
        self.generic_visit(node)

        if not isinstance(node.value, ast.Call):
            return node

        call = node.value

        if not isinstance(call.func, ast.Attribute):
            return node

        if not (
            isinstance(call.func.value, ast.Name)
            and call.func.value.id == "offspring"
            and call.func.attr == "extend"
        ):
            return node

        injection = ast.parse("""
if np.random.random() < 0.15:
    c1 = memetic_local_search(
        c1, fixed_costs, service_costs,
        facility_capacities, client_demands,
        max_neighbors=4
    )

if np.random.random() < 0.15:
    c2 = memetic_local_search(
        c2, fixed_costs, service_costs,
        facility_capacities, client_demands,
        max_neighbors=4
    )
""").body

        self.replacements += 1
        return injection + [node]


transformer = MemeticTransformer()
new_node = transformer.visit(original_node)

assert transformer.replacements == 1, (
    "Expected exactly one offspring insertion point"
)

module = ast.fix_missing_locations(
    ast.Module(body=[new_node], type_ignores=[])
)

exec(
    compile(module, "<memetic_ga>", "exec"),
    globals()
)

print("Memetic GA loaded successfully!")
print("Local Search insertion points:",
      transformer.replacements)
print("Original GA remains unchanged.")


Memetic GA loaded successfully!
Local Search insertion points: 1
Original GA remains unchanged.


In [32]:

# BONUS B3 - Find mean() error source

import ast
import json

with open("Assignment_2.ipynb", encoding="utf-8") as f:
    notebook = json.load(f)

print("===== SEARCHING FOR .mean() CALLS =====")

found = 0

for cell_number, cell in enumerate(notebook["cells"]):
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    for line_number, line in enumerate(
        source.splitlines(), 1
    ):
        if ".mean(" in line:
            print(
                f"Cell {cell_number}, "
                f"Line {line_number}: "
                f"{line.strip()}"
            )
            found += 1

print("\nTotal occurrences:", found)
print("Search completed!")


===== SEARCHING FOR .mean() CALLS =====
Cell 5, Line 52: avg_cost_per_facility = service_costs.mean(axis=0)
Cell 5, Line 53: global_avg_cost = avg_cost_per_facility.mean()
Cell 5, Line 307: min_capacity = 2 * client_demands.mean()
Cell 5, Line 365: "avg_fixed_cost": round(fixed_costs.mean(), 2),
Cell 5, Line 366: "avg_service_cost": round(service_costs.mean(), 2),
Cell 5, Line 367: "avg_demand": round(client_demands.mean(), 2),
Cell 5, Line 368: "avg_capacity": round(facility_capacities.mean(), 2)
Cell 8, Line 173: avg_service_cost_per_facility = service_costs.mean(axis=0)
Cell 24, Line 335: print(f"\nTabu Search  — avg gap: {valid['ts_gap_percent'].mean():.2f}%, "
Cell 24, Line 336: f"avg speedup: {valid['ts_speedup_factor'].mean():.2f}x")
Cell 24, Line 337: print(f"Genetic Algo — avg gap: {valid['ga_gap_percent'].mean():.2f}%, "
Cell 24, Line 338: f"avg speedup: {valid['ga_speedup_factor'].mean():.2f}x")
Cell 26, Line 94: means = [valid[valid['size'] == s][col].mean() for s in sizes_

In [33]:

# BONUS B3 - Inspect Greedy Initialization

import inspect
import dis

print("===== GREEDY FUNCTION =====")
print(inspect.signature(greedy_initial_solution))

print("\n===== BYTECODE INSPECTION =====")
for instruction in dis.get_instructions(greedy_initial_solution):
    if instruction.argval in ("mean", "greedy_initial_solution"):
        print(instruction.opname, instruction.argval)

print("\n===== INPUT TYPE CHECK =====")
print("Instance type:", type(test_instance))
print("Service costs type:", type(test_instance["service_costs"]))
print("Fixed costs type:", type(test_instance["fixed_costs"]))

print("\nInspection completed!")


===== GREEDY FUNCTION =====
(num_clients: int, num_facilities: int, service_costs: numpy.ndarray, facility_capacities: numpy.ndarray, client_demands: numpy.ndarray) -> Tuple[numpy.ndarray, numpy.ndarray]

===== BYTECODE INSPECTION =====
LOAD_ATTR mean

===== INPUT TYPE CHECK =====
Instance type: <class 'dict'>
Service costs type: <class 'list'>
Fixed costs type: <class 'list'>

Inspection completed!


In [34]:

# BONUS B3 - Prepare NumPy Instance

import numpy as np
import copy

memetic_instance = copy.deepcopy(test_instance)

array_keys = [
    "fixed_costs",
    "service_costs",
    "facility_capacities",
    "client_demands"
]

for key in array_keys:
    memetic_instance[key] = np.asarray(
        memetic_instance[key],
        dtype=float
    )

print("===== NUMPY DATA PREPARATION =====")

for key in array_keys:
    print(key, type(memetic_instance[key]).__name__)

print("Memetic instance prepared successfully!")


===== NUMPY DATA PREPARATION =====
fixed_costs ndarray
service_costs ndarray
facility_capacities ndarray
client_demands ndarray
Memetic instance prepared successfully!


In [35]:

# BONUS B3 - Memetic GA Test with NumPy Data

import time
import random
import numpy as np

random.seed(BONUS_SEED)
np.random.seed(BONUS_SEED)

start = time.perf_counter()

memetic_result = genetic_algorithm_memetic(
    memetic_instance,
    population_size=30,
    max_generations=100,
    crossover_rate=0.85,
    mutation_rate=0.10
)

runtime = time.perf_counter() - start

print("===== MEMETIC GA TEST =====")
print("Result type:", type(memetic_result))
print("Runtime:", round(runtime, 3), "seconds")
print("Memetic GA test completed!")


Starting Genetic Algorithm for CFLP (pop=30, max_gen=100)...
Genetic Algorithm completed after 34 generations.
Best cost: 18007.53
===== MEMETIC GA TEST =====
Result type: <class 'tuple'>
Runtime: 0.777 seconds
Memetic GA test completed!


In [36]:

# BONUS B3 - Inspect Memetic GA Result

import numpy as np

print("===== MEMETIC GA RESULT =====")
print("Number of returned items:", len(memetic_result))

for i, item in enumerate(memetic_result):
    print("\nItem", i)
    print("Type:", type(item).__name__)

    if isinstance(item, (list, tuple)):
        print("Length:", len(item))
        print("Element types:",
              [type(x).__name__ for x in item[:3]])

    elif isinstance(item, np.ndarray):
        print("Shape:", item.shape)

    else:
        print("Value:", item)

print("\nResult inspection completed!")


===== MEMETIC GA RESULT =====
Number of returned items: 2

Item 0
Type: tuple
Length: 2
Element types: ['ndarray', 'ndarray']

Item 1
Type: tuple
Length: 2
Element types: ['list', 'list']

Result inspection completed!


In [37]:

# BONUS B3 - Verify Final Memetic GA Solution

import numpy as np

# Extract final solution and trajectory
solution, trajectory_data = memetic_result
opened, assignments = solution

# Check feasibility
feasible = is_feasible(
    opened,
    assignments,
    memetic_instance["facility_capacities"],
    memetic_instance["client_demands"]
)

# Calculate actual cost
final_cost = compute_total_cost(
    memetic_instance["fixed_costs"],
    memetic_instance["service_costs"],
    opened,
    assignments
)

print("===== MEMETIC GA FINAL CHECK =====")
print("Final cost:", round(float(final_cost), 3))
print("Feasible:", feasible)
print("Runtime:", round(runtime, 3), "seconds")
print("Opened facilities:", int(np.sum(opened)))

assert feasible, "Memetic GA solution is infeasible!"
assert np.isfinite(final_cost), "Invalid final cost!"

print("Memetic GA final verification passed!")


===== MEMETIC GA FINAL CHECK =====
Final cost: 18007.527
Feasible: True
Runtime: 0.777 seconds
Opened facilities: 12
Memetic GA final verification passed!


In [38]:

# BONUS B3 - Original GA vs Memetic GA
# Single Instance Comparison

import time
import random
import numpy as np

random.seed(BONUS_SEED)
np.random.seed(BONUS_SEED)

start = time.perf_counter()

original_result = genetic_algorithm_solve_cflp(
    memetic_instance,
    population_size=30,
    max_generations=100,
    crossover_rate=0.85,
    mutation_rate=0.10
)

original_runtime = time.perf_counter() - start

original_solution, original_trajectory = original_result
original_opened, original_assignments = original_solution

original_cost = float(compute_total_cost(
    memetic_instance["fixed_costs"],
    memetic_instance["service_costs"],
    original_opened,
    original_assignments
))

original_feasible = is_feasible(
    original_opened,
    original_assignments,
    memetic_instance["facility_capacities"],
    memetic_instance["client_demands"]
)

memetic_solution, memetic_trajectory = memetic_result
memetic_opened, memetic_assignments = memetic_solution

memetic_cost = float(compute_total_cost(
    memetic_instance["fixed_costs"],
    memetic_instance["service_costs"],
    memetic_opened,
    memetic_assignments
))

improvement = (
    (original_cost - memetic_cost) / original_cost
) * 100

print("===== BONUS B3 COMPARISON =====")
print("Original GA cost:", round(original_cost, 3))
print("Memetic GA cost:", round(memetic_cost, 3))
print("Cost improvement (%):", round(improvement, 3))
print("Original GA runtime:", round(original_runtime, 3))
print("Memetic GA runtime:", round(runtime, 3))
print("Original feasible:", original_feasible)
print("Memetic feasible:", is_feasible(
    memetic_opened,
    memetic_assignments,
    memetic_instance["facility_capacities"],
    memetic_instance["client_demands"]
))

print("Comparison completed!")


Starting Genetic Algorithm for CFLP (pop=30, max_gen=100)...
Genetic Algorithm completed after 37 generations.
Best cost: 18007.53
===== BONUS B3 COMPARISON =====
Original GA cost: 18007.527
Memetic GA cost: 18007.527
Cost improvement (%): 0.0
Original GA runtime: 0.709
Memetic GA runtime: 0.777
Original feasible: True
Memetic feasible: True
Comparison completed!


In [39]:

# BONUS B3 - Six Instance Pilot Experiment

import os
import json
import time
import random
import numpy as np
import pandas as pd

results = []

for idx, file_path in enumerate(pilot_files):

    with open(file_path, encoding="utf-8") as f:
        instance = json.load(f)

    for key in [
        "fixed_costs",
        "service_costs",
        "facility_capacities",
        "client_demands"
    ]:
        instance[key] = np.asarray(instance[key], dtype=float)

    row = {"Instance": os.path.basename(file_path)}

    for method, solver in [
        ("Original", genetic_algorithm_solve_cflp),
        ("Memetic", genetic_algorithm_memetic)
    ]:

        random.seed(BONUS_SEED)
        np.random.seed(BONUS_SEED)

        start = time.perf_counter()

        solution, trajectory = solver(
            instance,
            population_size=30,
            max_generations=100,
            crossover_rate=0.85,
            mutation_rate=0.10
        )

        elapsed = time.perf_counter() - start

        opened, assignments = solution

        cost = compute_total_cost(
            instance["fixed_costs"],
            instance["service_costs"],
            opened,
            assignments
        )

        feasible = is_feasible(
            opened,
            assignments,
            instance["facility_capacities"],
            instance["client_demands"]
        )

        row[f"{method}_Cost"] = float(cost)
        row[f"{method}_Time"] = elapsed
        row[f"{method}_Feasible"] = bool(feasible)

    row["Improvement_%"] = (
        (row["Original_Cost"] - row["Memetic_Cost"])
        / row["Original_Cost"] * 100
    )

    results.append(row)
    print(f"Completed {idx+1}/6: {row['Instance']}")

memetic_df = pd.DataFrame(results)

memetic_df.to_csv(
    "bonus_B_memetic_pilot_447540190.csv",
    index=False
)

print("\n===== BONUS B3 RESULTS =====")
print(memetic_df.to_string(index=False))

print("\nMean cost improvement (%):",
      round(memetic_df["Improvement_%"].mean(), 4))

print("Mean original runtime:",
      round(memetic_df["Original_Time"].mean(), 3))

print("Mean memetic runtime:",
      round(memetic_df["Memetic_Time"].mean(), 3))

print("\nBonus B3 pilot results saved!")


Starting Genetic Algorithm for CFLP (pop=30, max_gen=100)...
Genetic Algorithm completed after 37 generations.
Best cost: 18007.53
Starting Genetic Algorithm for CFLP (pop=30, max_gen=100)...
Genetic Algorithm completed after 34 generations.
Best cost: 18007.53
Completed 1/6: 001_cflp_small_easy_fac16_cli75.json
Starting Genetic Algorithm for CFLP (pop=30, max_gen=100)...
Genetic Algorithm completed after 50 generations.
Best cost: 11178.58
Starting Genetic Algorithm for CFLP (pop=30, max_gen=100)...
Genetic Algorithm completed after 39 generations.
Best cost: 11158.91
Completed 2/6: 002_cflp_small_easy_fac20_cli65.json
Starting Genetic Algorithm for CFLP (pop=30, max_gen=100)...
Genetic Algorithm completed after 42 generations.
Best cost: 11950.82
Starting Genetic Algorithm for CFLP (pop=30, max_gen=100)...
Genetic Algorithm completed after 38 generations.
Best cost: 11950.82
Completed 3/6: 003_cflp_small_easy_fac18_cli62.json
Starting Genetic Algorithm for CFLP (pop=30, max_gen=100).

In [40]:

# BONUS B3 - Final Pilot Summary

import pandas as pd

df = pd.read_csv("bonus_B_memetic_pilot_447540190.csv")

print("===== BONUS B3 FINAL SUMMARY =====")
print(df.to_string(index=False))

print("\n===== OVERALL RESULTS =====")
print("Instances tested:", len(df))
print("Mean cost improvement (%):",
      round(df["Improvement_%"].mean(), 4))

print("Improved instances:",
      int((df["Improvement_%"] > 0).sum()))

print("Equal-cost instances:",
      int((df["Improvement_%"].abs() < 1e-8).sum()))

print("Worse instances:",
      int((df["Improvement_%"] < -1e-8).sum()))

print("All original solutions feasible:",
      df["Original_Feasible"].all())

print("All memetic solutions feasible:",
      df["Memetic_Feasible"].all())

print("Mean original runtime:",
      round(df["Original_Time"].mean(), 3))

print("Mean memetic runtime:",
      round(df["Memetic_Time"].mean(), 3))

print("\nBonus B3 summary completed!")


===== BONUS B3 FINAL SUMMARY =====
                             Instance  Original_Cost  Original_Time  Original_Feasible  Memetic_Cost  Memetic_Time  Memetic_Feasible  Improvement_%
 001_cflp_small_easy_fac16_cli75.json   18007.527403       0.658874               True  18007.527403      0.780465              True       0.000000
 002_cflp_small_easy_fac20_cli65.json   11178.576191       0.726148               True  11158.912569      0.819245              True       0.175905
 003_cflp_small_easy_fac18_cli62.json   11950.817555       0.659727               True  11950.817555      0.789401              True       0.000000
049_cflp_medium_easy_fac22_cli65.json   10655.491712       0.772837               True  10655.491712      1.102905              True       0.000000
050_cflp_medium_easy_fac19_cli72.json   12146.898418       0.811747               True  12146.898418      0.892258              True       0.000000
051_cflp_medium_easy_fac24_cli74.json   13129.000069       1.042855          

In [47]:

# BONUS B4 - Diversity Preservation (Corrected)

import ast
import copy
import json
import numpy as np

def preserve_diversity(
    offspring, offspring_fitness,
    fixed_costs, service_costs,
    facility_capacities, client_demands
):
    seen = set()
    new_pop = []
    new_fit = []

    for chrom, fit in zip(offspring, offspring_fitness):
        chrom = np.asarray(chrom, dtype=int).copy()
        key = tuple(chrom)

        if key in seen:
            for _ in range(10):
                candidate = mutate_chromosome(
                    chrom, mutation_rate=0.20
                )

                candidate, _ = repair_chromosome(
                    candidate, service_costs,
                    facility_capacities, client_demands
                )

                candidate = np.asarray(
                    candidate, dtype=int
                )

                new_key = tuple(candidate)

                if new_key not in seen:
                    chrom = candidate
                    key = new_key
                    fit = evaluate_chromosome(
                        chrom, fixed_costs, service_costs,
                        facility_capacities, client_demands
                    )
                    break

        seen.add(key)
        new_pop.append(chrom)
        new_fit.append(float(fit))

    return new_pop, new_fit


# Read only the original GA function's cell
with open("Assignment_2.ipynb", encoding="utf-8") as f:
    original_notebook = json.load(f)

ga_node = None

for cell in original_notebook["cells"]:
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    if "def genetic_algorithm_solve_cflp(" not in source:
        continue

    tree = ast.parse(source)

    for node in tree.body:
        if (
            isinstance(node, ast.FunctionDef)
            and node.name == "genetic_algorithm_solve_cflp"
        ):
            ga_node = copy.deepcopy(node)
            break

    if ga_node is not None:
        break

assert ga_node is not None, "Original GA not found"

ga_node.name = "genetic_algorithm_diversity"


class DiversityTransformer(ast.NodeTransformer):
    def __init__(self):
        self.count = 0

    def visit_Assign(self, node):
        if (
            len(node.targets) == 1
            and isinstance(node.targets[0], ast.Name)
            and node.targets[0].id == "population"
            and isinstance(node.value, ast.BinOp)
            and isinstance(node.value.op, ast.Add)
            and isinstance(node.value.left, ast.Name)
            and node.value.left.id == "elites"
        ):
            self.count += 1

            extra = ast.parse("""
offspring, offspring_fitness = preserve_diversity(
    offspring, offspring_fitness,
    fixed_costs, service_costs,
    facility_capacities, client_demands
)
""").body

            return extra + [node]

        return node


transformer = DiversityTransformer()
ga_node = transformer.visit(ga_node)

assert transformer.count == 1, (
    "Diversity insertion point not found"
)

module = ast.fix_missing_locations(
    ast.Module(body=[ga_node], type_ignores=[])
)

exec(
    compile(module, "<diversity_ga>", "exec"),
    globals()
)

print("BONUS B4 - Diversity GA loaded successfully!")
print("Diversity insertion points:", transformer.count)
print("Original GA remains unchanged.")


BONUS B4 - Diversity GA loaded successfully!
Diversity insertion points: 1
Original GA remains unchanged.


In [48]:

# BONUS B4 - Single Instance Test

import time
import random
import numpy as np

random.seed(BONUS_SEED)
np.random.seed(BONUS_SEED)

start = time.perf_counter()

diversity_result = genetic_algorithm_diversity(
    memetic_instance,
    population_size=30,
    max_generations=100,
    crossover_rate=0.85,
    mutation_rate=0.10
)

diversity_runtime = time.perf_counter() - start

diversity_solution, diversity_trajectory = diversity_result
diversity_opened, diversity_assignments = diversity_solution

diversity_feasible = is_feasible(
    diversity_opened,
    diversity_assignments,
    memetic_instance["facility_capacities"],
    memetic_instance["client_demands"]
)

diversity_cost = compute_total_cost(
    memetic_instance["fixed_costs"],
    memetic_instance["service_costs"],
    diversity_opened,
    diversity_assignments
)

print("\n===== BONUS B4 TEST =====")
print("Cost:", round(float(diversity_cost), 3))
print("Runtime:", round(diversity_runtime, 3))
print("Feasible:", diversity_feasible)

assert diversity_feasible, "Diversity GA produced an infeasible solution"
assert np.isfinite(diversity_cost), "Invalid cost"

print("BONUS B4 SINGLE INSTANCE TEST PASSED!")


Starting Genetic Algorithm for CFLP (pop=30, max_gen=100)...
Genetic Algorithm completed after 55 generations.
Best cost: 18007.53

===== BONUS B4 TEST =====
Cost: 18007.527
Runtime: 1.364
Feasible: True
BONUS B4 SINGLE INSTANCE TEST PASSED!


In [49]:

# BONUS B4 - Six Instance Pilot Comparison

import os
import json
import time
import random
import numpy as np
import pandas as pd

b4_results = []

for idx, file_path in enumerate(pilot_files, start=1):

    with open(file_path, encoding="utf-8") as f:
        instance = json.load(f)

    for key in [
        "fixed_costs",
        "service_costs",
        "facility_capacities",
        "client_demands"
    ]:
        instance[key] = np.asarray(instance[key], dtype=float)

    row = {"Instance": os.path.basename(file_path)}

    for label, solver in [
        ("Original", genetic_algorithm_solve_cflp),
        ("Diversity", genetic_algorithm_diversity)
    ]:

        random.seed(BONUS_SEED)
        np.random.seed(BONUS_SEED)

        start = time.perf_counter()

        (opened, assignments), trajectory = solver(
            instance,
            population_size=30,
            max_generations=100,
            crossover_rate=0.85,
            mutation_rate=0.10
        )

        elapsed = time.perf_counter() - start

        cost = float(compute_total_cost(
            instance["fixed_costs"],
            instance["service_costs"],
            opened,
            assignments
        ))

        feasible = bool(is_feasible(
            opened,
            assignments,
            instance["facility_capacities"],
            instance["client_demands"]
        ))

        assert feasible, f"{label} infeasible: {file_path}"
        assert np.isfinite(cost), "Invalid cost"

        row[f"{label}_Cost"] = cost
        row[f"{label}_Time"] = elapsed
        row[f"{label}_Feasible"] = feasible

    row["Improvement_%"] = (
        (row["Original_Cost"] - row["Diversity_Cost"])
        / row["Original_Cost"] * 100
    )

    b4_results.append(row)

    print(f"Completed {idx}/{len(pilot_files)}")

b4_df = pd.DataFrame(b4_results)

b4_df.to_csv(
    "bonus_B_diversity_pilot_447540190.csv",
    index=False
)

print("\n===== BONUS B4 FINAL RESULTS =====")
print(b4_df.to_string(index=False))

print("\nMean improvement (%):",
      round(b4_df["Improvement_%"].mean(), 4))

print("Improved instances:",
      int((b4_df["Improvement_%"] > 1e-8).sum()))

print("All feasible:",
      bool(b4_df["Diversity_Feasible"].all()))

print("\nBONUS B4 PILOT RESULTS SAVED!")


Starting Genetic Algorithm for CFLP (pop=30, max_gen=100)...
Genetic Algorithm completed after 37 generations.
Best cost: 18007.53
Starting Genetic Algorithm for CFLP (pop=30, max_gen=100)...
Genetic Algorithm completed after 55 generations.
Best cost: 18007.53
Completed 1/6
Starting Genetic Algorithm for CFLP (pop=30, max_gen=100)...
Genetic Algorithm completed after 50 generations.
Best cost: 11178.58
Starting Genetic Algorithm for CFLP (pop=30, max_gen=100)...
Genetic Algorithm completed after 47 generations.
Best cost: 11102.69
Completed 2/6
Starting Genetic Algorithm for CFLP (pop=30, max_gen=100)...
Genetic Algorithm completed after 42 generations.
Best cost: 11950.82
Starting Genetic Algorithm for CFLP (pop=30, max_gen=100)...
Genetic Algorithm completed after 50 generations.
Best cost: 11950.82
Completed 3/6
Starting Genetic Algorithm for CFLP (pop=30, max_gen=100)...
Genetic Algorithm completed after 44 generations.
Best cost: 10655.49
Starting Genetic Algorithm for CFLP (pop=

In [50]:

# BONUS B - Compare All Four Enhancements

import pandas as pd

files = {
    "B1 - Adaptive Mutation": "bonus_B_adaptive_pilot_447540190.csv",
    "B2 - Two-Point Crossover": "bonus_B_two_point_pilot_447540190.csv",
    "B3 - Memetic GA": "bonus_B_memetic_pilot_447540190.csv",
    "B4 - Diversity Preservation": "bonus_B_diversity_pilot_447540190.csv"
}

summary = []

for method, filename in files.items():
    df = pd.read_csv(filename)

    improvement_columns = [
        c for c in df.columns
        if "improvement" in c.lower()
    ]

    print("\n", method)
    print("Instances:", len(df))
    print("Columns:", list(df.columns))

    if improvement_columns:
        values = pd.to_numeric(
            df[improvement_columns[0]],
            errors="coerce"
        )
        mean_improvement = values.mean()

        print("Mean improvement (%):",
              round(mean_improvement, 4))

        summary.append({
            "Method": method,
            "Mean Improvement (%)": mean_improvement
        })
    else:
        print("Improvement column not found")

print("\n===== BONUS B COMPARISON =====")

comparison_df = pd.DataFrame(summary)
print(comparison_df.to_string(index=False))

if len(comparison_df) > 0:
    best = comparison_df.loc[
        comparison_df["Mean Improvement (%)"].idxmax()
    ]
    print("\nBest pilot method:", best["Method"])



 B1 - Adaptive Mutation
Instances: 6
Columns: ['Instance', 'Original_GA_Cost', 'Adaptive_GA_Cost', 'Improvement_%', 'Original_Time', 'Adaptive_Time']
Mean improvement (%): 0.1349

 B2 - Two-Point Crossover
Instances: 6
Columns: ['Instance', 'Original_GA_Cost', 'TwoPoint_GA_Cost', 'Improvement_%', 'Original_Time', 'TwoPoint_Time']
Mean improvement (%): -0.0844

 B3 - Memetic GA
Instances: 6
Columns: ['Instance', 'Original_Cost', 'Original_Time', 'Original_Feasible', 'Memetic_Cost', 'Memetic_Time', 'Memetic_Feasible', 'Improvement_%']
Mean improvement (%): 0.051

 B4 - Diversity Preservation
Instances: 6
Columns: ['Instance', 'Original_Cost', 'Original_Time', 'Original_Feasible', 'Diversity_Cost', 'Diversity_Time', 'Diversity_Feasible', 'Improvement_%']
Mean improvement (%): 0.1046

===== BONUS B COMPARISON =====
                     Method  Mean Improvement (%)
     B1 - Adaptive Mutation              0.134867
   B2 - Two-Point Crossover             -0.084397
            B3 - Memetic G

In [65]:

# BONUS B - FULL 144 INSTANCE EXPERIMENT
# B1: Adaptive Mutation versus Original GA
# Same feasibility repair for both algorithms

import os
import glob
import json
import time
import random
import contextlib
import io
import numpy as np
import pandas as pd

BONUS_SEED = 447540190

OUTPUT_FILE = "bonus_B_adaptive_full_repaired_447540190.csv"

INSTANCE_FOLDER = "cflp_instances_student_447540190"

instance_files = sorted(
    glob.glob(os.path.join(INSTANCE_FOLDER, "*.json"))
)

assert len(instance_files) == 144, (
    f"Expected 144 instances, found {len(instance_files)}"
)

assert callable(genetic_algorithm_solve_cflp)
assert callable(genetic_algorithm_adaptive_mutation)
assert callable(repair_ga_solution)

columns = [
    "Instance",
    "Original_GA_Cost",
    "Adaptive_GA_Cost",
    "Improvement_%",
    "Original_Time",
    "Adaptive_Time",
    "Original_Feasible",
    "Adaptive_Feasible"
]

# Resume only from the new repaired-results file
if os.path.exists(OUTPUT_FILE):
    full_df = pd.read_csv(OUTPUT_FILE)
    assert full_df["Instance"].is_unique
    assert set(columns).issubset(full_df.columns)
    print("Resuming:", len(full_df), "saved instances")
else:
    full_df = pd.DataFrame(columns=columns)
    print("Starting a new 144-instance experiment")

completed = set(full_df["Instance"].astype(str))

for idx, file_path in enumerate(instance_files, start=1):

    name = os.path.basename(file_path)

    if name in completed:
        continue

    with open(file_path, encoding="utf-8") as f:
        raw = json.load(f)

    inst = {
        "num_facilities": int(raw["num_facilities"]),
        "num_clients": int(raw["num_clients"]),
        "fixed_costs": np.asarray(
            raw["fixed_costs"], dtype=float
        ),
        "service_costs": np.asarray(
            raw["service_costs"], dtype=float
        ),
        "facility_capacities": np.asarray(
            raw["facility_capacities"], dtype=float
        ),
        "client_demands": np.asarray(
            raw["client_demands"], dtype=float
        )
    }

    measurements = []

    for solver in [
        genetic_algorithm_solve_cflp,
        genetic_algorithm_adaptive_mutation
    ]:

        random.seed(BONUS_SEED)
        np.random.seed(BONUS_SEED)

        start = time.perf_counter()

        # Run GA without excessive notebook output
        with contextlib.redirect_stdout(io.StringIO()):
            result = solver(
                inst,
                population_size=30,
                max_generations=100,
                crossover_rate=0.85,
                mutation_rate=0.10
            )

        # Repair infeasible solutions when necessary
        solution, trajectory = result

        opened, assignments = repair_ga_solution(
            solution, inst
        )

        # Include repair time
        elapsed = time.perf_counter() - start

        cost = float(compute_total_cost(
            inst["fixed_costs"],
            inst["service_costs"],
            opened,
            assignments
        ))

        feasible = bool(is_feasible(
            opened,
            assignments,
            inst["facility_capacities"],
            inst["client_demands"]
        ))

        assert feasible, (
            f"Infeasible solution: {name}"
        )

        assert np.isfinite(cost), (
            f"Invalid cost: {name}"
        )

        measurements.append(
            (cost, elapsed, feasible)
        )

    original_cost, original_time, original_ok = measurements[0]
    adaptive_cost, adaptive_time, adaptive_ok = measurements[1]

    improvement = (
        (original_cost - adaptive_cost)
        / original_cost * 100
    )

    new_row = pd.DataFrame([{
        "Instance": name,
        "Original_GA_Cost": original_cost,
        "Adaptive_GA_Cost": adaptive_cost,
        "Improvement_%": improvement,
        "Original_Time": original_time,
        "Adaptive_Time": adaptive_time,
        "Original_Feasible": original_ok,
        "Adaptive_Feasible": adaptive_ok
    }])

    full_df = pd.concat(
        [full_df, new_row],
        ignore_index=True
    )

    # Save after every completed instance
    full_df.to_csv(OUTPUT_FILE, index=False)
    completed.add(name)

    print(
        f"Completed {idx}/144: {name}",
        flush=True
    )

# Final summary
print("\n===== BONUS B FULL RESULTS =====")

print("Completed instances:", len(full_df))

print(
    "Mean improvement (%):",
    round(full_df["Improvement_%"].mean(), 4)
)

print(
    "Improved instances:",
    int((full_df["Improvement_%"] > 1e-8).sum())
)

print(
    "Equal-cost instances:",
    int((full_df["Improvement_%"].abs() <= 1e-8).sum())
)

print(
    "Worse instances:",
    int((full_df["Improvement_%"] < -1e-8).sum())
)

print(
    "Mean original runtime:",
    round(full_df["Original_Time"].mean(), 4)
)

print(
    "Mean adaptive runtime:",
    round(full_df["Adaptive_Time"].mean(), 4)
)

print(
    "All feasible:",
    bool(full_df["Original_Feasible"].all())
    and bool(full_df["Adaptive_Feasible"].all())
)

assert len(full_df) == 144
assert full_df["Instance"].is_unique

print("\nBONUS B FULL EXPERIMENT COMPLETED!")



Starting a new 144-instance experiment
Completed 1/144: 001_cflp_small_easy_fac16_cli75.json


C:\Users\madaw\AppData\Local\Temp\ipykernel_17544\2319551337.py:160: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  full_df = pd.concat(


Completed 2/144: 002_cflp_small_easy_fac20_cli65.json
Completed 3/144: 003_cflp_small_easy_fac18_cli62.json
Completed 4/144: 004_cflp_small_easy_fac17_cli77.json
Completed 5/144: 005_cflp_small_easy_fac17_cli73.json
Completed 6/144: 006_cflp_small_easy_fac20_cli62.json
Completed 7/144: 007_cflp_small_easy_fac20_cli64.json
Completed 8/144: 008_cflp_small_easy_fac19_cli77.json
Completed 9/144: 009_cflp_small_easy_fac18_cli58.json
Completed 10/144: 010_cflp_small_easy_fac20_cli73.json
Completed 11/144: 011_cflp_small_easy_fac19_cli76.json
Completed 12/144: 012_cflp_small_easy_fac18_cli73.json
Completed 13/144: 013_cflp_small_easy_fac18_cli70.json
Completed 14/144: 014_cflp_small_easy_fac15_cli74.json
Completed 15/144: 015_cflp_small_easy_fac15_cli52.json
Completed 16/144: 016_cflp_small_easy_fac19_cli50.json
Completed 17/144: 017_cflp_small_moderate_fac15_cli53.json
Completed 18/144: 018_cflp_small_moderate_fac19_cli59.json
Completed 19/144: 019_cflp_small_moderate_fac19_cli80.json
Comple

In [66]:

# BONUS B - Final Results Summary

import pandas as pd

df = pd.read_csv(
    "bonus_B_adaptive_full_repaired_447540190.csv"
)

print("===== BONUS B FINAL SUMMARY =====")
print("Total instances:", len(df))
print("Mean improvement (%):", round(df["Improvement_%"].mean(), 4))
print("Improved instances:", (df["Improvement_%"] > 1e-8).sum())
print("Equal instances:", (df["Improvement_%"].abs() <= 1e-8).sum())
print("Worse instances:", (df["Improvement_%"] < -1e-8).sum())
print("Mean Original GA time:", round(df["Original_Time"].mean(), 4))
print("Mean Adaptive GA time:", round(df["Adaptive_Time"].mean(), 4))
print("All Original feasible:", df["Original_Feasible"].all())
print("All Adaptive feasible:", df["Adaptive_Feasible"].all())


===== BONUS B FINAL SUMMARY =====
Total instances: 144
Mean improvement (%): 0.0326
Improved instances: 19
Equal instances: 119
Worse instances: 6
Mean Original GA time: 1.2042
Mean Adaptive GA time: 1.2264
All Original feasible: True
All Adaptive feasible: True


In [54]:

# BONUS B - Diagnose Instance 116 Only

import json
import glob
import random
import numpy as np

path_116 = glob.glob(
    "cflp_instances_student_447540190/116_*.json"
)[0]

with open(path_116, encoding="utf-8") as f:
    inst_116 = json.load(f)

for key in [
    "fixed_costs", "service_costs",
    "facility_capacities", "client_demands"
]:
    inst_116[key] = np.asarray(inst_116[key], dtype=float)

for name, solver in [
    ("Original GA", genetic_algorithm_solve_cflp),
    ("Adaptive GA", genetic_algorithm_adaptive_mutation)
]:
    random.seed(BONUS_SEED)
    np.random.seed(BONUS_SEED)

    result = solver(
        inst_116,
        population_size=30,
        max_generations=100,
        crossover_rate=0.85,
        mutation_rate=0.10
    )

    cost, feasible = evaluate_ga_result(result, inst_116)

    print(f"\n{name}")
    print("Cost:", round(cost, 3))
    print("Feasible:", feasible)

print("\nDiagnosis completed!")


Starting Genetic Algorithm for CFLP (pop=30, max_gen=100)...
Genetic Algorithm completed after 30 generations.
Best cost: inf

Original GA
Cost: 24193.907
Feasible: False
Starting Genetic Algorithm for CFLP (pop=30, max_gen=100)...
Genetic Algorithm completed after 30 generations.
Best cost: inf

Adaptive GA
Cost: 24193.907
Feasible: False

Diagnosis completed!


In [55]:

# BONUS B - Diagnose GA Capacity Issue

capacities = inst_116["facility_capacities"]
demands = inst_116["client_demands"]
service = inst_116["service_costs"]

all_open = np.ones(len(capacities), dtype=int)

assignments = assign_clients_to_nearest_open_facility(
    service, all_open, capacities, demands
)

print("===== INSTANCE 116 CAPACITY CHECK =====")
print("Total demand:", round(float(demands.sum()), 2))
print("Total capacity:", round(float(capacities.sum()), 2))
print("All-open greedy feasible:",
      is_feasible(all_open, assignments, capacities, demands))

loads = np.bincount(
    assignments, weights=demands,
    minlength=len(capacities)
)

violations = np.where(loads > capacities + 1e-6)[0]

print("Overloaded facilities:", len(violations))
print("Maximum overload:",
      round(float(np.maximum(loads - capacities, 0).max()), 3))


===== INSTANCE 116 CAPACITY CHECK =====
Total demand: 854.36
Total capacity: 947.49
All-open greedy feasible: False
Overloaded facilities: 1
Maximum overload: 6.528


In [56]:

# BONUS B - Test Feasible Assignment Repair

import gurobipy as gp
import numpy as np

capacities = np.asarray(inst_116["facility_capacities"])
demands = np.asarray(inst_116["client_demands"])
service = np.asarray(inst_116["service_costs"])

n_clients = len(demands)
n_facilities = len(capacities)

model = gp.Model("GA_Assignment_Repair_Test")
model.Params.OutputFlag = 0
model.Params.TimeLimit = 60

x = model.addVars(
    range(n_clients),
    range(n_facilities),
    vtype=gp.GRB.BINARY
)

for i in range(n_clients):
    model.addConstr(
        gp.quicksum(
            x[i, j] for j in range(n_facilities)
        ) == 1
    )

for j in range(n_facilities):
    model.addConstr(
        gp.quicksum(
            demands[i] * x[i, j]
            for i in range(n_clients)
        ) <= capacities[j]
    )

model.setObjective(
    gp.quicksum(
        service[i, j] * x[i, j]
        for i in range(n_clients)
        for j in range(n_facilities)
    ),
    gp.GRB.MINIMIZE
)

model.optimize()

print("===== BONUS B REPAIR DIAGNOSIS =====")
print("Solver status:", model.Status)
print("Feasible assignment found:", model.SolCount > 0)

if model.SolCount > 0:
    repaired_assignments = np.array([
        max(
            range(n_facilities),
            key=lambda j: x[i, j].X
        )
        for i in range(n_clients)
    ], dtype=int)

    repaired_opened = np.ones(n_facilities, dtype=int)

    print(
        "Repaired solution feasible:",
        is_feasible(
            repaired_opened,
            repaired_assignments,
            capacities,
            demands
        )
    )
    print("ASSIGNMENT REPAIR TEST PASSED!")


===== BONUS B REPAIR DIAGNOSIS =====
Solver status: 2
Feasible assignment found: True
Repaired solution feasible: True
ASSIGNMENT REPAIR TEST PASSED!


In [57]:

# BONUS B - Repair Infeasible GA Solutions

import gurobipy as gp
import numpy as np

def repair_ga_solution(solution, inst):
    opened, assignments = solution

    opened = np.asarray(opened, dtype=int).copy()
    assignments = np.asarray(assignments, dtype=int).copy()

    capacities = np.asarray(inst["facility_capacities"])
    demands = np.asarray(inst["client_demands"])
    fixed = np.asarray(inst["fixed_costs"])
    service = np.asarray(inst["service_costs"])

    if is_feasible(opened, assignments, capacities, demands):
        return opened, assignments

    n_clients = len(demands)
    n_facilities = len(capacities)

    model = gp.Model("GA_Feasibility_Repair")
    model.Params.OutputFlag = 0
    model.Params.TimeLimit = 60

    x = model.addVars(
        range(n_clients),
        range(n_facilities),
        vtype=gp.GRB.BINARY
    )

    y = model.addVars(
        range(n_facilities),
        vtype=gp.GRB.BINARY
    )

    for i in range(n_clients):
        model.addConstr(
            gp.quicksum(x[i, j] for j in range(n_facilities)) == 1
        )

    for j in range(n_facilities):
        model.addConstr(
            gp.quicksum(
                demands[i] * x[i, j]
                for i in range(n_clients)
            ) <= capacities[j] * y[j]
        )

        model.addConstr(y[j] >= int(opened[j]))

    model.setObjective(
        gp.quicksum(
            fixed[j] * y[j]
            for j in range(n_facilities)
        )
        +
        gp.quicksum(
            service[i, j] * x[i, j]
            for i in range(n_clients)
            for j in range(n_facilities)
        ),
        gp.GRB.MINIMIZE
    )

    model.optimize()

    if model.SolCount == 0:
        raise RuntimeError("GA repair failed")

    repaired_opened = np.array([
        round(y[j].X) for j in range(n_facilities)
    ], dtype=int)

    repaired_assignments = np.array([
        max(
            range(n_facilities),
            key=lambda j: x[i, j].X
        )
        for i in range(n_clients)
    ], dtype=int)

    assert is_feasible(
        repaired_opened,
        repaired_assignments,
        capacities,
        demands
    )

    return repaired_opened, repaired_assignments


print("BONUS B - GA repair function loaded!")


BONUS B - GA repair function loaded!


In [58]:

# BONUS B - Test GA Repair on Instance 116

import random
import numpy as np

for name, solver in [
    ("Original GA", genetic_algorithm_solve_cflp),
    ("Adaptive GA", genetic_algorithm_adaptive_mutation)
]:
    random.seed(BONUS_SEED)
    np.random.seed(BONUS_SEED)

    result = solver(
        inst_116,
        population_size=30,
        max_generations=100,
        crossover_rate=0.85,
        mutation_rate=0.10
    )

    solution, trajectory = result
    repaired_opened, repaired_assignments = repair_ga_solution(
        solution, inst_116
    )

    feasible = is_feasible(
        repaired_opened,
        repaired_assignments,
        inst_116["facility_capacities"],
        inst_116["client_demands"]
    )

    repaired_cost = compute_total_cost(
        inst_116["fixed_costs"],
        inst_116["service_costs"],
        repaired_opened,
        repaired_assignments
    )

    print(f"\n{name}")
    print("Feasible after repair:", feasible)
    print("Repaired cost:", round(float(repaired_cost), 3))

    assert feasible, f"{name} repair failed"

print("\nBONUS B INSTANCE 116 REPAIR TEST PASSED!")


Starting Genetic Algorithm for CFLP (pop=30, max_gen=100)...
Genetic Algorithm completed after 30 generations.
Best cost: inf

Original GA
Feasible after repair: True
Repaired cost: 19405.974
Starting Genetic Algorithm for CFLP (pop=30, max_gen=100)...
Genetic Algorithm completed after 30 generations.
Best cost: inf

Adaptive GA
Feasible after repair: True
Repaired cost: 19405.974

BONUS B INSTANCE 116 REPAIR TEST PASSED!


In [59]:
import shutil
import os

source = "bonus_B_adaptive_full_447540190.csv"
backup = "bonus_B_adaptive_first115_backup_447540190.csv"

if os.path.exists(source):
    shutil.copy2(source, backup)
    print("Backup saved successfully!")
else:
    print("No existing results file found.")

Backup saved successfully!


In [62]:

# BONUS B - Final Preflight Check

import time
import random
import numpy as np

OUTPUT_FILE = "bonus_B_adaptive_full_repaired_447540190.csv"

print("Output file:", OUTPUT_FILE)

assert OUTPUT_FILE == "bonus_B_adaptive_full_repaired_447540190.csv"

for name, solver in [
    ("Original GA", genetic_algorithm_solve_cflp),
    ("Adaptive GA", genetic_algorithm_adaptive_mutation)
]:
    random.seed(BONUS_SEED)
    np.random.seed(BONUS_SEED)

    start = time.perf_counter()

    result = solver(
        inst_116,
        population_size=30,
        max_generations=100,
        crossover_rate=0.85,
        mutation_rate=0.10
    )

    solution, trajectory = result
    opened, assignments = repair_ga_solution(solution, inst_116)

    elapsed = time.perf_counter() - start

    cost = compute_total_cost(
        inst_116["fixed_costs"],
        inst_116["service_costs"],
        opened,
        assignments
    )

    feasible = is_feasible(
        opened,
        assignments,
        inst_116["facility_capacities"],
        inst_116["client_demands"]
    )

    print(name, "| Cost:", round(float(cost), 3),
          "| Time:", round(elapsed, 3),
          "| Feasible:", feasible)

    assert feasible and np.isfinite(cost)

print("BONUS B FINAL PREFLIGHT PASSED!")


Output file: bonus_B_adaptive_full_repaired_447540190.csv
Starting Genetic Algorithm for CFLP (pop=30, max_gen=100)...
Genetic Algorithm completed after 30 generations.
Best cost: inf
Original GA | Cost: 19405.974 | Time: 2.579 | Feasible: True
Starting Genetic Algorithm for CFLP (pop=30, max_gen=100)...
Genetic Algorithm completed after 30 generations.
Best cost: inf
Adaptive GA | Cost: 19405.974 | Time: 3.509 | Feasible: True
BONUS B FINAL PREFLIGHT PASSED!


In [67]:

# MY ANSWERS - SECTION A1
import pandas as pd

df = pd.read_csv("results_student_447540190.csv")

print("Available columns:")
print(df.columns.tolist())
print("\nTotal instances:", len(df))
print("\nGurobi statuses:")
if "gurobi_status" in df.columns:
    print(df["gurobi_status"].value_counts(dropna=False))


Available columns:
['instance_id', 'instance_name', 'size', 'difficulty', 'num_facilities', 'num_clients', 'gurobi_cost', 'gurobi_time', 'gurobi_status', 'ts_cost', 'ts_time', 'ts_iterations', 'ts_status', 'ts_gap_percent', 'ts_speedup_factor', 'ga_cost', 'ga_time', 'ga_generations', 'ga_status', 'ga_gap_percent', 'ga_speedup_factor']

Total instances: 144

Gurobi statuses:
gurobi_status
Optimal    144
Name: count, dtype: int64


In [68]:

# MY ANSWERS - SECTION A1 FINAL RESULTS

optimal_df = df[
    df["gurobi_status"].astype(str).str.lower() == "optimal"
]

ts_mean = optimal_df["ts_gap_percent"].mean()
ga_mean = optimal_df["ga_gap_percent"].mean()

print("===== SECTION A1 RESULTS =====")
print("Optimal instances:", len(optimal_df))
print("Tabu Search mean gap (%):", round(ts_mean, 4))
print("Genetic Algorithm mean gap (%):", round(ga_mean, 4))

if ts_mean < ga_mean:
    print("Method with lower overall gap: TS")
elif ga_mean < ts_mean:
    print("Method with lower overall gap: GA")
else:
    print("Both methods have equal mean gaps")


===== SECTION A1 RESULTS =====
Optimal instances: 144
Tabu Search mean gap (%): 10.3489
Genetic Algorithm mean gap (%): 11.1145
Method with lower overall gap: TS


In [69]:

# MY ANSWERS - SECTION A2
# Mean Optimality Gap by Instance Size

import pandas as pd

df = pd.read_csv("results_student_447540190.csv")

size_order = ["Small", "Medium", "Large"]

a2_results = (
    df.assign(
        size=df["size"].astype(str).str.strip().str.title()
    )
    .groupby("size")[
        ["ts_gap_percent", "ga_gap_percent"]
    ]
    .mean()
    .reindex(size_order)
)

print("===== SECTION A2 RESULTS =====")
print(a2_results.round(4).to_string())

print("\nNumber of instances by size:")
print(
    df["size"]
    .astype(str)
    .str.strip()
    .str.title()
    .value_counts()
    .reindex(size_order)
)


===== SECTION A2 RESULTS =====
        ts_gap_percent  ga_gap_percent
size                                  
Small           8.9403          9.9864
Medium         10.2813         10.9426
Large          11.8249         12.4144

Number of instances by size:
size
Small     48
Medium    48
Large     48
Name: count, dtype: int64


In [70]:

# MY ANSWERS - SECTION A3
# Mean Optimality Gap by Difficulty Level

import pandas as pd

df = pd.read_csv("results_student_447540190.csv")

difficulty_order = ["Easy", "Moderate", "Expensive"]

a3_results = (
    df.assign(
        difficulty=df["difficulty"].astype(str).str.strip().str.title()
    )
    .groupby("difficulty")[
        ["ts_gap_percent", "ga_gap_percent"]
    ]
    .mean()
    .reindex(difficulty_order)
)

print("===== SECTION A3 RESULTS =====")
print(a3_results.round(4).to_string())

print("\nHardest difficulty for TS:",
      a3_results["ts_gap_percent"].idxmax())

print("Hardest difficulty for GA:",
      a3_results["ga_gap_percent"].idxmax())

print("\nNumber of instances by difficulty:")
print(
    df["difficulty"]
    .astype(str)
    .str.strip()
    .str.title()
    .value_counts()
    .reindex(difficulty_order)
)


===== SECTION A3 RESULTS =====
            ts_gap_percent  ga_gap_percent
difficulty                                
Easy                8.3008          8.5819
Moderate           13.6482         14.7502
Expensive           9.0975         10.0113

Hardest difficulty for TS: Moderate
Hardest difficulty for GA: Moderate

Number of instances by difficulty:
difficulty
Easy         48
Moderate     48
Expensive    48
Name: count, dtype: int64


In [71]:

# MY ANSWERS - SECTION A4
# Head-to-Head Comparison: TS vs GA

import pandas as pd

df = pd.read_csv("results_student_447540190.csv")

ts_wins = (df["ts_gap_percent"] < df["ga_gap_percent"]).sum()
ga_wins = (df["ga_gap_percent"] < df["ts_gap_percent"]).sum()
ties = (df["ts_gap_percent"] == df["ga_gap_percent"]).sum()

total = len(df)

print("===== SECTION A4 RESULTS =====")
print("Total instances:", total)

print("TS wins:", ts_wins)
print("TS winning fraction:", round(ts_wins / total, 4))
print("TS winning percentage:", round(ts_wins / total * 100, 2))

print("GA wins:", ga_wins)
print("GA winning fraction:", round(ga_wins / total, 4))
print("GA winning percentage:", round(ga_wins / total * 100, 2))

print("Ties:", ties)
print("Ties percentage:", round(ties / total * 100, 2))

assert ts_wins + ga_wins + ties == total


===== SECTION A4 RESULTS =====
Total instances: 144
TS wins: 78
TS winning fraction: 0.5417
TS winning percentage: 54.17
GA wins: 3
GA winning fraction: 0.0208
GA winning percentage: 2.08
Ties: 63
Ties percentage: 43.75


In [72]:

# MY ANSWERS - SECTION B1
# Mean Computation Time by Instance Size

import pandas as pd

df = pd.read_csv("results_student_447540190.csv")

size_order = ["Small", "Medium", "Large"]

b1_results = (
    df.assign(
        size=df["size"].astype(str).str.strip().str.title()
    )
    .groupby("size")[
        ["gurobi_time", "ts_time", "ga_time"]
    ]
    .mean()
    .reindex(size_order)
)

print("===== SECTION B1 RESULTS =====")
print(b1_results.round(4).to_string())

gurobi_large = b1_results.loc["Large", "gurobi_time"]
ts_large = b1_results.loc["Large", "ts_time"]

ts_speedup_large = gurobi_large / ts_large

print("\nTS speedup over Gurobi (Large):",
      round(ts_speedup_large, 4), "x")

print("\nNumber of instances by size:")
print(
    df["size"]
    .astype(str)
    .str.strip()
    .str.title()
    .value_counts()
    .reindex(size_order)
)


===== SECTION B1 RESULTS =====
        gurobi_time  ts_time  ga_time
size                                 
Small        5.6565   1.0560   0.7987
Medium      11.7097   1.6655   1.3556
Large        9.6906   1.7786   1.7620

TS speedup over Gurobi (Large): 5.4484 x

Number of instances by size:
size
Small     48
Medium    48
Large     48
Name: count, dtype: int64


In [73]:

# MY ANSWERS - SECTION B2
# GA Speedup vs TS Speedup

import pandas as pd

df = pd.read_csv("results_student_447540190.csv")

ga_speedup = df["ga_speedup_factor"].mean()
ts_speedup = df["ts_speedup_factor"].mean()

print("===== SECTION B2 RESULTS =====")
print("Mean GA speedup over Gurobi:", round(ga_speedup, 4), "x")
print("Mean TS speedup over Gurobi:", round(ts_speedup, 4), "x")

if ga_speedup > ts_speedup:
    print("Higher mean speedup: GA")
    print("Difference:", round(ga_speedup - ts_speedup, 4), "x")
elif ts_speedup > ga_speedup:
    print("Higher mean speedup: TS")
    print("Difference:", round(ts_speedup - ga_speedup, 4), "x")
else:
    print("Both algorithms have equal mean speedup")


===== SECTION B2 RESULTS =====
Mean GA speedup over Gurobi: 5.8929 x
Mean TS speedup over Gurobi: 7.6565 x
Higher mean speedup: TS
Difference: 1.7636 x


In [74]:

# MY ANSWERS - SECTION B3
# Runtime Growth by Instance Size

import pandas as pd

df = pd.read_csv("results_student_447540190.csv")

b3_times = (
    df.assign(
        size=df["size"].astype(str).str.strip().str.title()
    )
    .groupby("size")[
        ["gurobi_time", "ts_time", "ga_time"]
    ]
    .mean()
    .reindex(["Small", "Medium", "Large"])
)

print("===== SECTION B3 RESULTS =====")

for algorithm, column in [
    ("Gurobi", "gurobi_time"),
    ("Tabu Search", "ts_time"),
    ("Genetic Algorithm", "ga_time")
]:
    small = b3_times.loc["Small", column]
    medium = b3_times.loc["Medium", column]
    large = b3_times.loc["Large", column]

    small_to_medium = medium / small
    medium_to_large = large / medium

    print("\n" + algorithm)
    print("Small -> Medium:", round(small_to_medium, 4), "x")
    print("Medium -> Large:", round(medium_to_large, 4), "x")


===== SECTION B3 RESULTS =====

Gurobi
Small -> Medium: 2.0701 x
Medium -> Large: 0.8276 x

Tabu Search
Small -> Medium: 1.5772 x
Medium -> Large: 1.0679 x

Genetic Algorithm
Small -> Medium: 1.6973 x
Medium -> Large: 1.2999 x


In [75]:

# MY ANSWERS - SECTION C1
# Gap Heatmap by Size and Difficulty

import pandas as pd

df = pd.read_csv("results_student_447540190.csv")

df["size_group"] = df["size"].astype(str).str.strip().str.title()
df["difficulty_group"] = df["difficulty"].astype(str).str.strip().str.title()

ts_heatmap = df.pivot_table(
    index="size_group",
    columns="difficulty_group",
    values="ts_gap_percent",
    aggfunc="mean"
)

ga_heatmap = df.pivot_table(
    index="size_group",
    columns="difficulty_group",
    values="ga_gap_percent",
    aggfunc="mean"
)

size_order = ["Small", "Medium", "Large"]
difficulty_order = ["Easy", "Moderate", "Expensive"]

ts_heatmap = ts_heatmap.reindex(
    index=size_order, columns=difficulty_order
)
ga_heatmap = ga_heatmap.reindex(
    index=size_order, columns=difficulty_order
)

ts_max = ts_heatmap.stack().idxmax()
ga_max = ga_heatmap.stack().idxmax()

print("===== SECTION C1 RESULTS =====")
print("\nTS mean gap heatmap:")
print(ts_heatmap.round(4).to_string())

print("\nGA mean gap heatmap:")
print(ga_heatmap.round(4).to_string())

print("\nHighest TS gap:", ts_max,
      round(ts_heatmap.loc[ts_max], 4), "%")

print("Highest GA gap:", ga_max,
      round(ga_heatmap.loc[ga_max], 4), "%")


===== SECTION C1 RESULTS =====

TS mean gap heatmap:
difficulty_group    Easy  Moderate  Expensive
size_group                                   
Small             7.2889   11.1977     8.3344
Medium            8.2154   13.1679     9.4606
Large             9.3982   16.5790     9.4976

GA mean gap heatmap:
difficulty_group    Easy  Moderate  Expensive
size_group                                   
Small             7.6447   12.9739     9.3408
Medium            8.6064   13.6447    10.5767
Large             9.4948   17.6319    10.1164

Highest TS gap: ('Large', 'Moderate') 16.579 %
Highest GA gap: ('Large', 'Moderate') 17.6319 %


In [76]:

# BONUS A - Summary of tested enhancements

import pandas as pd

df = pd.read_csv("bonus_A_adaptive_student_447540190.csv")

print("Columns:")
print(df.columns.tolist())

print("\nFirst 3 rows:")
print(df.head(3).to_string(index=False))

print("\nTotal instances:", len(df))


Columns:
['instance_name', 'size', 'difficulty', 'gurobi_cost', 'original_ts_cost', 'improved_ts_cost', 'original_ts_time', 'improved_ts_time', 'original_gap_percent', 'improved_gap_percent', 'gap_reduction_pp', 'feasible', 'initial_repaired']

First 3 rows:
                       instance_name  size difficulty  gurobi_cost  original_ts_cost  improved_ts_cost  original_ts_time  improved_ts_time  original_gap_percent  improved_gap_percent  gap_reduction_pp  feasible initial_repaired
001_cflp_small_easy_fac16_cli75.json small       easy 17219.957857      18007.527403      18007.527403          0.427347          0.525654              4.573586              4.573586      0.000000e+00      True              NaN
002_cflp_small_easy_fac20_cli65.json small       easy 10470.464833      11102.690547      11102.690547          0.491138          0.927885              6.038182              6.038182     -1.687539e-14      True              NaN
003_cflp_small_easy_fac18_cli62.json small       easy 116

In [77]:

# BONUS A - Final Adaptive Tenure Summary

import pandas as pd

df = pd.read_csv("bonus_A_adaptive_student_447540190.csv")

print("===== BONUS A RESULTS =====")
print("Total instances:", len(df))

print("Original mean gap (%):",
      round(df["original_gap_percent"].mean(), 4))

print("Improved mean gap (%):",
      round(df["improved_gap_percent"].mean(), 4))

print("Mean gap reduction (percentage points):",
      round(df["gap_reduction_pp"].mean(), 4))

print("Improved instances:",
      (df["gap_reduction_pp"] > 1e-8).sum())

print("Equal instances:",
      (df["gap_reduction_pp"].abs() <= 1e-8).sum())

print("Worse instances:",
      (df["gap_reduction_pp"] < -1e-8).sum())

print("Original mean runtime (s):",
      round(df["original_ts_time"].mean(), 4))

print("Improved mean runtime (s):",
      round(df["improved_ts_time"].mean(), 4))

print("All feasible:",
      df["feasible"].all())


===== BONUS A RESULTS =====
Total instances: 144
Original mean gap (%): 10.3489
Improved mean gap (%): 10.3366
Mean gap reduction (percentage points): 0.0123
Improved instances: 42
Equal instances: 53
Worse instances: 49
Original mean runtime (s): 1.5
Improved mean runtime (s): 1.1752
All feasible: True


In [78]:

# BONUS A - Check available screening results

import pandas as pd
import glob
import os

files = sorted(glob.glob("bonus_A*.csv"))

for file in files:
    df = pd.read_csv(file)

    print("\n" + "=" * 50)
    print("File:", os.path.basename(file))
    print("Number of instances:", len(df))
    print("Columns:", df.columns.tolist())

    gap_cols = [
        c for c in df.columns
        if "gap" in c.lower()
    ]

    if gap_cols:
        print("\nMean gap values:")
        print(df[gap_cols].mean(numeric_only=True).round(4))



File: bonus_A_adaptive_student_447540190.csv
Number of instances: 144
Columns: ['instance_name', 'size', 'difficulty', 'gurobi_cost', 'original_ts_cost', 'improved_ts_cost', 'original_ts_time', 'improved_ts_time', 'original_gap_percent', 'improved_gap_percent', 'gap_reduction_pp', 'feasible', 'initial_repaired']

Mean gap values:
original_gap_percent    10.3489
improved_gap_percent    10.3366
gap_reduction_pp         0.0123
dtype: float64

File: bonus_A_partial_student_447540190.csv
Number of instances: 144
Columns: ['instance_name', 'size', 'difficulty', 'gurobi_cost', 'original_ts_cost', 'improved_ts_cost', 'original_ts_time', 'improved_ts_time', 'original_gap_percent', 'improved_gap_percent', 'gap_reduction_pp', 'feasible', 'initial_repaired']

Mean gap values:
original_gap_percent    10.3489
improved_gap_percent    10.3366
gap_reduction_pp         0.0123
dtype: float64


In [79]:

# BONUS B - Pilot Screening Results

import pandas as pd
import glob
import os

for file in sorted(glob.glob("bonus_B_*pilot*.csv")):
    df = pd.read_csv(file)

    print("\n" + "=" * 45)
    print("File:", os.path.basename(file))
    print("Instances:", len(df))
    print("Columns:", df.columns.tolist())

    for col in df.columns:
        if "gap" in col.lower() or "improvement" in col.lower():
            if pd.api.types.is_numeric_dtype(df[col]):
                print(col, "=", round(df[col].mean(), 4))



File: bonus_B_adaptive_pilot_447540190.csv
Instances: 6
Columns: ['Instance', 'Original_GA_Cost', 'Adaptive_GA_Cost', 'Improvement_%', 'Original_Time', 'Adaptive_Time']
Improvement_% = 0.1349

File: bonus_B_diversity_pilot_447540190.csv
Instances: 6
Columns: ['Instance', 'Original_Cost', 'Original_Time', 'Original_Feasible', 'Diversity_Cost', 'Diversity_Time', 'Diversity_Feasible', 'Improvement_%']
Improvement_% = 0.1046

File: bonus_B_memetic_pilot_447540190.csv
Instances: 6
Columns: ['Instance', 'Original_Cost', 'Original_Time', 'Original_Feasible', 'Memetic_Cost', 'Memetic_Time', 'Memetic_Feasible', 'Improvement_%']
Improvement_% = 0.051

File: bonus_B_two_point_pilot_447540190.csv
Instances: 6
Columns: ['Instance', 'Original_GA_Cost', 'TwoPoint_GA_Cost', 'Improvement_%', 'Original_Time', 'TwoPoint_Time']
Improvement_% = -0.0844


In [81]:

# BONUS B - Correct Pilot Screening Results

import pandas as pd

files = {
    "Adaptive mutation": "bonus_B_adaptive_pilot_447540190.csv",
    "Two-point crossover": "bonus_B_two_point_pilot_447540190.csv",
    "Memetic GA": "bonus_B_memetic_pilot_447540190.csv",
    "Diversity preservation": "bonus_B_diversity_pilot_447540190.csv"
}

baseline = pd.read_csv("results_student_447540190.csv")
baseline["key"] = baseline["instance_name"].astype(str).str.split("/").str[-1]
baseline = baseline.set_index("key")

for method, path in files.items():
    df = pd.read_csv(path)

    original_col = next(
        c for c in df.columns
        if c.lower() in ("original_ga_cost", "original_cost")
    )

    improved_col = next(
        c for c in df.columns
        if c.lower().endswith("_cost")
        and c != original_col
        and c.lower() != "gurobi_cost"
    )

    original_gaps = []
    improved_gaps = []

    for _, row in df.iterrows():
        name = str(row["Instance"]).split("/")[-1]
        optimal = float(baseline.loc[name, "gurobi_cost"])

        original_gaps.append(
            (float(row[original_col]) - optimal) / optimal * 100
        )
        improved_gaps.append(
            (float(row[improved_col]) - optimal) / optimal * 100
        )

    print(method)
    print("Mean gap before (%):", round(pd.Series(original_gaps).mean(), 4))
    print("Mean gap after (%):", round(pd.Series(improved_gaps).mean(), 4))
    print("Improvement (pp):",
          round(pd.Series(original_gaps).mean() -
                pd.Series(improved_gaps).mean(), 4))
    print("-" * 40)



Adaptive mutation
Mean gap before (%): 7.0327
Mean gap after (%): 6.8877
Improvement (pp): 0.145
----------------------------------------
Two-point crossover
Mean gap before (%): 6.8877
Mean gap after (%): 6.9772
Improvement (pp): -0.0895
----------------------------------------
Memetic GA
Mean gap before (%): 7.0327
Mean gap after (%): 6.9772
Improvement (pp): 0.0555
----------------------------------------
Diversity preservation
Mean gap before (%): 7.0327
Mean gap after (%): 6.9212
Improvement (pp): 0.1114
----------------------------------------
